<a href="https://colab.research.google.com/github/Silas-Kamanda/Projects/blob/main/Copy_of_LU_Swahili_ASR_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

LoRA with no augmentation" See below

In [ ]:
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.44.1
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

2. LoRA + Gaussian Noise Only

In [ ]:
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.44.1
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:

# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Gaussian Noise Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-gaussian-only",
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "GaussianNoise",
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/Lora_Gaussian"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (GAUSSIAN NOISE ONLY)
print("--- Preparing Dataset for Whisper (Gaussian Noise Only) ---")

# ONLY GAUSSIAN NOISE IN THE PIPELINE
time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=1.0)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Gaussian Noise Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_gaussian.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Gaussian Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & Gaussian Noise Augmentation only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

Baseline - LoRA Without Augmentation

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Baseline/No Augmentation + WandB) ---")

# Disable frozen modules and debugging validations
import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

try:
    import json
    import re
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    import torchaudio
    from torchaudio.transforms import SpecAugment
    import transformers
    import datasets
    import accelerate
    import evaluate
    import wandb
    from tqdm.notebook import tqdm
    from sklearn.model_selection import train_test_split
    from datasets import Dataset, DatasetDict
    from transformers import (
        WhisperProcessor,
        WhisperFeatureExtractor,
        WhisperTokenizerFast,
        WhisperForConditionalGeneration,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
        EarlyStoppingCallback
    )
    from peft import (
        prepare_model_for_kbit_training,
        LoraConfig,
        PeftModel,
        PeftConfig,
        get_peft_model
    )
    from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
    from dataclasses import dataclass
    from typing import Any, Dict, List, Union
    import warnings
    import gradio as gr
    import csv
    import soundfile as sf
    import gc
    warnings.filterwarnings("ignore")
except ImportError as e:
    print(f"Import error: {e}")
    raise

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"WandB version: {wandb.__version__}")
try:
    import peft
    print(f"PEFT version: {peft.__version__}")
    import audiomentations
    print(f"Audiomentations version: {audiomentations.__version__}")
except ImportError as e:
    print(f"Failed to import library: {e}")
    raise ImportError("Required library missing. Check installation logs.")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Count: {torch.cuda.device_count()}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-baseline-no-aug", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "None", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_Colab_NewRun/LoRA_No_Augmentation"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
print(f"CSV found at: {csv_path}")
print(f"Audio base directory: {audio_base_path}")
print(f"Model outputs to: {output_dir}")
os.makedirs(output_dir, exist_ok=True)

try:
    df = pd.read_csv(csv_path).head(9000)
    print(f"Initial dataframe shape: {df.shape}")
except FileNotFoundError:
    print(f"Error loading CSV from {csv_path}. Upload swahili_dataset.csv to /content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/")
    raise

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text) # Keeps apostrophes
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]
print(f"Shape after cleaning: {df.shape}")

try:
    print("Skipping audio validation to save time. Assuming all audio files are valid.")
    if len(df) == 0: raise ValueError("No valid samples after transcript cleaning.")
    # Ensuring 80/10/10 split on 9000 samples -> 7200 train, 900 val, 900 test
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True) # 0.2 * 9000 = 1800 for temp
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True) # 0.5 * 1800 = 900 each
    del df, temp_df; gc.collect()
except Exception as e_split:
    print(f"Dataset splitting error: {e_split}")
    raise

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"\nDataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")
if len(raw_datasets["train"]) > 0: print(f"Sample 0 (raw): {raw_datasets['train'][0]}")
print(f"Raw dataset columns: {raw_datasets['train'].features}")
print("-" * 30)

model_name_loaded = "openai/whisper-small" #model being loaded
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
try:
    feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
    tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
    processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
    print(f"Whisper Processor for {language} ({language_abbr}), task: {task} loaded from {model_name_loaded}.")
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
        print(f"Set processor.tokenizer.pad_token_id to eos_token_id: {processor.tokenizer.eos_token_id}")
except Exception as e_proc:
    print(f"Error loading Whisper processor: {e_proc}")
    raise
print("-" * 30)

# 3. Prepare Dataset for Whisper (BASELINE - NO AUGMENTATION)
print("--- Preparing Dataset for Whisper (Baseline) ---")

# EMPTY THE TIME-DOMAIN PIPELINE
time_augment_pipeline = Compose([])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            print(f"Error in load_process_audio_batch for {audio_path}: {e_lp}")
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
try:
    for split in raw_datasets:
        original_len = len(raw_datasets[split])
        raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")
        if len(raw_datasets[split]) != original_len:
            print(f"Filtered out {original_len - len(raw_datasets[split])} missing paths from {split} split.")
        if len(raw_datasets[split]) == 0 and split == "train": raise ValueError(f"Train split empty after path filtering.")
    print(f"Train size after path filter: {len(raw_datasets['train'])}")
except Exception as e_filt:
    print(f"Path filtering error: {e_filt}")
    raise

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    print(f"Processing {split_name} audio (augment={is_train})...")
    if len(dataset_split) == 0:
        print(f"Skipping processing for empty split: {split_name}")
        processed_splits[split_name] = dataset_split
        continue
    try:
        processed_splits[split_name] = dataset_split.map(
            lambda batch: load_process_audio_batch(batch, do_augment=is_train),
            batched=True, batch_size=8, num_proc=1, # num_proc=1 for Colab stability with audio
            desc=f"Processing {split_name} audio"
        )
        print(f"{split_name} size before input_features filter: {len(processed_splits[split_name])}")
        processed_splits[split_name] = processed_splits[split_name].filter(
            lambda example: example["input_features"] is not None,
            num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
        )
        print(f"{split_name} size after input_features filter: {len(processed_splits[split_name])}")
        if len(processed_splits[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after input_features filter.")
    except Exception as e_proc_split:
        print(f"Error processing {split_name}: {e_proc_split}")
        raise
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()
print("Audio features processed.")
if len(raw_datasets_with_features.get("train", [])) > 0:
    print(f"Columns after feature extraction: {raw_datasets_with_features['train'].features}")

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
try:
    processed_datasets = raw_datasets_with_features.map(
        prepare_dataset_labels, batched=True, batch_size=8,
        remove_columns=["audio_path", "sentence"],
        num_proc=1, desc="Tokenizing text"
    )
    del raw_datasets_with_features; gc.collect()
except Exception as e_tok:
    print(f"Tokenization error: {e_tok}")
    raise

for split_name in processed_datasets:
    print(f"{split_name} size before final combined filter: {len(processed_datasets[split_name])}")
    if len(processed_datasets[split_name]) == 0: continue
    try:
        processed_datasets[split_name] = processed_datasets[split_name].filter(
            lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                       ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
            num_proc=1, desc=f"Final filtering on {split_name}"
        )
        print(f"{split_name} size after final combined filter: {len(processed_datasets[split_name])}")
        if len(processed_datasets[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after final combined filter.")
    except Exception as e_filt_final:
        print(f"Final filtering error on {split_name}: {e_filt_final}")
        raise
print("Dataset prepared for Whisper training.")
if len(processed_datasets.get("train",[])) > 0:
    print(f"Processed dataset final columns: {list(processed_datasets['train'].features.keys())}")
    print(f"Sample input_features type: {type(processed_datasets['train'][0]['input_features'])}")
    sample_feat_np = np.asarray(processed_datasets['train'][0]['input_features'])
    print(f"Sample input_features shape: {sample_feat_np.shape}")
    print(f"Sample labels (first 10): {processed_datasets['train'][0]['labels'][:10]}")

print(f"Dataset sizes - Train: {len(processed_datasets.get('train',[]))}, Val: {len(processed_datasets.get('val',[]))}, Test: {len(processed_datasets.get('test',[]))}")
print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
print("--- Defining Seq2Seq Data Collator ---")
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Filter out features where input_features or labels might be None or labels empty
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {} # Return empty dict if no valid features after filtering

        input_features_list = [f["input_features"] for f in valid_features]
        # Pad the input features
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        # Tokenize and pad the label features
        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appendend in previous tokenization, slice it
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data Collator defined.")
print("-" * 30)

# 5. Define Evaluation Metrics
print("--- Defining Evaluation Metrics ---")
metric = evaluate.load("wer")

def normalize_text(text): # Ensure consistent normalization
    if not isinstance(text, str): text = str(text) # Handle potential non-string inputs
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text) # Keeps apostrophes
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    # Normalize text
    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    # Print a few examples for qualitative inspection
    for i in range(min(3, len(pred_str))): # Print up to 3 examples
        print(f"Pred [{i}]: {pred_str[i]}")
        print(f"Ref  [{i}]: {label_str[i]}")

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    # Log to WandB
    try:
        # Check if trainer object exists and has state
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer}) # Log WER without step if trainer state not available
    except Exception as e_wandb_log:
        print(f"WandB logging error in compute_metrics: {e_wandb_log}")

    return {"wer": wer}
print("Compute Metrics function defined with WandB logging.")
print("-" * 30)

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
try:
    model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
    model.config.suppress_tokens = []
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    if torch.cuda.is_available(): model.to("cuda")
    print("Whisper model loaded and LoRA configured.")
    print(f"Model is on device: {next(model.parameters()).device}")
except Exception as e_model:
    print(f"Error loading model or applying LoRA: {e_model}")
    raise
print("Checking GPU memory after model loading...")
!nvidia-smi
print("-" * 30)

#  7. Define Training Arguments
print("--- Defining Training Arguments for Seq2Seq + LoRA ---")
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)
print("\nTraining Arguments:")
print(f" -> LR: {training_args.learning_rate}, Max Epochs: {training_args.num_train_epochs}")
eff_bs = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * (torch.cuda.device_count() if torch.cuda.is_available() else 1)
print(f" -> Eff. BS: {eff_bs}, FP16: {training_args.fp16}")
print(f" -> Generation Num Beams for Eval: {training_args.generation_num_beams}")
print(f" -> Early Stopping will be handled by a callback. Weight Decay: {training_args.weight_decay}")
print(f" -> Output Dir: {training_args.output_dir}")
print(f" -> Logging to WandB enabled.")
print("-" * 30)

# 8. Initialize Trainer
print("--- Initializing Seq2Seq Trainer ---")
if len(processed_datasets.get("train", [])) == 0: raise ValueError("Training dataset is empty. Cannot initialize trainer.")
if len(processed_datasets.get("val", [])) == 0: raise ValueError("Validation dataset is empty. Cannot initialize trainer for evaluation.")

early_stopping_patience_value = 5 # Ensure this value is properly set
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)
print(f"Early stopping enabled with patience: {early_stopping_patience_value} evaluations.")

try:
    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=processed_datasets["train"],
        eval_dataset=processed_datasets["val"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[early_stopping_callback]
    )
    print("Trainer initialized.")
except Exception as e_trainer:
    print(f"Error initializing trainer: {e_trainer}")
    raise
print("-" * 30)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Baseline) ---")
final_trainer_state = None
try:
    print("Starting training from scratch with baseline LoRA configuration and WandB logging...")
    torch.cuda.empty_cache()
    gc.collect()
    print("Checking GPU memory before training...")
    !nvidia-smi
    train_result = trainer.train()
    final_trainer_state = trainer.state
    print("\nTraining finished.")
    print(f"Train metrics: {train_result.metrics}")
    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)
    print(f"Training metrics saved to {metrics_path}")

    print("\nSaving final PEFT adapter model (best model based on 'wer')...")
    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    print(f"✅ Final PEFT adapter (final state) saved to {os.path.join(output_dir, 'final_adapter')}")
    processor.save_pretrained(output_dir)
    print(f"✅ Processor saved to {output_dir}")

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None}) # Log best eval WER

except RuntimeError as e_rt:
    if "out of memory" in str(e_rt).lower():
        print("\nCUDA OOM Error! Try reducing LoRA rank 'r', effective batch_size, or target_modules, or using a smaller base model.")
    print(f"Runtime Error during training: {e_rt}")
    !nvidia-smi
    raise
except Exception as e_tr:
    print(f"\nTraining error: {type(e_tr).__name__} - {e_tr}")
    !nvidia-smi
    raise

# Final evaluation using the best model loaded by the Trainer
if final_trainer_state and trainer.state.best_model_checkpoint:
    print(f"\nEvaluating on validation set using the best model from {trainer.state.best_model_checkpoint}...")
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        print(f"Final Evaluation results (best model): {eval_results}")
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        print(f"Evaluation metrics for best model saved to {eval_metrics_path}")
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")
else:
    print("\nSkipping final evaluation as training did not complete successfully or no best model checkpoint found.")
print("-" * 30)

# 10. Generate Transcriptions on Test Set
print("--- Generating Transcriptions for Test Set using PEFT Model ---")
# Use the best model checkpoint identified by the trainer for inference
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_baseline.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    print(f"Loading best PEFT adapter from: {best_checkpoint_dir} for inference...")
    try:
        base_model_name_or_path = model_name_loaded
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32

        base_model = WhisperForConditionalGeneration.from_pretrained(
            base_model_name_or_path, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()

        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir) # Load processor from checkpoint
        print("Best PEFT model and processor loaded successfully for inference.")
    except Exception as e_load:
        print(f"Error loading best PEFT model for inference from {best_checkpoint_dir}: {e_load}.")
else:
    print(f"Best model checkpoint not found or training didn't complete. Check: {best_checkpoint_dir}")


def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]):
        print("Skipping transcription generation (model/processor not loaded or load failed).")
        return
    results = []
    device = inference_model.device
    print(f"Generating transcriptions for {len(test_dataset)} samples on device: {device}...")

    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            # Duration calculation (rough estimate based on features)
            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01 # Approx for Whisper features
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            trans = f"Error: {type(e_inf).__name__} - {str(e_inf)[:100]}"
            print(f"Inference error for one sample: {e_inf}")
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        print(f"✅ Transcriptions saved to {output_csv_path}")
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        # Calculate WER on test set
        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        print(f"Error saving transcriptions CSV or calculating test WER: {e_csv}")

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)
else:
    print("Skipping inference generation on test set (model/processor load failed or test set empty).")
print("-" * 30)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded. Training may have failed or was skipped."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        print(f"Gradio input: {audio_filepath}")
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        print(f" -> Gradio Output: {transcription}")
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        print(f"Gradio Error: {type(e_grad).__name__} - {e_grad}")
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Base)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA (r={lora_config.r}, alpha={lora_config.lora_alpha}, targets={lora_config.target_modules}) & NO Augmentation. Dataset: {wandb.config.num_samples_train} train samples.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        print(f"Error launching Gradio: {e_launch}.")
else:
    print("\nSkipping Gradio launch (model/processor not loaded, possibly due to issues in training or loading).")
print("\n--- Script Finished ---")
wandb.finish()

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Baseline/No Augmentation + WandB) ---")

# Disable frozen modules and debugging validations
import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

try:
    import json
    import re
    import numpy as np
    import pandas as pd
    import torch
    import torchvision
    import torchaudio
    from torchaudio.transforms import SpecAugment
    import transformers
    import datasets
    import accelerate
    import evaluate
    import wandb
    from tqdm.notebook import tqdm
    from sklearn.model_selection import train_test_split
    from datasets import Dataset, DatasetDict
    from transformers import (
        WhisperProcessor,
        WhisperFeatureExtractor,
        WhisperTokenizerFast,
        WhisperForConditionalGeneration,
        Seq2SeqTrainingArguments,
        Seq2SeqTrainer,
        EarlyStoppingCallback
    )
    from peft import (
        prepare_model_for_kbit_training,
        LoraConfig,
        PeftModel,
        PeftConfig,
        get_peft_model
    )
    from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
    from dataclasses import dataclass
    from typing import Any, Dict, List, Union
    import warnings
    import gradio as gr
    import csv
    import soundfile as sf
    import gc
    warnings.filterwarnings("ignore")
except ImportError as e:
    print(f"Import error: {e}")
    raise

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Torchaudio version: {torchaudio.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"Datasets version: {datasets.__version__}")
print(f"Accelerate version: {accelerate.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"WandB version: {wandb.__version__}")
try:
    import peft
    print(f"PEFT version: {peft.__version__}")
    import audiomentations
    print(f"Audiomentations version: {audiomentations.__version__}")
except ImportError as e:
    print(f"Failed to import library: {e}")
    raise ImportError("Required library missing. Check installation logs.")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Device Count: {torch.cuda.device_count()}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-baseline-no-aug", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "None", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_Colab_NewRun/LoRA_No_Augmentation"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
print(f"CSV found at: {csv_path}")
print(f"Audio base directory: {audio_base_path}")
print(f"Model outputs to: {output_dir}")
os.makedirs(output_dir, exist_ok=True)

try:
    df = pd.read_csv(csv_path).head(9000)
    print(f"Initial dataframe shape: {df.shape}")
except FileNotFoundError:
    print(f"Error loading CSV from {csv_path}. Upload swahili_dataset.csv to /content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/")
    raise

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text) # Keeps apostrophes
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]
print(f"Shape after cleaning: {df.shape}")

try:
    print("Skipping audio validation to save time. Assuming all audio files are valid.")
    if len(df) == 0: raise ValueError("No valid samples after transcript cleaning.")
    # Ensuring 80/10/10 split on 9000 samples -> 7200 train, 900 val, 900 test
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True) # 0.2 * 9000 = 1800 for temp
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True) # 0.5 * 1800 = 900 each
    del df, temp_df; gc.collect()
except Exception as e_split:
    print(f"Dataset splitting error: {e_split}")
    raise

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"\nDataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")
if len(raw_datasets["train"]) > 0: print(f"Sample 0 (raw): {raw_datasets['train'][0]}")
print(f"Raw dataset columns: {raw_datasets['train'].features}")
print("-" * 30)

model_name_loaded = "openai/whisper-small" #model being loaded
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
try:
    feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
    tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
    processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
    print(f"Whisper Processor for {language} ({language_abbr}), task: {task} loaded from {model_name_loaded}.")
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id
        print(f"Set processor.tokenizer.pad_token_id to eos_token_id: {processor.tokenizer.eos_token_id}")
except Exception as e_proc:
    print(f"Error loading Whisper processor: {e_proc}")
    raise
print("-" * 30)

# 3. Prepare Dataset for Whisper (BASELINE - NO AUGMENTATION)
print("--- Preparing Dataset for Whisper (Baseline) ---")

# EMPTY THE TIME-DOMAIN PIPELINE
time_augment_pipeline = Compose([])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            print(f"Error in load_process_audio_batch for {audio_path}: {e_lp}")
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
try:
    for split in raw_datasets:
        original_len = len(raw_datasets[split])
        raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")
        if len(raw_datasets[split]) != original_len:
            print(f"Filtered out {original_len - len(raw_datasets[split])} missing paths from {split} split.")
        if len(raw_datasets[split]) == 0 and split == "train": raise ValueError(f"Train split empty after path filtering.")
    print(f"Train size after path filter: {len(raw_datasets['train'])}")
except Exception as e_filt:
    print(f"Path filtering error: {e_filt}")
    raise

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    print(f"Processing {split_name} audio (augment={is_train})...")
    if len(dataset_split) == 0:
        print(f"Skipping processing for empty split: {split_name}")
        processed_splits[split_name] = dataset_split
        continue
    try:
        processed_splits[split_name] = dataset_split.map(
            lambda batch: load_process_audio_batch(batch, do_augment=is_train),
            batched=True, batch_size=8, num_proc=1, # num_proc=1 for Colab stability with audio
            desc=f"Processing {split_name} audio"
        )
        print(f"{split_name} size before input_features filter: {len(processed_splits[split_name])}")
        processed_splits[split_name] = processed_splits[split_name].filter(
            lambda example: example["input_features"] is not None,
            num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
        )
        print(f"{split_name} size after input_features filter: {len(processed_splits[split_name])}")
        if len(processed_splits[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after input_features filter.")
    except Exception as e_proc_split:
        print(f"Error processing {split_name}: {e_proc_split}")
        raise
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()
print("Audio features processed.")
if len(raw_datasets_with_features.get("train", [])) > 0:
    print(f"Columns after feature extraction: {raw_datasets_with_features['train'].features}")

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
try:
    processed_datasets = raw_datasets_with_features.map(
        prepare_dataset_labels, batched=True, batch_size=8,
        remove_columns=["audio_path", "sentence"],
        num_proc=1, desc="Tokenizing text"
    )
    del raw_datasets_with_features; gc.collect()
except Exception as e_tok:
    print(f"Tokenization error: {e_tok}")
    raise

for split_name in processed_datasets:
    print(f"{split_name} size before final combined filter: {len(processed_datasets[split_name])}")
    if len(processed_datasets[split_name]) == 0: continue
    try:
        processed_datasets[split_name] = processed_datasets[split_name].filter(
            lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                       ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
            num_proc=1, desc=f"Final filtering on {split_name}"
        )
        print(f"{split_name} size after final combined filter: {len(processed_datasets[split_name])}")
        if len(processed_datasets[split_name]) == 0 and split_name == "train":
            raise ValueError("Training dataset empty after final combined filter.")
    except Exception as e_filt_final:
        print(f"Final filtering error on {split_name}: {e_filt_final}")
        raise
print("Dataset prepared for Whisper training.")
if len(processed_datasets.get("train",[])) > 0:
    print(f"Processed dataset final columns: {list(processed_datasets['train'].features.keys())}")
    print(f"Sample input_features type: {type(processed_datasets['train'][0]['input_features'])}")
    sample_feat_np = np.asarray(processed_datasets['train'][0]['input_features'])
    print(f"Sample input_features shape: {sample_feat_np.shape}")
    print(f"Sample labels (first 10): {processed_datasets['train'][0]['labels'][:10]}")

print(f"Dataset sizes - Train: {len(processed_datasets.get('train',[]))}, Val: {len(processed_datasets.get('val',[]))}, Test: {len(processed_datasets.get('test',[]))}")
print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
print("--- Defining Seq2Seq Data Collator ---")
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Filter out features where input_features or labels might be None or labels empty
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {} # Return empty dict if no valid features after filtering

        input_features_list = [f["input_features"] for f in valid_features]
        # Pad the input features
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        # Tokenize and pad the label features
        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # If bos token is appendend in previous tokenization, slice it
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
print("Data Collator defined.")
print("-" * 30)

# 5. Define Evaluation Metrics
print("--- Defining Evaluation Metrics ---")
metric = evaluate.load("wer")

def normalize_text(text): # Ensure consistent normalization
    if not isinstance(text, str): text = str(text) # Handle potential non-string inputs
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text) # Keeps apostrophes
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids

    # Replace -100 with pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode predictions and labels
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    # Normalize text
    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    # Print a few examples for qualitative inspection
    for i in range(min(3, len(pred_str))): # Print up to 3 examples
        print(f"Pred [{i}]: {pred_str[i]}")
        print(f"Ref  [{i}]: {label_str[i]}")

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    # Log to WandB
    try:
        # Check if trainer object exists and has state
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer}) # Log WER without step if trainer state not available
    except Exception as e_wandb_log:
        print(f"WandB logging error in compute_metrics: {e_wandb_log}")

    return {"wer": wer}
print("Compute Metrics function defined with WandB logging.")
print("-" * 30)

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
try:
    model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
    model.config.suppress_tokens = []
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
        lora_dropout=0.05,
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    if torch.cuda.is_available(): model.to("cuda")
    print("Whisper model loaded and LoRA configured.")
    print(f"Model is on device: {next(model.parameters()).device}")
except Exception as e_model:
    print(f"Error loading model or applying LoRA: {e_model}")
    raise
print("Checking GPU memory after model loading...")
!nvidia-smi
print("-" * 30)

#  7. Define Training Arguments
print("--- Defining Training Arguments for Seq2Seq + LoRA ---")
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)
print("\nTraining Arguments:")
print(f" -> LR: {training_args.learning_rate}, Max Epochs: {training_args.num_train_epochs}")
eff_bs = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * (torch.cuda.device_count() if torch.cuda.is_available() else 1)
print(f" -> Eff. BS: {eff_bs}, FP16: {training_args.fp16}")
print(f" -> Generation Num Beams for Eval: {training_args.generation_num_beams}")
print(f" -> Early Stopping will be handled by a callback. Weight Decay: {training_args.weight_decay}")
print(f" -> Output Dir: {training_args.output_dir}")
print(f" -> Logging to WandB enabled.")
print("-" * 30)

# 8. Initialize Trainer
print("--- Initializing Seq2Seq Trainer ---")
if len(processed_datasets.get("train", [])) == 0: raise ValueError("Training dataset is empty. Cannot initialize trainer.")
if len(processed_datasets.get("val", [])) == 0: raise ValueError("Validation dataset is empty. Cannot initialize trainer for evaluation.")

early_stopping_patience_value = 5 # Ensure this value is properly set
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)
print(f"Early stopping enabled with patience: {early_stopping_patience_value} evaluations.")

try:
    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=processed_datasets["train"],
        eval_dataset=processed_datasets["val"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[early_stopping_callback]
    )
    print("Trainer initialized.")
except Exception as e_trainer:
    print(f"Error initializing trainer: {e_trainer}")
    raise
print("-" * 30)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Baseline) ---")
final_trainer_state = None
try:
    print("Starting training from scratch with baseline LoRA configuration and WandB logging...")
    torch.cuda.empty_cache()
    gc.collect()
    print("Checking GPU memory before training...")
    !nvidia-smi
    train_result = trainer.train()
    final_trainer_state = trainer.state
    print("\nTraining finished.")
    print(f"Train metrics: {train_result.metrics}")
    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)
    print(f"Training metrics saved to {metrics_path}")

    print("\nSaving final PEFT adapter model (best model based on 'wer')...")
    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    print(f"✅ Final PEFT adapter (final state) saved to {os.path.join(output_dir, 'final_adapter')}")
    processor.save_pretrained(output_dir)
    print(f"✅ Processor saved to {output_dir}")

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None}) # Log best eval WER

except RuntimeError as e_rt:
    if "out of memory" in str(e_rt).lower():
        print("\nCUDA OOM Error! Try reducing LoRA rank 'r', effective batch_size, or target_modules, or using a smaller base model.")
    print(f"Runtime Error during training: {e_rt}")
    !nvidia-smi
    raise
except Exception as e_tr:
    print(f"\nTraining error: {type(e_tr).__name__} - {e_tr}")
    !nvidia-smi
    raise

# Final evaluation using the best model loaded by the Trainer
if final_trainer_state and trainer.state.best_model_checkpoint:
    print(f"\nEvaluating on validation set using the best model from {trainer.state.best_model_checkpoint}...")
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        print(f"Final Evaluation results (best model): {eval_results}")
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        print(f"Evaluation metrics for best model saved to {eval_metrics_path}")
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")
else:
    print("\nSkipping final evaluation as training did not complete successfully or no best model checkpoint found.")
print("-" * 30)

# 10. Generate Transcriptions on Test Set
print("--- Generating Transcriptions for Test Set using PEFT Model ---")
# Use the best model checkpoint identified by the trainer for inference
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_baseline.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    print(f"Loading best PEFT adapter from: {best_checkpoint_dir} for inference...")
    try:
        base_model_name_or_path = model_name_loaded
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32

        base_model = WhisperForConditionalGeneration.from_pretrained(
            base_model_name_or_path, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()

        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir) # Load processor from checkpoint
        print("Best PEFT model and processor loaded successfully for inference.")
    except Exception as e_load:
        print(f"Error loading best PEFT model for inference from {best_checkpoint_dir}: {e_load}.")
else:
    print(f"Best model checkpoint not found or training didn't complete. Check: {best_checkpoint_dir}")


def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]):
        print("Skipping transcription generation (model/processor not loaded or load failed).")
        return
    results = []
    device = inference_model.device
    print(f"Generating transcriptions for {len(test_dataset)} samples on device: {device}...")

    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            # Duration calculation (rough estimate based on features)
            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01 # Approx for Whisper features
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            trans = f"Error: {type(e_inf).__name__} - {str(e_inf)[:100]}"
            print(f"Inference error for one sample: {e_inf}")
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        print(f"✅ Transcriptions saved to {output_csv_path}")
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        # Calculate WER on test set
        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        print(f"Error saving transcriptions CSV or calculating test WER: {e_csv}")

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)
else:
    print("Skipping inference generation on test set (model/processor load failed or test set empty).")
print("-" * 30)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded. Training may have failed or was skipped."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        print(f"Gradio input: {audio_filepath}")
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        print(f" -> Gradio Output: {transcription}")
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        print(f"Gradio Error: {type(e_grad).__name__} - {e_grad}")
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Base)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA (r={lora_config.r}, alpha={lora_config.lora_alpha}, targets={lora_config.target_modules}) & NO Augmentation. Dataset: {wandb.config.num_samples_train} train samples.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        print(f"Error launching Gradio: {e_launch}.")
else:
    print("\nSkipping Gradio launch (model/processor not loaded, possibly due to issues in training or loading).")
print("\n--- Script Finished ---")
wandb.finish()

3. LoRA + Time Stretch Only

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Time Stretch Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-timestretch-only", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "TimeStretch", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (TIME STRETCH ONLY)
print("--- Preparing Dataset for Whisper (Time Stretch Only) ---")

# ONLY TIME STRETCH IN THE PIPELINE (p=1.0 to ensure it applies when do_augment=True)
time_augment_pipeline = Compose([
    TimeStretch(min_rate=0.85, max_rate=1.15, p=1.0, leave_length_unchanged=False)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Time Stretch Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_timestretch.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Time Stretch Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & Time Stretch Augmentation only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

4. LoRA + Pitch Shift Only

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + Pitch Shift Augmentation + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-pitchshift-only",
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "PitchShift",
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (PITCH SHIFT ONLY)
print("--- Preparing Dataset for Whisper (Pitch Shift Only) ---")

# ONLY PITCH SHIFT IN THE PIPELINE (p=1.0)
time_augment_pipeline = Compose([
    PitchShift(min_semitones=-3, max_semitones=3, p=1.0)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.0
spec_augment_probability = 0.0

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (Pitch Shift Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_pitchshift.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA Pitch Shift Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & Pitch Shift Augmentation only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()+9+/9

5. LoRA + SpecAugment Only

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb

In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + SpecAugment Only + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-specaugment-only", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "SpecAugment", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (SPECAUGMENT ONLY)
print("--- Preparing Dataset for Whisper (SpecAugment Only) ---")

# EMPTY THE TIME-DOMAIN PIPELINE
time_augment_pipeline = Compose([])

# KEEP SPECAUGMENT ACTIVE
spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
# SET SPECAUGMENT PROBABILITY TO 0.5 (50% chance of application during training)
spec_augment_probability = 0.5

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            # APPLY SPECAUGMENT ON THE FREQUENCY/TIME DOMAIN
            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (SpecAugment Only) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_specaugment.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA SpecAugment Only)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & SpecAugment Only.",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

6: LoRA + Gaussian Noise + Pitch + TimeStretch + SpecAugment (Ensemble)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb


In [ ]:
# 0. Setup and Imports
print("--- Setting up Environment for Colab Pro (Whisper + PEFT + ALL Augmentations + WandB) ---")

import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import pandas as pd
import torch
import torchvision
import torchaudio
from torchaudio.transforms import SpecAugment
import transformers
import datasets
import accelerate
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor,
    WhisperFeatureExtractor,
    WhisperTokenizerFast,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    PeftModel,
    PeftConfig,
    get_peft_model
)
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
import gradio as gr
import csv
import soundfile as sf
import gc
warnings.filterwarnings("ignore")

# Verify library versions and CUDA
print("\n--- Library Versions & CUDA Status ---")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    torch.cuda.empty_cache()
    gc.collect()
else:
    print("CUDA not available. This script requires a GPU.")
    raise RuntimeError("GPU required for training.")
print("-" * 30)

#  Initialize WandB
print("--- Initializing Weights & Biases ---")
try:
    wandb.login()
    wandb.init(
        project="swahili-asr-whisper-lora",
        name="whisper-small-lora-aug-all-combined", # UPDATED RUN NAME
        config={
            "model_name": "openai/whisper-small",
            "dataset": "Mozilla Common Voice Swahili v12.0",
            "num_samples_total": 9000,
            "num_samples_train": 7200,
            "lora_rank": 32,
            "lora_alpha": 64,
            "lora_target_modules": "q_proj,v_proj,k_proj,out_proj,fc1,fc2",
            "learning_rate": 1e-4,
            "num_epochs": 25,
            "batch_size_effective": 32,
            "augmentation": "GaussianNoise+TimeStretch+PitchShift+SpecAugment", # UPDATED CONFIG
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "audio_validation": "skipped",
            "generation_num_beams_eval": 5
        }
    )
    print("WandB initialized successfully.")
except Exception as e_wandb:
    print(f"Error initializing WandB: {e_wandb}")
    raise

# 1. Prepare Dataset
print("--- Starting Dataset Preparation ---")
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/LoRa_All_Augmentations_Combined"

assert os.path.exists(csv_path), f"Error: CSV file not found at {csv_path}. Upload to Google Drive."
assert os.path.exists(audio_base_path), f"Error: Audio directory not found at {audio_base_path}. Upload to Google Drive."
os.makedirs(output_dir, exist_ok=True)

df = pd.read_csv(csv_path).head(9000)

def update_audio_path(path_val):
    return os.path.join(audio_base_path, os.path.basename(str(path_val)))
df["path"] = df["path"].apply(update_audio_path)

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

print("Skipping audio validation to save time. Assuming all audio files are valid.")
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])
raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()
print(f"Dataset sizes - Train: {len(raw_datasets['train'])}, Val: {len(raw_datasets['val'])}, Test: {len(raw_datasets['test'])}")

model_name_loaded = "openai/whisper-small"
language = "Swahili"; language_abbr = "sw"; task = "transcribe"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
tokenizer = WhisperTokenizerFast.from_pretrained(model_name_loaded, language=language, task=task)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language=language, task=task)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# 3. Prepare Dataset for Whisper (ALL AUGMENTATIONS COMBINED)
print("--- Preparing Dataset for Whisper (All Augmentations Combined) ---")

time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=0.3),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.3, leave_length_unchanged=False),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.3)
])

spec_augment_transform = torchaudio.transforms.SpecAugment(
    n_time_masks=2,
    time_mask_param=40,
    n_freq_masks=2,
    freq_mask_param=15,
)
spec_augment_probability = 0.5

def load_process_audio_batch(batch, do_augment=False):
    input_features_list = []
    SAMPLING_RATE = feature_extractor.sampling_rate
    for audio_path in batch["audio_path"]:
        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != SAMPLING_RATE:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=SAMPLING_RATE)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            if do_augment:
                speech_numpy = time_augment_pipeline(samples=speech_numpy, sample_rate=SAMPLING_RATE)

            features_np_batch = feature_extractor(speech_numpy, sampling_rate=SAMPLING_RATE, return_tensors="np").input_features
            current_features_np = features_np_batch[0]

            if do_augment:
                if torch.rand(1).item() < spec_augment_probability:
                    features_tensor = torch.from_numpy(current_features_np).float()
                    augmented_features_tensor = spec_augment_transform(features_tensor)
                    current_features_np = augmented_features_tensor.numpy()

            input_features_list.append(current_features_np.astype(np.float32))
        except Exception as e_lp:
            input_features_list.append(None)
    batch["input_features"] = input_features_list
    return batch

print("Pre-filtering invalid audio paths...")
def path_exists(example): return os.path.exists(example["audio_path"])
for split in raw_datasets:
    raw_datasets[split] = raw_datasets[split].filter(path_exists, num_proc=1, desc=f"Pre-filtering {split} paths")

print("Loading and processing audio features...")
processed_splits = {}
for split_name, dataset_split in raw_datasets.items():
    is_train = (split_name == "train")
    if len(dataset_split) == 0:
        processed_splits[split_name] = dataset_split
        continue
    processed_splits[split_name] = dataset_split.map(
        lambda batch: load_process_audio_batch(batch, do_augment=is_train),
        batched=True, batch_size=8, num_proc=1,
        desc=f"Processing {split_name} audio"
    )
    processed_splits[split_name] = processed_splits[split_name].filter(
        lambda example: example["input_features"] is not None,
        num_proc=1, desc=f"Filtering invalid input_features in {split_name}"
    )
raw_datasets_with_features = DatasetDict(processed_splits)
del processed_splits, raw_datasets; gc.collect()

def prepare_dataset_labels(batch):
    labels_batch = []
    for sentence_text in batch["sentence"]:
        if not sentence_text or not isinstance(sentence_text, str):
            labels_batch.append([])
        else:
            labels_batch.append(tokenizer(sentence_text, padding=False).input_ids)
    batch["labels"] = labels_batch
    return batch

print("Tokenizing text labels...")
processed_datasets = raw_datasets_with_features.map(
    prepare_dataset_labels, batched=True, batch_size=8,
    remove_columns=["audio_path", "sentence"],
    num_proc=1, desc="Tokenizing text"
)
del raw_datasets_with_features; gc.collect()

for split_name in processed_datasets:
    processed_datasets[split_name] = processed_datasets[split_name].filter(
        lambda ex: ex["input_features"] is not None and isinstance(ex["input_features"], list) and \
                   ex["labels"] is not None and isinstance(ex["labels"], list) and len(ex["labels"]) > 0,
        num_proc=1, desc=f"Final filtering on {split_name}"
    )

print("Checking GPU memory after dataset preparation...")
!nvidia-smi
print("-" * 30)

# 4. Define Data Collator for Seq2Seq
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        valid_features = [f for f in features if f.get("input_features") is not None and \
                                               f.get("labels") is not None and len(f["labels"]) > 0]
        if not valid_features: return {}

        input_features_list = [f["input_features"] for f in valid_features]
        batch = self.processor.feature_extractor.pad({"input_features": input_features_list}, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in valid_features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# 5. Define Evaluation Metrics
metric = evaluate.load("wer")

def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True, group_tokens=False)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True, group_tokens=False)

    pred_str = [normalize_text(s) for s in pred_str]
    label_str = [normalize_text(s) for s in label_str]

    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    try:
        if 'trainer' in globals() and hasattr(trainer, 'state') and trainer.state is not None:
            wandb.log({"eval_wer": wer, "step": trainer.state.global_step})
        else:
            wandb.log({"eval_wer": wer})
    except Exception as e_wandb_log:
        pass

    return {"wer": wer}

# 6. Load Whisper Model and Apply LoRA
print("--- Loading Whisper Model and Applying LoRA ---")
model_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=model_dtype)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)
model.config.suppress_tokens = []
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

if torch.cuda.is_available(): model.to("cuda")

#  7. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    learning_rate=1e-4,
    warmup_steps=200,
    num_train_epochs=25,
    eval_strategy="steps",
    fp16=True,
    save_steps=250,
    eval_steps=250,
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_num_beams=5,
    report_to=["wandb"],
    dataloader_num_workers=2,
    seed=42,
    weight_decay=0.01,
)

# 8. Initialize Trainer
early_stopping_patience_value = 5
early_stopping_callback = EarlyStoppingCallback(early_stopping_patience=early_stopping_patience_value)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_datasets["train"],
    eval_dataset=processed_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
    callbacks=[early_stopping_callback]
)

#  9. Start Training
print("--- Starting Whisper Fine-Tuning with LoRA (All Augmentations Combined) ---")
final_trainer_state = None
try:
    torch.cuda.empty_cache()
    gc.collect()
    train_result = trainer.train()
    final_trainer_state = trainer.state

    metrics_path = os.path.join(output_dir, "train_results.json")
    with open(metrics_path, "w") as f: json.dump(train_result.metrics, f, indent=4)

    model.save_pretrained(os.path.join(output_dir, "final_adapter"))
    processor.save_pretrained(output_dir)

    wandb.log({"final_train_loss": train_result.metrics.get("train_loss", 0),
               "final_train_wer_on_eval_during_train": final_trainer_state.best_metric if final_trainer_state else None})

except RuntimeError as e_rt:
    print(f"Runtime Error during training: {e_rt}")
    raise

# Final evaluation
if final_trainer_state and trainer.state.best_model_checkpoint:
    try:
        eval_results = trainer.evaluate(eval_dataset=processed_datasets["val"])
        eval_metrics_path = os.path.join(output_dir, "eval_results_best_model.json")
        with open(eval_metrics_path, "w") as f: json.dump(eval_results, f, indent=4)
        wandb.log({"final_eval_wer_best_model": eval_results.get("eval_wer", 0)})
    except Exception as e_eval:
        print(f"Evaluation error after training: {e_eval}")

# 10. Generate Transcriptions on Test Set
best_checkpoint_dir = trainer.state.best_model_checkpoint if final_trainer_state and trainer.state.best_model_checkpoint else None
output_csv_file = os.path.join(output_dir, "test_transcriptions_lora_all_aug.csv")

inference_model = None
inference_processor = None

if best_checkpoint_dir and os.path.exists(best_checkpoint_dir):
    try:
        base_model_dtype = torch.float16 if training_args.fp16 and torch.cuda.is_available() else torch.float32
        base_model = WhisperForConditionalGeneration.from_pretrained(
            model_name_loaded, torch_dtype=base_model_dtype
        ).to("cuda" if torch.cuda.is_available() else "cpu")

        inference_model = PeftModel.from_pretrained(base_model, best_checkpoint_dir)
        inference_model.eval()
        inference_processor = WhisperProcessor.from_pretrained(best_checkpoint_dir)
    except Exception as e_load:
        pass

def run_inference_peft(test_dataset: Dataset, output_csv_path: str):
    if not all([inference_model, inference_processor]): return
    results = []
    device = inference_model.device
    generation_config = inference_model.generation_config
    generation_config.num_beams = training_args.generation_num_beams
    generation_config.max_new_tokens = 256

    for example in tqdm(test_dataset, desc="Generating Transcriptions"):
        trans, ref, dur_str = "Error: Transcription Failed", "N/A", "N/A"
        try:
            input_features_np = np.asarray(example["input_features"]).astype(np.float32)
            input_tensor = torch.tensor(input_features_np).unsqueeze(0).to(device)
            input_tensor = input_tensor.to(inference_model.dtype)

            labels_array = np.array(example["labels"])
            labels_np = np.where(labels_array == -100, inference_processor.tokenizer.pad_token_id, labels_array)
            ref = inference_processor.tokenizer.decode(labels_np, skip_special_tokens=True)

            num_frames = input_features_np.shape[1]
            hop_length_sec = feature_extractor.hop_length / feature_extractor.sampling_rate if hasattr(feature_extractor, 'hop_length') else 0.01
            dur_calc = num_frames * hop_length_sec
            dur_str = f"{dur_calc:.3f}"

            with torch.no_grad():
                predicted_ids = inference_model.generate(
                    input_features=input_tensor,
                    forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                    generation_config=generation_config
                )[0]
            trans = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            wandb.log({"sample_transcription_test": trans, "sample_reference_test": ref, "sample_duration_sec_test": dur_calc})
        except Exception as e_inf:
            pass
        results.append({"audio_duration_sec": dur_str, "transcription": trans, "reference": ref})

    try:
        df_results = pd.DataFrame(results)
        df_results.to_csv(output_csv_path, index=False, encoding='utf-8', quoting=csv.QUOTE_ALL)
        wandb.log({"test_transcriptions_table": wandb.Table(dataframe=df_results)})

        if not df_results.empty:
            test_pred_str = [normalize_text(s) for s in df_results["transcription"]]
            test_label_str = [normalize_text(s) for s in df_results["reference"]]
            test_wer = 100 * metric.compute(predictions=test_pred_str, references=test_label_str)
            print(f"WER on Test Set: {test_wer:.2f}%")
            wandb.log({"final_test_set_wer": test_wer})

    except Exception as e_csv:
        pass

if inference_model and inference_processor and len(processed_datasets.get("test", [])) > 0:
    run_inference_peft(processed_datasets["test"], output_csv_file)

# 11. Gradio Interface
print("--- Setting up Gradio Interface ---")
def transcribe_audio_gradio_whisper(audio_filepath):
    if not all([inference_model, inference_processor]): return "Error: Model/Processor not loaded."
    if not audio_filepath or not os.path.exists(audio_filepath): return "Error: Audio file missing or path is invalid."
    try:
        wf, sr = torchaudio.load(audio_filepath)
        if wf.shape[0] > 1: wf = torch.mean(wf, dim=0, keepdim=True)
        if sr != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
            wf = resampler(wf)

        inputs = inference_processor(wf.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(inference_model.device).to(inference_model.dtype)

        generation_config_gradio = inference_model.generation_config
        generation_config_gradio.num_beams = training_args.generation_num_beams
        generation_config_gradio.max_new_tokens = 256

        with torch.no_grad():
            predicted_ids = inference_model.generate(
                input_features=input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=language_abbr, task=task),
                generation_config=generation_config_gradio
            )[0]
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
        wandb.log({"gradio_transcription": transcription, "audio_file_name_gradio": os.path.basename(audio_filepath)})
        return transcription
    except Exception as e_grad:
        return f"Transcription failed due to an error: {type(e_grad).__name__}."

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio_gradio_whisper,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR ({model_name_loaded} + LoRA + ALL Aug)",
        description=f"Upload a Swahili audio file to transcribe. Model: {model_name_loaded} (base), fine-tuned with LoRA & ALL Augmentations Combined (Gaussian, Time Stretch, Pitch Shift, SpecAugment).",
        allow_flagging="never",
        examples=[]
    )
    print("\nLaunching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        pass

print("\n--- Script Finished ---")
wandb.finish()

### Visualize WandB Evaluation WER Trends
This block fetches the logged metrics across all the different augmentation experiments from Weights & Biases and plots them together for comparison.

In [ ]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import userdata

print("Fetching data from WandB...")
try:
    # Retrieve the API key from Colab Secrets and strip any hidden newlines/spaces
    wandb_key = userdata.get('WANDB_API_KEY').strip()

    # Explicitly call login with the key
    wandb.login(key=wandb_key)

    # Initialize the WandB Public API
    api = wandb.Api()

    # Try default entity first, then fallback to the team/org name from the logs
    project = "swahili-asr-whisper-lora"
    entity = api.default_entity

    print(f"Attempting to fetch runs from: {entity}/{project}")
    try:
        runs = api.runs(f"{entity}/{project}")
    except Exception:
        entity = "sairascyrus-laikipia-university"
        print(f"Fallback: Attempting to fetch runs from: {entity}/{project}")
        runs = api.runs(f"{entity}/{project}")

    plt.figure(figsize=(14, 8))
    sns.set_theme(style="whitegrid")

    has_data = False

    # Target run names to include in the plot
    target_runs = [
        "whisper-small-lora-baseline-no-aug",
        "whisper-small-lora-aug-gaussian-only",
        "whisper-small-lora-aug-timestretch-only",
        "whisper-small-lora-aug-pitchshift-only",
        "whisper-small-lora-aug-specaugment-only",
        "whisper-small-lora-aug-all-combined"
    ]

    for run in runs:
        # Skip runs that are not in our target list
        if run.name not in target_runs:
            continue

        # Fetch the step and eval_wer history for each run
        history = run.history(keys=["step", "eval_wer"])
        if not history.empty and "eval_wer" in history.columns and "step" in history.columns:
            history = history.dropna(subset=["step", "eval_wer"])
            if not history.empty:
                has_data = True
                # Plot the trend for this run
                plt.plot(history["step"], history["eval_wer"], label=run.name, marker="o", linewidth=2)

    if has_data:
        # Add a horizontal line for the previous best baseline
        plt.axhline(y=13.32, color='red', linestyle='--', linewidth=2.5, label='Previous Best Baseline (WER: 13.32)')

        plt.title("Evaluation WER Trends across Different Augmentations", fontsize=16)
        plt.xlabel("Training Step", fontsize=14)
        plt.ylabel("Evaluation WER", fontsize=14)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print("No 'eval_wer' data found in the selected runs for this project.")

except userdata.SecretNotFoundError:
    print("WANDB_API_KEY not found in Colab Secrets.")
    print("Please add your WandB API key to the Colab Secrets (the 🔑 icon on the left) with the name 'WANDB_API_KEY'.")
except Exception as e:
    print(f"Failed to fetch data or plot trends: {e}")
    print("Please verify your WandB authentication and project/entity names.")


In [ ]:
import wandb
import pandas as pd
from google.colab import userdata

try:
    wandb_key = userdata.get('WANDB_API_KEY').strip()
    wandb.login(key=wandb_key)
    api = wandb.Api()

    project = "swahili-asr-whisper-lora"
    entity = "sairascyrus-laikipia-university" # Using the fallback entity that worked in the previous cell
    runs = api.runs(f"{entity}/{project}")

    target_runs = [
        "whisper-small-lora-baseline-no-aug",
        "whisper-small-lora-aug-gaussian-only",
        "whisper-small-lora-aug-timestretch-only",
        "whisper-small-lora-aug-pitchshift-only",
        "whisper-small-lora-aug-specaugment-only",
        "whisper-small-lora-aug-all-combined"
    ]

    results = []
    for run in runs:
        if run.name in target_runs:
            history = run.history(keys=["step", "eval_wer"])
            if not history.empty and "eval_wer" in history.columns:
                min_wer = history["eval_wer"].min()
                results.append({"Run Name": run.name, "Best Eval WER": min_wer})

    if results:
        df_results = pd.DataFrame(results).sort_values(by="Best Eval WER")
        display(df_results)
    else:
        print("No eval_wer data found for the target runs.")

except Exception as e:
    print(f"An error occurred: {e}")

## 7. Standalone Gradio Deployment Script

This script loads a fine-tuned PEFT (LoRA) adapter onto the base Whisper model and launches a Gradio web interface for live audio transcription.

In [ ]:
import os
import torch
import gradio as gr
import torchaudio
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor
)
from peft import PeftModel


# --- Configuration --- #
# Path to your saved PEFT adapter (e.g., from the best checkpoint or final save)
PEFT_ADAPTER_PATH = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/final_adapter"
# Name of the base Whisper model used for fine-tuning
BASE_MODEL_NAME = "openai/whisper-small"

# Language and task for the Whisper model
LANGUAGE_ABBR = "sw"
TASK = "transcribe"

# Device and data type for model inference
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

# --- Model Loading Function --- #
def load_inference_model(adapter_path):
    """
    Loads the base Whisper model, applies the fine-tuned PEFT adapter,
    and loads the appropriate processor for inference.
    """
    print(f"Loading base model: {BASE_MODEL_NAME}...")
    try:
        # 1. Load the base model
        base_model = WhisperForConditionalGeneration.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=MODEL_DTYPE
        ).to(DEVICE)

        print(f"Loading PEFT adapter from: {adapter_path}...")
        # 2. Load and apply the LoRA adapter
        inference_model = PeftModel.from_pretrained(base_model, adapter_path)
        inference_model.eval()  # Set the model to evaluation mode

        # 3. Load the processor from the base model name, configured for the language and task
        print(f"Loading processor from base model: {BASE_MODEL_NAME}...")
        processor = WhisperProcessor.from_pretrained(
            BASE_MODEL_NAME,
            language="Swahili",  # Or your specific language
            task="transcribe"
        )

        print("✅ Model and processor loaded successfully for inference.")
        return inference_model, processor

    except Exception as e:
        print(f"❌ Error loading model or processor: {e}")
        print(f"Please ensure the 'PEFT_ADAPTER_PATH' ({adapter_path}) is correct and exists.")
        return None, None

# --- Transcription Function for Gradio --- #
def transcribe_audio(audio_filepath):
    """
    Transcribes an audio file using the loaded Whisper PEFT model.
    """
    if not inference_model or not inference_processor:
        return "Error: Model or processor not loaded. Please check setup steps."

    if not audio_filepath or not os.path.exists(audio_filepath):
        return "Error: Audio file is missing or path is invalid. Please upload a file."

    try:
        # Load and process the audio file
        waveform, sampling_rate = torchaudio.load(audio_filepath)

        # Ensure mono channel and 16kHz sampling rate required by Whisper
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
            waveform = resampler(waveform)

        # Extract features and move to the correct device and dtype
        input_features = inference_processor(
            waveform.squeeze().numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        ).input_features.to(DEVICE).to(MODEL_DTYPE)

        # Generate token IDs
        with torch.no_grad():
            # Using default generation parameters or specify if needed
            predicted_ids = inference_model.generate(
                input_features,
                forced_decoder_ids=inference_processor.get_decoder_prompt_ids(language=LANGUAGE_ABBR, task=TASK),
                num_beams=5, # You can adjust this
                max_new_tokens=256 # Maximum length of generated text
            )[0]

        # Decode the token IDs to text
        transcription = inference_processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)

        print(f"Input: {os.path.basename(audio_filepath)} -> Transcription: {transcription}")
        return transcription

    except Exception as e:
        print(f"An error occurred during transcription: {e}")
        return f"Transcription failed. Error: {type(e).__name__}."

# --- Main Application Logic & Gradio Interface Launch --- #
print("--- Initializing Swahili ASR Gradio App ---")
inference_model, inference_processor = load_inference_model(PEFT_ADAPTER_PATH)

if inference_model and inference_processor:
    interface = gr.Interface(
        fn=transcribe_audio,
        inputs=gr.Audio(type="filepath", label="Upload Swahili Audio File"),
        outputs=gr.Text(label="Transcription"),
        title=f"Swahili ASR with Fine-Tuned Whisper-Small + LoRA",
        description=(
            f"An interactive demo for transcribing Swahili speech to text. "
            f"This app uses the '{BASE_MODEL_NAME}' model fine-tuned with a LoRA adapter "
            f"loaded from '{os.path.basename(PEFT_ADAPTER_PATH)}'."
        ),
        allow_flagging="never",
        examples=[] # Add example audio files here if you have any
    )

    print("\n🚀 Launching Gradio interface...")
    try:
        interface.launch(share=True, debug=False)
    except Exception as e_launch:
        print(f"Error launching Gradio interface: {e_launch}. This might happen if a previous Gradio server is still running.")
else:
    print("\nCould not launch Gradio interface because the model failed to load.")

print("\n--- Gradio Script Finished ---")


Evaluation on FLEURS dataset

In [ ]:
!apt-get install -y ffmpeg
!pip install numpy==1.25.2
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer torchaudio gradio peft==0.14.0

In [ ]:
# CELL 2: INFERENCE AND EVALUATION ON FLEURS (MULTI-MODEL LOOP)

import os
import re
import gc
import torch
import torchaudio
import pandas as pd
from tqdm.notebook import tqdm
import evaluate
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

# ==========================================
# 1. PATH CONFIGURATIONS
# ==========================================
print("--- Configuring Paths ---")
fleurs_base_dir = "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset"
fleurs_csv_path = os.path.join(fleurs_base_dir, "test.tsv")
fleurs_audio_dir = os.path.join(fleurs_base_dir, "test", "test")


adapter_paths = {
    "LoRA_Baseline": "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_Colab_NewRun/LoRA_No_Augmentation/final_adapter",
    "LoRA_Gaussian": "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/Lora_Gaussian/final_adapter",
    "LoRA_Ensemble": "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/LoRa_All_Augmentations_Combined/final_adapter"
}

output_csv = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/fleurs_ablation_evaluation_results.csv"

# ==========================================
# 2. ALIGNED NORMALIZATION FUNCTION
# ==========================================
def normalize_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

metric = evaluate.load("wer")
# ==========================================
# 3. LOAD FLEURS DATASET
# ==========================================
print("--- Loading FLEURS Dataset ---")

# Manually assign column names because the file lacks a header row
col_names = ['id', 'path', 'raw_transcript', 'transcript', 'segmented', 'duration', 'gender']

# Load the TSV file using header=None and our custom names
df_fleurs = pd.read_csv(fleurs_csv_path, sep='\t', header=None, names=col_names)

def build_fleurs_path(filename):
    return os.path.join(fleurs_audio_dir, os.path.basename(str(filename)))

path_column = 'path'
text_column = 'transcript'

df_fleurs["absolute_path"] = df_fleurs[path_column].apply(build_fleurs_path)
print(f"Loaded {len(df_fleurs)} test samples from FLEURS.")

# ==========================================
# 4. LOAD BASE MODEL & PROCESSOR
# ==========================================
print("\n--- Loading Base Whisper Model ---")
model_name = "openai/whisper-small"
language_abbr = "sw"
task = "transcribe"

device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.float16 if torch.cuda.is_available() else "auto"

processor = WhisperProcessor.from_pretrained(model_name, language="Swahili", task=task)
forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_abbr, task=task)

base_model = WhisperForConditionalGeneration.from_pretrained(
    model_name, torch_dtype=model_dtype
).to(device)
base_model.eval()

# ==========================================
# 5. INFERENCE FUNCTION
# ==========================================
def run_inference(model, model_name_tag):
    print(f"\nEvaluating: {model_name_tag}")
    predictions = []

    for _, row in tqdm(df_fleurs.iterrows(), total=len(df_fleurs), desc=f"{model_name_tag}"):
        audio_path = row["absolute_path"]

        try:
            speech_array, sampling_rate = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1:
                speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sampling_rate != 16000:
                resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
                speech_array = resampler(speech_array)

            speech_numpy = speech_array.squeeze().numpy()

            input_features = processor(speech_numpy, sampling_rate=16000, return_tensors="pt").input_features
            input_features = input_features.to(device).to(model.dtype)

            with torch.no_grad():
                predicted_ids = model.generate(
                    input_features=input_features,
                    forced_decoder_ids=forced_decoder_ids,
                    max_new_tokens=256,
                    num_beams=5
                )[0]

            raw_prediction = processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)
            norm_pred = normalize_text(raw_prediction)
            predictions.append(norm_pred)

        except Exception as e:
            print(f"Failed processing {audio_path}: {e}")
            predictions.append("") # Append empty string on failure to keep alignment

    return predictions

# ==========================================
# 6. EXECUTE LOOP & CALCULATE WER
# ==========================================
# Initialize the final dataframe with references
df_final = df_fleurs[["absolute_path", text_column]].copy()
df_final["norm_reference"] = df_final[text_column].apply(normalize_text)

wer_results = {}

# A. Evaluate Base Model First
base_preds = run_inference(base_model, "Base Model (Zero-Shot)")
df_final["Base_Pred"] = base_preds
wer_base = 100 * metric.compute(predictions=base_preds, references=df_final["norm_reference"].tolist())
wer_results["Base Model"] = wer_base
print(f"--> Base Model WER: {wer_base:.2f}%")

# B. Loop Through Adapters
for adapter_name, adapter_path in adapter_paths.items():
    if not os.path.exists(adapter_path):
        print(f"Skipping {adapter_name} - Path not found: {adapter_path}")
        continue

    print(f"\n--- Loading Adapter: {adapter_name} ---")
    ft_model = PeftModel.from_pretrained(base_model, adapter_path)
    ft_model.eval()

    # Run Inference
    preds = run_inference(ft_model, adapter_name)
    df_final[f"{adapter_name}_Pred"] = preds

    # Calculate WER
    wer = 100 * metric.compute(predictions=preds, references=df_final["norm_reference"].tolist())
    wer_results[adapter_name] = wer
    print(f"--> {adapter_name} WER: {wer:.2f}%")

    # Extreme Memory Cleanup for Colab
    del ft_model
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 7. SAVE & SUMMARIZE
# ==========================================
print("\n=====================================")
print("FINAL FLEURS EVALUATION SUMMARY")
print("=====================================")
for name, score in wer_results.items():
    print(f"{name:<20}: {score:.2f}%")

df_final.to_csv(output_csv, index=False)
print(f"\nAll predictions saved side-by-side to {output_csv}")

Zero-Shot Evaluation of openai/whisper-small on Local Mozilla Common Voice

In [ ]:
!apt-get install -y ffmpeg
!pip install numpy==1.25.2
!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.45.2
!pip install pandas==2.2.2 evaluate jiwer torchaudio gradio peft==0.14.0

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Zero-Shot Evaluation of openai/whisper-small on Local Mozilla Common Voice (CSV format)
"""

import os
import re
import torch
import torchaudio
import pandas as pd
import evaluate
from tqdm.notebook import tqdm
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# ==========================================
# 1. CONFIGURATION & EXACT PATHS
# ==========================================
print("--- Configuring Paths ---")

# Exact paths provided
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"

output_csv = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/zero_shot_mozilla_csv_results.csv"

# Verify paths exist before starting
assert os.path.exists(csv_path), f"Could not find transcript file at {csv_path}"
assert os.path.exists(audio_base_path), f"Could not find audio folder at {audio_base_path}"

MODEL_NAME = "openai/whisper-small"
LANGUAGE_CODE = "sw"
TASK = "transcribe"

device = "cuda" if torch.cuda.is_available() else "cpu"
model_dtype = torch.float16 if torch.cuda.is_available() else "auto"

# ==========================================
# 2. TEXT NORMALIZATION
# ==========================================
def normalize_text(text):
    """Standardizes text for fair WER calculation (lowercase, remove punctuation)."""
    if not isinstance(text, str): text = str(text)
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", '', text) # Remove punctuation except apostrophes
    text = re.sub(r'\s+', ' ', text)     # Remove extra spaces
    return text.strip()

metric = evaluate.load("wer")

# ==========================================
# 3. LOAD LOCAL DATASET (EXACT 900 TEST FILES)
# ==========================================
print("--- Loading Local CSV Dataset ---")
df_cv = pd.read_csv(csv_path)

# 1. Grab the 9,000 file subset you used for the project
df_subset = df_cv.head(9000)

# 2. Isolate the final 10% (900 files) used for the test split
df_cv = df_subset.iloc[8100:9000].copy()

text_column = 'sentence' if 'sentence' in df_cv.columns else 'transcript'
path_column = 'path' if 'path' in df_cv.columns else 'audio_path'

# Ensure the script knows how to find the full audio path
def build_cv_path(filename):
    # Ensure .mp3 extension exists to prevent file-not-found errors
    if not str(filename).endswith(".mp3") and not str(filename).endswith(".wav"):
        filename = str(filename) + ".mp3"
    return os.path.join(audio_base_path, os.path.basename(str(filename)))

df_cv["absolute_path"] = df_cv[path_column].apply(build_cv_path)
print(f"Loaded {len(df_cv)} test samples from local drive.")

# ==========================================
# 4. LOAD MODEL & PROCESSOR
# ==========================================
print(f"\n--- Loading Base Model: {MODEL_NAME} ---")
processor = WhisperProcessor.from_pretrained(MODEL_NAME, language="Swahili", task=TASK)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=model_dtype
).to(device)

model.eval() # Strictly set to evaluation mode (no training)
forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE_CODE, task=TASK)

# ==========================================
# ==========================================
# 5. INFERENCE LOOP (CRASH-PROOFED)
# ==========================================
print("\n--- Starting Zero-Shot Inference ---")
results = []

for _, row in tqdm(df_cv.iterrows(), total=len(df_cv), desc="Evaluating Zero-Shot"):
    audio_path = row["absolute_path"]
    raw_reference = row[text_column]

    try:
        # Load audio
        speech_array, sampling_rate = torchaudio.load(audio_path)

        # ---> SAFETY SHIELD 1: Check if file is empty or corrupted <---
        if speech_array.numel() == 0 or speech_array.shape[1] < 1600: # Less than 0.1 seconds
            raise ValueError("Audio is empty, corrupted, or too short.")

        if speech_array.shape[0] > 1:
            speech_array = torch.mean(speech_array, dim=0, keepdim=True) # convert stereo to mono
        if sampling_rate != 16000:
            resampler = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=16000)
            speech_array = resampler(speech_array)

        speech_numpy = speech_array.squeeze().numpy()

        # Process audio features
        input_features = processor(speech_numpy, sampling_rate=16000, return_tensors="pt").input_features
        input_features = input_features.to(device).to(model_dtype)

        # Generate Prediction
        with torch.no_grad():
            predicted_ids = model.generate(
                input_features=input_features,
                forced_decoder_ids=forced_decoder_ids,
                max_new_tokens=256,
                num_beams=5
            )[0]

        raw_prediction = processor.tokenizer.decode(predicted_ids, skip_special_tokens=True)

        norm_ref = normalize_text(raw_reference)
        norm_pred = normalize_text(raw_prediction)

        results.append({
            "audio_file": os.path.basename(audio_path),
            "raw_reference": raw_reference,
            "norm_reference": norm_ref,
            "raw_prediction": raw_prediction,
            "norm_prediction": norm_pred
        })

    except Exception as e:
        # ---> SAFETY SHIELD 2: Catch errors gracefully and skip <---
        print(f"Skipping {os.path.basename(audio_path)} - Error: {e}")

    finally:
        # Clean up GPU memory periodically to prevent standard Out-Of-Memory errors
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# ==========================================
# 6. CALCULATE WER & SAVE
# ==========================================
df_results = pd.DataFrame(results)

if not df_results.empty:
    wer = 100 * metric.compute(predictions=df_results["norm_prediction"].tolist(),
                               references=df_results["norm_reference"].tolist())

    print("\n" + "="*50)
    print(f"ZERO-SHOT EVALUATION SUMMARY")
    print("="*50)
    print(f"Model          : {MODEL_NAME}")
    print(f"Dataset        : Local Mozilla Common Voice (CSV)")
    print(f"Total Samples  : {len(df_results)}")
    print(f"Word Error Rate: {wer:.2f}%")
    print("="*50)

    df_results.to_csv(output_csv, index=False)
    print(f"\nSaved side-by-side predictions to: {output_csv}")
else:
    print("No successful inferences. Could not calculate WER.")

Multiple Seeds Experiment

In [ ]:
!apt-get install -y ffmpeg
!pip uninstall -y fastai thinc spacy
!pip install numpy==1.25.2
!pip install torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121

!pip install transformers==4.51.1 datasets==3.5.0 accelerate==1.3.0 bitsandbytes==0.44.1
!pip install pandas==2.2.2 evaluate jiwer sentencepiece librosa num2words soundfile gradio
!pip install peft==0.14.0 audiomentations==0.36.0
!pip uninstall -y wandb
!pip install -U wandb

In [ ]:
# CELL 2: MULTI-SEED DEPLOYED ENSEMBLE

print("--- Setting up Environment for Colab Pro (Multi-Seed Ensemble) ---")
import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import scipy.stats as stats
import pandas as pd
import torch
import torchaudio
import gc
import random
import csv
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor, WhisperFeatureExtractor, WhisperTokenizerFast,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments,
    Seq2SeqTrainer, EarlyStoppingCallback, set_seed
)
from peft import prepare_model_for_kbit_training, LoraConfig, PeftModel, get_peft_model
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION & SEEDS
# ==========================================
SEEDS = [42, 1024, 2024, 8888, 12345]
csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
base_output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/LoRa_All_Augmentations_Combined_MultiSeed"

os.makedirs(base_output_dir, exist_ok=True)
results_csv = os.path.join(base_output_dir, "ensemble_multiseed_results.csv")
wers = []

# ==========================================
# 2. LOAD DATASET (ONCE)
# ==========================================
print("--- Loading Raw Dataset ---")
df = pd.read_csv(csv_path).head(9000)
df["path"] = df["path"].apply(lambda x: os.path.join(audio_base_path, os.path.basename(str(x))))

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

# Standard project split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])

raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()

model_name_loaded = "openai/whisper-small"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language="Swahili", task="transcribe")
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# ==========================================
# 3. PRE-PROCESS EVAL & TEST SPLITS (STATIC)
# ==========================================
print("--- Pre-Processing Validation and Test Splits (Static) ---")
def process_static_audio(batch):
    input_features_list = []
    labels_batch = []
    for audio_path, text in zip(batch["audio_path"], batch["sentence"]):
        try:
            speech_array, sr = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sr != 16000:
                speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)

            features = feature_extractor(speech_array.squeeze().numpy(), sampling_rate=16000, return_tensors="np").input_features[0]
            input_features_list.append(features)
            labels_batch.append(processor.tokenizer(text, padding=False).input_ids)
        except:
            input_features_list.append(None)
            labels_batch.append(None)
    batch["input_features"] = input_features_list
    batch["labels"] = labels_batch
    return batch

val_dataset = raw_datasets["val"].map(process_static_audio, batched=True, batch_size=8).filter(lambda x: x["input_features"] is not None)
test_dataset = raw_datasets["test"].map(process_static_audio, batched=True, batch_size=8).filter(lambda x: x["input_features"] is not None)

# Base pipelines
time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=0.3),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.3, leave_length_unchanged=False),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.3)
])

# Evaluation metrics and collator
metric = evaluate.load("wer")
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = [normalize_text(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_text(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features):
        valid = [f for f in features if f.get("input_features") is not None and f.get("labels") is not None]
        if not valid: return {}
        batch = self.processor.feature_extractor.pad({"input_features": [f["input_features"] for f in valid]}, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in valid], return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all(): labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# ==========================================
# 4. MULTI-SEED EVALUATION LOOP
# ==========================================
for i, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"🚀 RUNNING ENSEMBLE SEED {i+1}/{len(SEEDS)}: {seed}")
    print(f"{'='*60}")

    # A. STRICT SEEDING (Changes initialization, shuffling, and augmentations)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    # B. DYNAMIC AUGMENTATION FOR THIS SEED
    print("Applying seed-specific augmentations to Train split...")
    def process_train_audio_augmented(batch):
        input_features_list = []
        labels_batch = []
        for audio_path, text in zip(batch["audio_path"], batch["sentence"]):
            try:
                speech_array, sr = torchaudio.load(audio_path)
                if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
                if sr != 16000:
                    speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)

                # Apply Unique Random Augmentation based on the current seed
                speech_numpy = time_augment_pipeline(samples=speech_array.squeeze().numpy(), sample_rate=16000)

                features = feature_extractor(speech_numpy, sampling_rate=16000, return_tensors="np").input_features[0]
                input_features_list.append(features)
                labels_batch.append(processor.tokenizer(text, padding=False).input_ids)
            except:
                input_features_list.append(None)
                labels_batch.append(None)
        batch["input_features"] = input_features_list
        batch["labels"] = labels_batch
        return batch

    train_dataset = raw_datasets["train"].map(process_train_audio_augmented, batched=True, batch_size=8, desc=f"Augmenting Train (Seed {seed})")
    train_dataset = train_dataset.filter(lambda x: x["input_features"] is not None)

    # C. INIT WANDB FOR THIS SEED
    wandb.init(
        project="swahili-asr-whisper-lora",
        name=f"ensemble-run-seed-{seed}",
        reinit=True
    )

    # D. INITIALIZE FRESH BASE MODEL & LORA
    print("Initializing LoRA adapters...")
    model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=torch.float16)
    model.config.use_cache = False

    # ---------------------------------------------------------
    # NEW PATCH: STRICT GENERATION CONFIG TO STOP HALLUCINATIONS
    # ---------------------------------------------------------
    model.generation_config.language = "sw"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="sw", task="transcribe")
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.eos_token_id
    model.generation_config.suppress_tokens = []
    # ---------------------------------------------------------

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"], lora_dropout=0.05, bias="none")
    model = get_peft_model(model, lora_config)
    if torch.cuda.is_available(): model.to("cuda")

    # E. TRAINER SETUP
    seed_output_dir = os.path.join(base_output_dir, f"run_seed_{seed}")
    training_args = Seq2SeqTrainingArguments(
        output_dir=seed_output_dir,
        per_device_train_batch_size=8,        # Optimized batch size
        gradient_accumulation_steps=4,        # Effective batch size = 32
        learning_rate=1e-4,
        warmup_steps=200,
        num_train_epochs=25,
        eval_strategy="steps",
        fp16=True,
        save_steps=250,
        eval_steps=250,
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_num_beams=5,
        report_to=["wandb"],
        seed=seed,       # Data shuffling changes per seed
        data_seed=seed   # Dataloader seed changes
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor),
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )

    # F. TRAIN AND EVALUATE
    print(f"Starting Training for Seed {seed}...")
    trainer.train()

    # Test Evaluation
    eval_results = trainer.evaluate(eval_dataset=test_dataset)
    current_wer = eval_results.get("eval_wer", 100.0)
    print(f"✅ Final Test WER for Seed {seed}: {current_wer:.2f}%")

    wers.append({"seed": seed, "wer": current_wer})
    pd.DataFrame(wers).to_csv(results_csv, index=False)

    wandb.finish()

    # EXTREME MEMORY CLEANUP
    del model, trainer, train_dataset
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 5. FINAL STATISTICAL CALCULATION
# ==========================================
wer_values = [run["wer"] for run in wers]
n = len(wer_values)
mean_wer = np.mean(wer_values)
sd_wer = np.std(wer_values, ddof=1)
ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=mean_wer, scale=sd_wer/np.sqrt(n))

print("\n" + "="*60)
print(f"📊 FINAL STATISTICAL REPORT (n={n} seeds) - ENSEMBLE")
print("="*60)
print(f"Mean WER:           {mean_wer:.2f}%")
print(f"Standard Deviation: {sd_wer:.3f}")
print(f"95% CI bounds:      [{ci_lower:.2f}%, {ci_upper:.2f}%]")
print(f"Format for Thesis:  {mean_wer:.2f} ± {sd_wer:.2f} [{ci_lower:.2f}%, {ci_upper:.2f}%]")
print("="*60)

In [ ]:
# CELL 2: RESUME MULTI-SEED DEPLOYED ENSEMBLE (REMAINING SEEDS ONLY)

print("--- Setting up Environment to Resume (Seeds 3-5) ---")
import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import scipy.stats as stats
import pandas as pd
import torch
import torchaudio
import gc
import random
import csv
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor, WhisperFeatureExtractor, WhisperTokenizerFast,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments,
    Seq2SeqTrainer, EarlyStoppingCallback, set_seed
)
from peft import prepare_model_for_kbit_training, LoraConfig, PeftModel, get_peft_model
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION & SEEDS (REMAINING ONLY)
# ==========================================
# Only the remaining seeds: 3, 4, and 5
REMAINING_SEEDS = [2024, 8888, 12345]

csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
base_output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/LoRa_All_Augmentations_Combined_MultiSeed"

os.makedirs(base_output_dir, exist_ok=True)
results_csv = os.path.join(base_output_dir, "ensemble_multiseed_results.csv")

# LOAD PREVIOUS PROGRESS
if os.path.exists(results_csv):
    print(f"Loading previous results from {results_csv}...")
    wers = pd.read_csv(results_csv).to_dict('records')
    print(f"Recovered {len(wers)} previous runs!")
else:
    print("Warning: Could not find previous results CSV. Starting fresh list.")
    wers = []

# ==========================================
# 2. LOAD DATASET (ONCE)
# ==========================================
print("--- Loading Raw Dataset ---")
df = pd.read_csv(csv_path).head(9000)
df["path"] = df["path"].apply(lambda x: os.path.join(audio_base_path, os.path.basename(str(x))))

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

# Standard project split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])

raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()

model_name_loaded = "openai/whisper-small"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language="Swahili", task="transcribe")
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# ==========================================
# 3. PRE-PROCESS EVAL & TEST SPLITS (STATIC)
# ==========================================
print("--- Pre-Processing Validation and Test Splits (Static) ---")
def process_static_audio(batch):
    input_features_list = []
    labels_batch = []
    for audio_path, text in zip(batch["audio_path"], batch["sentence"]):
        try:
            speech_array, sr = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sr != 16000:
                speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)

            features = feature_extractor(speech_array.squeeze().numpy(), sampling_rate=16000, return_tensors="np").input_features[0]
            input_features_list.append(features)
            labels_batch.append(processor.tokenizer(text, padding=False).input_ids)
        except:
            input_features_list.append(None)
            labels_batch.append(None)
    batch["input_features"] = input_features_list
    batch["labels"] = labels_batch
    return batch

val_dataset = raw_datasets["val"].map(process_static_audio, batched=True, batch_size=8).filter(lambda x: x["input_features"] is not None)
test_dataset = raw_datasets["test"].map(process_static_audio, batched=True, batch_size=8).filter(lambda x: x["input_features"] is not None)

# Base pipelines
time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=0.3),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.3, leave_length_unchanged=False),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.3)
])

# Evaluation metrics and collator
metric = evaluate.load("wer")
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = [normalize_text(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_text(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features):
        valid = [f for f in features if f.get("input_features") is not None and f.get("labels") is not None]
        if not valid: return {}
        batch = self.processor.feature_extractor.pad({"input_features": [f["input_features"] for f in valid]}, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in valid], return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all(): labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# ==========================================
# 4. MULTI-SEED EVALUATION LOOP (REMAINING)
# ==========================================
for i, seed in enumerate(REMAINING_SEEDS):
    print(f"\n{'='*60}")
    print(f"🚀 RUNNING ENSEMBLE SEED {i+3}/5: {seed}") # Adjusted print statement
    print(f"{'='*60}")

    # A. STRICT SEEDING (Changes initialization, shuffling, and augmentations)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    # B. DYNAMIC AUGMENTATION FOR THIS SEED
    print("Applying seed-specific augmentations to Train split...")
    def process_train_audio_augmented(batch):
        input_features_list = []
        labels_batch = []
        for audio_path, text in zip(batch["audio_path"], batch["sentence"]):
            try:
                speech_array, sr = torchaudio.load(audio_path)
                if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
                if sr != 16000:
                    speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)

                # Apply Unique Random Augmentation based on the current seed
                speech_numpy = time_augment_pipeline(samples=speech_array.squeeze().numpy(), sample_rate=16000)

                features = feature_extractor(speech_numpy, sampling_rate=16000, return_tensors="np").input_features[0]
                input_features_list.append(features)
                labels_batch.append(processor.tokenizer(text, padding=False).input_ids)
            except:
                input_features_list.append(None)
                labels_batch.append(None)
        batch["input_features"] = input_features_list
        batch["labels"] = labels_batch
        return batch

    train_dataset = raw_datasets["train"].map(process_train_audio_augmented, batched=True, batch_size=8, desc=f"Augmenting Train (Seed {seed})")
    train_dataset = train_dataset.filter(lambda x: x["input_features"] is not None)

    # C. INIT WANDB FOR THIS SEED
    wandb.init(
        project="swahili-asr-whisper-lora",
        name=f"ensemble-run-seed-{seed}",
        reinit=True
    )

    # D. INITIALIZE FRESH BASE MODEL & LORA
    print("Initializing LoRA adapters...")
    model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=torch.float16)
    model.config.use_cache = False

    model.generation_config.language = "sw"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="sw", task="transcribe")
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.eos_token_id
    model.generation_config.suppress_tokens = []
    # ---------------------------------------------------------

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"], lora_dropout=0.05, bias="none")
    model = get_peft_model(model, lora_config)
    if torch.cuda.is_available(): model.to("cuda")

    # E. TRAINER SETUP
    seed_output_dir = os.path.join(base_output_dir, f"run_seed_{seed}")
    training_args = Seq2SeqTrainingArguments(
        output_dir=seed_output_dir,
        per_device_train_batch_size=8,        # Optimized batch size
        gradient_accumulation_steps=4,        # Effective batch size = 32
        learning_rate=1e-4,
        warmup_steps=200,
        num_train_epochs=25,
        eval_strategy="steps",
        fp16=True,
        save_steps=250,
        eval_steps=250,
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_num_beams=5,
        report_to=["wandb"],
        seed=seed,       # Data shuffling changes per seed
        data_seed=seed   # Dataloader seed changes
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor),
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )

    # F. TRAIN AND EVALUATE
    print(f"Starting Training for Seed {seed}...")
    trainer.train()

    # Test Evaluation
    eval_results = trainer.evaluate(eval_dataset=test_dataset)
    current_wer = eval_results.get("eval_wer", 100.0)
    print(f"✅ Final Test WER for Seed {seed}: {current_wer:.2f}%")

    wers.append({"seed": seed, "wer": current_wer})
    # Overwrite the CSV with the updated list (including the older seeds)
    pd.DataFrame(wers).to_csv(results_csv, index=False)

    wandb.finish()

    # EXTREME MEMORY CLEANUP
    del model, trainer, train_dataset
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 5. FINAL STATISTICAL CALCULATION
# ==========================================
wer_values = [run["wer"] for run in wers]
n = len(wer_values)

if n > 1:
    mean_wer = np.mean(wer_values)
    sd_wer = np.std(wer_values, ddof=1)
    ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=mean_wer, scale=sd_wer/np.sqrt(n))

    print("\n" + "="*60)
    print(f"📊 FINAL STATISTICAL REPORT (n={n} seeds) - ENSEMBLE")
    print("="*60)
    print(f"Mean WER:           {mean_wer:.2f}%")
    print(f"Standard Deviation: {sd_wer:.3f}")
    print(f"95% CI bounds:      [{ci_lower:.2f}%, {ci_upper:.2f}%]")
    print(f":  {mean_wer:.2f} ± {sd_wer:.2f} [{ci_lower:.2f}%, {ci_upper:.2f}%]")
    print("="*60)
else:
    print("\nStatistical Report requires more than 1 run. Check your CSV results.")

In [ ]:
# CELL 2: FINAL SEED EXECUTION & STATISTICAL REPORT

print("--- Setting up Environment for Final Seed (12345) ---")
import os
os.environ["PYTHONBREAKPOINT"] = "0"
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

import json
import re
import numpy as np
import scipy.stats as stats
import pandas as pd
import torch
import torchaudio
import gc
import random
import csv
import evaluate
import wandb
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from transformers import (
    WhisperProcessor, WhisperFeatureExtractor, WhisperTokenizerFast,
    WhisperForConditionalGeneration, Seq2SeqTrainingArguments,
    Seq2SeqTrainer, EarlyStoppingCallback, set_seed
)
from peft import prepare_model_for_kbit_training, LoraConfig, PeftModel, get_peft_model
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import warnings
warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURATION & FINAL SEED
# ==========================================
# ONLY the very last seed
FINAL_SEED = [12345]

csv_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
audio_base_path = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
base_output_dir = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/LoRa_All_Augmentations_Combined_MultiSeed"

os.makedirs(base_output_dir, exist_ok=True)
results_csv = os.path.join(base_output_dir, "ensemble_multiseed_results.csv")

# LOAD PREVIOUS PROGRESS (Seeds 42, 1024, 2024, 8888)
if os.path.exists(results_csv):
    print(f"Loading previous results from {results_csv}...")
    wers = pd.read_csv(results_csv).to_dict('records')
    print(f"Recovered {len(wers)} previous runs!")
else:
    print("Warning: Could not find previous results CSV. The final stats will only reflect this single run.")
    wers = []

# ==========================================
# 2. LOAD DATASET (ONCE)
# ==========================================
print("--- Loading Raw Dataset ---")
df = pd.read_csv(csv_path).head(9000)
df["path"] = df["path"].apply(lambda x: os.path.join(audio_base_path, os.path.basename(str(x))))

def clean_transcript_whisper(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

df["transcript"] = df["transcript"].apply(clean_transcript_whisper)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]

# Standard project split
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, temp_df; gc.collect()

def create_hf_dataset(df_split):
    return Dataset.from_list([{"audio_path": r["path"], "sentence": r["transcript"]} for _, r in df_split.iterrows()])

raw_datasets = DatasetDict({
    "train": create_hf_dataset(train_df),
    "val": create_hf_dataset(val_df),
    "test": create_hf_dataset(test_df)
})
del train_df, val_df, test_df; gc.collect()

model_name_loaded = "openai/whisper-small"
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_name_loaded)
processor = WhisperProcessor.from_pretrained(model_name_loaded, language="Swahili", task="transcribe")
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

# ==========================================
# 3. PRE-PROCESS EVAL & TEST SPLITS (STATIC)
# ==========================================
print("--- Pre-Processing Validation and Test Splits (Static) ---")
def process_static_audio(batch):
    input_features_list = []
    labels_batch = []
    for audio_path, text in zip(batch["audio_path"], batch["sentence"]):
        try:
            speech_array, sr = torchaudio.load(audio_path)
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sr != 16000:
                speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)

            features = feature_extractor(speech_array.squeeze().numpy(), sampling_rate=16000, return_tensors="np").input_features[0]
            input_features_list.append(features)
            labels_batch.append(processor.tokenizer(text, padding=False).input_ids)
        except:
            input_features_list.append(None)
            labels_batch.append(None)
    batch["input_features"] = input_features_list
    batch["labels"] = labels_batch
    return batch

val_dataset = raw_datasets["val"].map(process_static_audio, batched=True, batch_size=8).filter(lambda x: x["input_features"] is not None)
test_dataset = raw_datasets["test"].map(process_static_audio, batched=True, batch_size=8).filter(lambda x: x["input_features"] is not None)

# Base pipelines
time_augment_pipeline = Compose([
    AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.010, p=0.3),
    TimeStretch(min_rate=0.85, max_rate=1.15, p=0.3, leave_length_unchanged=False),
    PitchShift(min_semitones=-3, max_semitones=3, p=0.3)
])

# Evaluation metrics and collator
metric = evaluate.load("wer")
def normalize_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = [normalize_text(s) for s in processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)]
    label_str = [normalize_text(s) for s in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]
    return {"wer": 100 * metric.compute(predictions=pred_str, references=label_str)}

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    def __call__(self, features):
        valid = [f for f in features if f.get("input_features") is not None and f.get("labels") is not None]
        if not valid: return {}
        batch = self.processor.feature_extractor.pad({"input_features": [f["input_features"] for f in valid]}, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in valid], return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all(): labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# ==========================================
# 4. FINAL SEED EXECUTION
# ==========================================
for seed in FINAL_SEED:
    print(f"\n{'='*60}")
    print(f"🚀 RUNNING FINAL ENSEMBLE SEED 5/5: {seed}")
    print(f"{'='*60}")

    # A. STRICT SEEDING
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    # B. DYNAMIC AUGMENTATION FOR THIS SEED
    print("Applying seed-specific augmentations to Train split...")
    def process_train_audio_augmented(batch):
        input_features_list = []
        labels_batch = []
        for audio_path, text in zip(batch["audio_path"], batch["sentence"]):
            try:
                speech_array, sr = torchaudio.load(audio_path)
                if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
                if sr != 16000:
                    speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)

                # Apply Unique Random Augmentation based on the current seed
                speech_numpy = time_augment_pipeline(samples=speech_array.squeeze().numpy(), sample_rate=16000)

                features = feature_extractor(speech_numpy, sampling_rate=16000, return_tensors="np").input_features[0]
                input_features_list.append(features)
                labels_batch.append(processor.tokenizer(text, padding=False).input_ids)
            except:
                input_features_list.append(None)
                labels_batch.append(None)
        batch["input_features"] = input_features_list
        batch["labels"] = labels_batch
        return batch

    train_dataset = raw_datasets["train"].map(process_train_audio_augmented, batched=True, batch_size=8, desc=f"Augmenting Train (Seed {seed})")
    train_dataset = train_dataset.filter(lambda x: x["input_features"] is not None)

    # C. INIT WANDB FOR THIS SEED
    wandb.init(
        project="swahili-asr-whisper-lora",
        name=f"ensemble-run-seed-{seed}",
        reinit=True
    )

    # D. INITIALIZE FRESH BASE MODEL & LORA
    print("Initializing LoRA adapters...")
    model = WhisperForConditionalGeneration.from_pretrained(model_name_loaded, torch_dtype=torch.float16)
    model.config.use_cache = False

    model.generation_config.language = "sw"
    model.generation_config.task = "transcribe"
    model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="sw", task="transcribe")
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.eos_token_id
    model.generation_config.suppress_tokens = []
    # ---------------------------------------------------------

    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora_config = LoraConfig(r=32, lora_alpha=64, target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"], lora_dropout=0.05, bias="none")
    model = get_peft_model(model, lora_config)
    if torch.cuda.is_available(): model.to("cuda")

    # E. TRAINER SETUP
    seed_output_dir = os.path.join(base_output_dir, f"run_seed_{seed}")
    training_args = Seq2SeqTrainingArguments(
        output_dir=seed_output_dir,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=1e-4,
        warmup_steps=200,
        num_train_epochs=25,
        eval_strategy="steps",
        fp16=True,
        save_steps=250,
        eval_steps=250,
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        predict_with_generate=True,
        generation_num_beams=5,
        report_to=["wandb"],
        seed=seed,
        data_seed=seed
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor),
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=5)]
    )

    # F. TRAIN AND EVALUATE
    print(f"Starting Training for Final Seed {seed}...")
    trainer.train()

    # Test Evaluation
    eval_results = trainer.evaluate(eval_dataset=test_dataset)
    current_wer = eval_results.get("eval_wer", 100.0)
    print(f"✅ Final Test WER for Seed {seed}: {current_wer:.2f}%")

    wers.append({"seed": seed, "wer": current_wer})

    # Overwrite the CSV with the COMPLETE list of all 5 runs
    pd.DataFrame(wers).to_csv(results_csv, index=False)

    wandb.finish()

    # EXTREME MEMORY CLEANUP
    del model, trainer, train_dataset
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 5. FINAL STATISTICAL CALCULATION
# ==========================================
wer_values = [run["wer"] for run in wers]
n = len(wer_values)

if n > 1:
    mean_wer = np.mean(wer_values)
    sd_wer = np.std(wer_values, ddof=1)
    ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=mean_wer, scale=sd_wer/np.sqrt(n))

    print("\n" + "="*60)
    print(f"📊 FINAL STATISTICAL REPORT (n={n} seeds) - ENSEMBLE")
    print("="*60)
    print(f"Mean WER:           {mean_wer:.2f}%")
    print(f"Standard Deviation: {sd_wer:.3f}")
    print(f"95% CI bounds:      [{ci_lower:.2f}%, {ci_upper:.2f}%]")
    print(f"Format for Thesis:  {mean_wer:.2f} ± {sd_wer:.2f} [{ci_lower:.2f}%, {ci_upper:.2f}%]")
    print("="*60)

    print("\n✅ You are officially done with the multi-seed experiment!")
else:
    print("\nStatistical Report requires more than 1 run. Check your CSV results.")

In [ ]:
# CELL: DIAGNOSTIC RESCORE FOR SEED 8888

import os
import re
import gc
import numpy as np
import pandas as pd
import torch
import torchaudio
import evaluate
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

print("--- Starting Diagnostic Rescore for Seed 8888 ---")

# ==========================================
# 1. PATHS & CONFIGURATION
# ==========================================
# Pointing directly to your Seed 8888 saved adapter
ADAPTER_PATH = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/LoRa_All_Augmentations_Combined_MultiSeed/run_seed_8888/checkpoint-3250"
CSV_PATH = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
AUDIO_BASE_PATH = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
BATCH_SIZE = 8
MAX_NEW_TOKENS = 225

# ==========================================
# 2. RECREATE EXACT TEST SPLIT
# ==========================================
print("Reconstructing exact 900-file test split...")
df = pd.read_csv(CSV_PATH).head(9000)

def clean_transcript(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

df["transcript"] = df["transcript"].apply(clean_transcript)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]
df["absolute_path"] = df["path"].apply(lambda x: os.path.join(AUDIO_BASE_PATH, os.path.basename(str(x))))

# EXACT split used in your training scripts
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, train_df, val_df, temp_df; gc.collect()
print(f"Test split ready: {len(test_df)} samples.")

# ==========================================
# 3. LOAD MODEL & MERGE LORA
# ==========================================
print("\nLoading Base Model and merging Seed 8888 Adapter...")
processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Swahili", task="transcribe")
forced_decoder_ids = processor.get_decoder_prompt_ids(language="sw", task="transcribe")

base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", torch_dtype=DTYPE)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

# Merge the adapter weights directly into the base model and drop the PEFT wrapper
model = model.merge_and_unload()
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model = model.to(DEVICE).eval()

# ==========================================
# 4. PURE INFERENCE LOOP (BATCHED)
# ==========================================
print("\nStarting Pure Generation Loop (Bypassing Trainer)...")
predictions = []
references = []

# Convert dataframe to list of dicts for easy batching
test_data = test_df.to_dict('records')

for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Decoding Test Set"):
    batch = test_data[i : i + BATCH_SIZE]
    audio_arrays = []

    for item in batch:
        speech_array, sr = torchaudio.load(item["absolute_path"])
        if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
        if sr != 16000:
            speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)
        audio_arrays.append(speech_array.squeeze().numpy())
        references.append(item["transcript"])

    # Process audio through feature extractor
    inputs = processor(audio_arrays, sampling_rate=16000, return_tensors="pt", padding=True)
    input_features = inputs.input_features.to(DEVICE).to(DTYPE)

    # Strict forced generation
    with torch.no_grad():
        generated_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids,
            max_new_tokens=MAX_NEW_TOKENS
        )

    batch_preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
    predictions.extend([clean_transcript(p) for p in batch_preds])

# ==========================================
# 5. DIAGNOSTICS & VERDICT
# ==========================================
metric = evaluate.load("wer")
wer = 100.0 * metric.compute(predictions=predictions, references=references)

ref_len = np.array([max(1, len(r.split())) for r in references])
hyp_len = np.array([len(p.split()) for p in predictions])
ratio = hyp_len / ref_len
runaway = int((ratio > 2.0).sum())

print("\n" + "=" * 60)
print(f"Corrected Pure-Inference WER   : {wer:.2f}%")
print(f"Training-reported WER (broken) : 32.99%")
print("-" * 60)
print(f"Mean hyp/ref length ratio      : {ratio.mean():.2f}  (≈1.0 = healthy; »1 = repetition)")
print(f"Utterances with hyp >2x ref    : {runaway} / {len(references)}")
print("=" * 60)

print(f"\nSample Predictions (First 5):")
for r, p in list(zip(references, predictions))[:5]:
    print(f"  REF: {r}")
    print(f"  HYP: {p}\n")

print("Verdict:")
if wer < 20 and ratio.mean() < 1.5:
    print("  ✅ ADAPTER IS FINE. The 32.99% was a measurement artifact from the Trainer loop.")
    print("  Action: Rescore all your models using this pure inference script.")
elif ratio.mean() >= 1.5 or runaway > len(references) * 0.1:
    print("  ⚠️ HALLUCINATION LOOP. The model is stuttering and looping.")
    print("  Action: The augmentations broke the attention mechanism.")
else:
    print("  ❌ DESTRUCTIVE INTERFERENCE IS REAL. Generation looks healthy, but WER is genuinely bad.")
    print("  Action: The model learned garbage. Keep 32.99% as proof of augmentation instability.")

In [ ]:
# CELL: PURE-INFERENCE RESCORING FOR ALL SEEDS (FIXED FOR CHECKPOINTS)

import os
import re
import gc
import numpy as np
import scipy.stats as stats
import pandas as pd
import torch
import torchaudio
import evaluate
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel
import warnings
warnings.filterwarnings("ignore")

print("--- Starting Pure-Inference Rescore for All Seeds ---")

# ==========================================
# 1. PATHS & CONFIGURATION
# ==========================================
SEEDS = [42, 1024, 2024, 8888, 12345]
BASE_OUTPUT_DIR = "/content/drive/MyDrive/Colab/SwahiliASR_Whisper_LoRA_Colab_NewRun/LoRa_All_Augmentations_Combined_MultiSeed"
CSV_PATH = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
AUDIO_BASE_PATH = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"

FINAL_RESULTS_CSV = os.path.join(BASE_OUTPUT_DIR, "rescored_ensemble_results_pure_inference.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
BATCH_SIZE = 16
MAX_NEW_TOKENS = 225

# ==========================================
# 2. HELPER TO FIND THE RIGHT FOLDER
# ==========================================
def find_latest_checkpoint(seed_dir):
    """Finds the checkpoint-XXXX folder with the highest number."""
    if not os.path.exists(seed_dir):
        return None

    # Check if adapter is sitting right in the root (rare but possible)
    if os.path.exists(os.path.join(seed_dir, "adapter_config.json")):
        return seed_dir

    # Look for checkpoint folders
    checkpoints = [d for d in os.listdir(seed_dir) if d.startswith("checkpoint-")]
    if not checkpoints:
        return None

    # Sort them by the integer step number and pick the highest one
    checkpoints.sort(key=lambda x: int(x.split("-")[1]))
    latest_checkpoint = checkpoints[-1]

    return os.path.join(seed_dir, latest_checkpoint)

# ==========================================
# 3. RECREATE EXACT TEST SPLIT
# ==========================================
print("Reconstructing exact 900-file test split...")
df = pd.read_csv(CSV_PATH).head(9000)

def clean_transcript(text):
    if pd.isna(text): return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s']", '', text)
    return re.sub(r'\s+', ' ', text).strip()

df["transcript"] = df["transcript"].apply(clean_transcript)
df = df[df["transcript"].notnull() & (df["transcript"] != "")]
df["absolute_path"] = df["path"].apply(lambda x: os.path.join(AUDIO_BASE_PATH, os.path.basename(str(x))))

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)
del df, train_df, val_df, temp_df; gc.collect()

test_data = test_df.to_dict('records')
print(f"Test split ready: {len(test_data)} samples.")

processor = WhisperProcessor.from_pretrained("openai/whisper-small", language="Swahili", task="transcribe")
forced_decoder_ids = processor.get_decoder_prompt_ids(language="sw", task="transcribe")
metric = evaluate.load("wer")

# ==========================================
# 4. BATCH INFERENCE & RESCORING LOOP
# ==========================================
rescored_wers = []

for seed in SEEDS:
    seed_path = os.path.join(BASE_OUTPUT_DIR, f"run_seed_{seed}")
    adapter_path = find_latest_checkpoint(seed_path)

    if adapter_path is None:
        print(f"\n⚠️ Skipping Seed {seed}: Could not find a valid checkpoint in {seed_path}")
        continue

    print(f"\n{'='*60}")
    print(f"🚀 RESCORING SEED: {seed}")
    print(f"📁 Loading from: {os.path.basename(adapter_path)}")
    print(f"{'='*60}")

    print("Loading Base Model and merging LoRA Adapter...")
    base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small", torch_dtype=DTYPE)
    model = PeftModel.from_pretrained(base_model, adapter_path)

    # Merge weights and drop PEFT abstraction
    model = model.merge_and_unload()
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []
    model = model.to(DEVICE).eval()

    predictions = []
    references = []

    for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc=f"Decoding Seed {seed}"):
        batch = test_data[i : i + BATCH_SIZE]
        audio_arrays = []

        for item in batch:
            speech_array, sr = torchaudio.load(item["absolute_path"])
            if speech_array.shape[0] > 1: speech_array = torch.mean(speech_array, dim=0, keepdim=True)
            if sr != 16000:
                speech_array = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(speech_array)
            audio_arrays.append(speech_array.squeeze().numpy())
            references.append(item["transcript"])

        inputs = processor(audio_arrays, sampling_rate=16000, return_tensors="pt", padding=True)
        input_features = inputs.input_features.to(DEVICE).to(DTYPE)

        with torch.no_grad():
            generated_ids = model.generate(
                input_features,
                forced_decoder_ids=forced_decoder_ids,
                max_new_tokens=MAX_NEW_TOKENS
            )

        batch_preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
        predictions.extend([clean_transcript(p) for p in batch_preds])

    # Calculate True WER
    wer = 100.0 * metric.compute(predictions=predictions, references=references)
    print(f"✅ True Pure-Inference WER for Seed {seed}: {wer:.2f}%")

    rescored_wers.append({"seed": seed, "wer": wer})

    # Save incremental progress
    pd.DataFrame(rescored_wers).to_csv(FINAL_RESULTS_CSV, index=False)

    # Extreme memory cleanup before next seed
    del model, base_model, predictions, references
    gc.collect()
    torch.cuda.empty_cache()

# ==========================================
# 5. FINAL STATISTICAL CALCULATION
# ==========================================
if len(rescored_wers) > 1:
    wer_values = [run["wer"] for run in rescored_wers]
    n = len(rescored_wers)
    mean_wer = np.mean(wer_values)
    sd_wer = np.std(wer_values, ddof=1)
    ci_lower, ci_upper = stats.t.interval(0.95, df=n-1, loc=mean_wer, scale=sd_wer/np.sqrt(n))

    print("\n" + "="*60)
    print(f"📊 FINAL RESCORED STATISTICAL REPORT (n={n} seeds)")
    print("="*60)
    print("Individual Runs:")
    for run in rescored_wers:
        print(f"  - Seed {run['seed']}: {run['wer']:.2f}%")
    print("-" * 60)
    print(f"Mean WER:           {mean_wer:.2f}%")
    print(f"Standard Deviation: {sd_wer:.3f}")
    print(f"95% CI bounds:      [{ci_lower:.2f}%, {ci_upper:.2f}%]")
    print(f"Format for Thesis:  {mean_wer:.2f} ± {sd_wer:.2f} [{ci_lower:.2f}%, {ci_upper:.2f}%]")
    print("="*60)
    print(f"✅ Saved true results to: {FINAL_RESULTS_CSV}")
else:
    print("\nNot enough runs successfully rescored to calculate statistics.")

In [ ]:
!pip install -q openai-whisper datasets jiwer evaluate

In [ ]:
# =============================================================================
# ZERO-SHOT VERIFICATION USING THE REFERENCE WHISPER IMPLEMENTATION

import os, re, json, gzip
import numpy as np, pandas as pd
import torch, evaluate

RUN_COMMONVOICE = False   # set True once FLEURS has reproduced
CSV_PATH        = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
AUDIO_BASE_PATH = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips/"
OUT_DIR         = "/content/drive/MyDrive/Colab/zeroshot_diagnostic"
N_ROWS          = 9000
os.makedirs(OUT_DIR, exist_ok=True)


import builtins, datetime
_LOG = os.path.join(OUT_DIR, "console_log.txt")
open(_LOG, "w").write(f"run started {datetime.datetime.now():%Y-%m-%d %H:%M:%S}\n")
_orig_print = builtins.print
def print(*a, **k):
    _orig_print(*a, **k)
    try:
        with open(_LOG, "a") as fh:
            _orig_print(*a, **{**k, "file": fh})
    except Exception:
        pass
builtins.print = print

import whisper
from whisper.normalizers import BasicTextNormalizer
metric = evaluate.load("wer")
basic  = BasicTextNormalizer()

def thesis_norm(t):
    if not isinstance(t, str): return ""
    t = t.lower().strip()
    t = re.sub(r"[^\w\s']", "", t)
    return re.sub(r"\s+", " ", t).strip()

print("whisper", whisper.__version__ if hasattr(whisper,"__version__") else "(pkg)",
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
model = whisper.load_model("small")     # same checkpoint as openai/whisper-small

def transcribe_all(items, label):
    """items: list of (audio_path_or_array, reference). Full reference decoding."""
    preds, refs = [], []
    from tqdm.auto import tqdm
    t0 = __import__("time").time()
    for i, (audio, ref) in enumerate(tqdm(items, desc=label, leave=False)):
        if i and i % 25 == 0:
            el = __import__("time").time() - t0
            eta = el / i * (len(items) - i)
            with open(os.path.join(OUT_DIR, "progress.txt"), "w") as fh:
                fh.write(f"{label}: {i}/{len(items)} clips, {el/60:.1f} min elapsed, "
                         f"~{eta/60:.1f} min remaining\n")
        r = model.transcribe(
            audio, language="sw", task="transcribe",
            beam_size=5,
            temperature=(0.0, 0.2, 0.4, 0.6, 0.8, 1.0),   # fallback ladder
            compression_ratio_threshold=2.4,               # catches repetition
            logprob_threshold=-1.0,
            no_speech_threshold=0.6,
            fp16=torch.cuda.is_available(),
            verbose=False,
        )
        preds.append(r["text"]); refs.append(ref)
    return preds, refs

def score(preds, refs, label, corpus):
    rows = []
    for nm, fn in [("thesis normalizer", thesis_norm), ("Whisper BasicTextNormalizer", basic)]:
        p = [fn(x) for x in preds]; r = [fn(x) for x in refs]
        keep = [(a, b) for a, b in zip(p, r) if b.strip()]
        p, r = [a for a, _ in keep], [b for _, b in keep]
        wer = 100.0 * metric.compute(predictions=p, references=r)
        ratio = sum(len(x.split()) for x in p) / sum(len(x.split()) for x in r)
        loops = sum(1 for x in p
                    if len(x.split()) > 8
                    and len(x.encode())/max(1, len(gzip.compress(x.encode()))) > 2.4)
        rows.append(dict(corpus=corpus, decoding=label, normalizer=nm,
                         wer=wer, len_ratio=ratio, loops=loops, n=len(p)))
        print(f"  {nm:<30} WER {wer:6.2f}%   len {ratio:4.2f}   loops {loops}")
    return rows

results = []

# ---------------- FLEURS: the corpus with a published reference ----------------
print("\n" + "="*72)
print("FLEURS Swahili test set   (published reference: 73.7% for whisper-small)")
print("="*72)
from datasets import load_dataset, Audio
fl = load_dataset("google/fleurs", "sw_ke", split="test", trust_remote_code=True)
fl = fl.cast_column("audio", Audio(sampling_rate=16000))
items = [(np.asarray(x["audio"]["array"], dtype=np.float32), x["transcription"]) for x in fl]
print(f"clips: {len(items)}")
preds, refs = transcribe_all(items, "FLEURS reference decoding")
results += score(preds, refs, "reference (beam5 + fallback)", "FLEURS")
pd.DataFrame({"reference": refs, "prediction": preds}).to_csv(
    f"{OUT_DIR}/fleurs_reference_decoding.csv", index=False)

# ---------------- Common Voice: the thesis in-corpus baseline ----------------
if RUN_COMMONVOICE:
    print("\n" + "="*72)
    print("Common Voice Swahili test set   (thesis Section 4.6 reports 87.73%)")
    print("="*72)
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(CSV_PATH).head(N_ROWS)
    df["path"] = df["path"].apply(lambda x: os.path.join(AUDIO_BASE_PATH, os.path.basename(str(x))))
    df["transcript"] = df["transcript"].apply(thesis_norm)
    df = df[df["transcript"].notnull() & (df["transcript"] != "")]
    _, tmp = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)
    _, te  = train_test_split(tmp, test_size=0.5, random_state=42, shuffle=True)
    items = [(r["path"], r["transcript"]) for _, r in te.iterrows()]
    print(f"clips: {len(items)}")
    preds, refs = transcribe_all(items, "Common Voice reference decoding")
    results += score(preds, refs, "reference (beam5 + fallback)", "CommonVoice")
    pd.DataFrame({"reference": refs, "prediction": preds}).to_csv(
        f"{OUT_DIR}/commonvoice_reference_decoding.csv", index=False)

# ---------------------------------- verdict ----------------------------------
print("\n" + "="*72); print("SUMMARY"); print("="*72)
print(pd.DataFrame(results).to_string(index=False))
pd.DataFrame(results).to_csv(f"{OUT_DIR}/reference_decoding_summary.csv", index=False)

fw = [r["wer"] for r in results if r["corpus"]=="FLEURS"]
if fw:
    best = min(fw)
    print(f"\nPublished whisper-small on FLEURS Swahili : 73.70%")
    print(f"Best reproduction here                    : {best:.2f}%   (gap {best-73.7:+.2f})")
    if abs(best-73.7) < 6:
        print("-> Pipeline REPRODUCES the published figure. The evaluation is sound,")
        print("   and the Common Voice number above can be reported with confidence.")
    else:
        print("-> Pipeline does NOT reproduce it. Do not report either number until")
        print("   the discrepancy is understood: check the FLEURS split, the audio")
        print("   sampling rate, and whether references carry punctuation.")
print(f"\nSaved to {OUT_DIR}")
print("Full console log: " + _LOG)

In [ ]:
!pip install -q --upgrade "transformers>=4.46" "peft>=0.13" "accelerate>=1.0"
!pip install -q --upgrade "datasets>=3.0" "bitsandbytes>=0.46"
!pip install -q --upgrade "evaluate>=0.4" "jiwer>=3.0"
!pip install -q --upgrade "soundfile>=0.12" "librosa>=0.10" "soxr>=0.3"

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 Swahili ASR — QLoRA (4-bit NF4) fine-tuning of Whisper with Gaussian-noise augmentation.-*-

from __future__ import annotations

import argparse
import dataclasses
import gc
import json
import logging
import math
import os
import re
import sys
import time
from typing import Any, Dict, List, Optional

import numpy as np

logger = logging.getLogger("swahili_qlora")


SAMPLING_RATE = 16_000
THESIS_SPLIT_SEED = 42          # reproduces the exact 7200/900/900 clip-level split
NOISE_MIN_AMPLITUDE = 0.001
NOISE_MAX_AMPLITUDE = 0.010
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]



_PUNCT_RE = re.compile(r"[^\w\s']")
_WS_RE = re.compile(r"\s+")


def normalize_text(text: Any) -> str:
    """Lower-case, drop punctuation except the apostrophe, collapse whitespace.

    This is byte-for-byte the cleaning applied to the training transcripts in the
    thesis notebook, reused at scoring time so WER is not inflated by formatting.
    """
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""
    text = str(text).lower().strip()
    text = _PUNCT_RE.sub("", text)
    text = _WS_RE.sub(" ", text)
    return text.strip()


def add_gaussian_noise(
    samples: np.ndarray,
    rng: np.random.Generator,
    p: float = 0.3,
    min_amplitude: float = NOISE_MIN_AMPLITUDE,
    max_amplitude: float = NOISE_MAX_AMPLITUDE,
) -> np.ndarray:
    """x'[n] = x[n] + e[n],  e ~ N(0, sigma^2),  sigma ~ U(min_amplitude, max_amplitude).

    Matches `audiomentations.AddGaussianNoise(min_amplitude, max_amplitude, p)` as used
    in the thesis, but implemented directly so the run has one fewer dependency and the
    noise draw is reproducible from a single seeded Generator.
    """
    if p <= 0.0 or rng.random() >= p:
        return samples
    sigma = rng.uniform(min_amplitude, max_amplitude)
    noise = rng.standard_normal(samples.shape).astype(np.float32)
    return (samples + sigma * noise).astype(np.float32)


# --------------------------------------------------------------------------------------
# WER metric
# --------------------------------------------------------------------------------------

class _JiwerWer:


    def compute(self, predictions: List[str], references: List[str]) -> float:
        import jiwer
        return float(jiwer.wer(list(references), list(predictions)))


def load_wer_metric():
    try:
        import evaluate as hf_evaluate
        metric = hf_evaluate.load("wer")
        logger.info("WER metric: huggingface `evaluate`")
        return metric
    except Exception as exc:
        logger.warning("Could not load `evaluate` WER (%s); using jiwer directly "
                       "(numerically identical).", type(exc).__name__)
        return _JiwerWer()


# --------------------------------------------------------------------------------------
# LoRA / QLoRA configuration
# --------------------------------------------------------------------------------------

def build_lora_kwargs(
    r: int = 32,
    alpha: int = 64,
    dropout: float = 0.05,
    layer_adaptive: bool = False,
    encoder_r: int = 64,
    decoder_r: int = 16,
) -> Dict[str, Any]:
    """Return kwargs for `peft.LoraConfig`.

    Default (layer_adaptive=False) reproduces the thesis adapter: r=32, alpha=64,
    dropout=0.05 on q/k/v/out/fc1/fc2.

    With layer_adaptive=True, ranks follow Han et al. (2024): a higher rank on the
    encoder (acoustic adaptation is where the Swahili-specific work happens) and a
    lower rank on the decoder. `alpha` is scaled with `r` so the LoRA scaling factor
    alpha/r stays at 2.0 everywhere — otherwise a rank change silently becomes a
    learning-rate change too, and the ablation is confounded.

    PEFT matches `rank_pattern` keys as the regex `.*\\.{key}$` against each full
    module path (e.g. "model.encoder.layers.0.self_attn.q_proj"), which is why the
    keys below are written as regexes rather than plain substrings.
    """
    kwargs: Dict[str, Any] = dict(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=list(LORA_TARGET_MODULES),
        bias="none",
    )
    if layer_adaptive:
        scaling = alpha / r  # preserve alpha/r == 2.0
        kwargs["rank_pattern"] = {
            r"encoder\..*": encoder_r,
            r"decoder\..*": decoder_r,
        }
        kwargs["alpha_pattern"] = {
            r"encoder\..*": int(round(encoder_r * scaling)),
            r"decoder\..*": int(round(decoder_r * scaling)),
        }
    return kwargs


# --------------------------------------------------------------------------------------
# Dataset transform + collator
# --------------------------------------------------------------------------------------

def load_audio(path: str, target_sr: int = SAMPLING_RATE) -> np.ndarray:
    arr = None
    sr = None

    # 1. First attempt: soundfile
    try:
        import soundfile as sf
        arr, sr = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        arr = None

    # 2. Second attempt: librosa (Wrapped in try-except to prevent the crash)
    if arr is None:
        try:
            import librosa
            arr, sr = librosa.load(path, sr=None, mono=True)
        except Exception:
            arr = None

    # 3. CRASH PREVENTION: If the file is completely corrupted or unreadable
    if arr is None or len(arr) == 0:
        logger = logging.getLogger("swahili_qlora")
        logger.error(f"CORRUPT AUDIO FILE DETECTED & SKIPPED: {path}")
        # Return 1 second of silent audio so the evaluation batch can continue without crashing
        return np.zeros(target_sr, dtype=np.float32)

    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)

    if sr != target_sr:
        try:
            import soxr
            arr = soxr.resample(arr, sr, target_sr)
        except ImportError:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
        arr = np.asarray(arr, dtype=np.float32)

    return arr


def make_transform(processor, augment: bool, seed: int, noise_prob: float):
    """Build an on-the-fly `set_transform` callable.

    Decoding, noise injection and log-Mel extraction all happen per batch access, so
    every epoch sees a fresh noise draw. `augment` must be False for val/test.
    """
    feature_extractor = processor.feature_extractor
    tokenizer = processor.tokenizer
    rng = np.random.default_rng(seed)

    def _transform(batch: Dict[str, List[Any]]) -> Dict[str, List[Any]]:
        paths = batch["audio_path"]
        sentences = batch["sentence"]

        waveforms = []
        for path in paths:
            arr = load_audio(path)
            if augment:
                arr = add_gaussian_noise(arr, rng, p=noise_prob)
            waveforms.append(arr)

        features = feature_extractor(
            waveforms,
            sampling_rate=SAMPLING_RATE,
            return_tensors="np",
        ).input_features

        labels = tokenizer(
            [normalize_text(s) for s in sentences]
        ).input_ids

        return {
            "input_features": [f.astype(np.float32) for f in features],
            "labels": labels,
        }

    return _transform


@dataclasses.dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Pad log-Mel features and right-pad labels with -100 so they are ignored by the loss.

    Whisper pads/truncates audio to a fixed 30s window, so `input_features` are already
    uniform; `feature_extractor.pad` is used anyway to stay correct if that changes.
    """

    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        import torch

        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Whisper's decoder prepends the start token itself; drop it if the tokenizer
        # already added one, or the model sees it twice.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


# --------------------------------------------------------------------------------------
# Trainer with generation-based WER evaluation
# --------------------------------------------------------------------------------------

def _build_trainer_class():
    """Defined lazily so the module imports without transformers present (for unit tests)."""
    import torch
    from transformers import Seq2SeqTrainer

    class WhisperWERTrainer(Seq2SeqTrainer):
        """Seq2SeqTrainer that scores WER by running real generation.

        `Seq2SeqTrainer(predict_with_generate=True)` is unreliable when the model is a
        PEFT adapter over a 4-bit backbone, so generation is done here explicitly under
        autocast. Best-checkpoint selection and early stopping are handled in this class
        rather than via `load_best_model_at_end` / `EarlyStoppingCallback`, which both
        have rough edges around adapter-only checkpoints.
        """

        def __init__(
            self,
            *args,
            processor=None,
            wer_metric=None,
            gen_max_new_tokens: int = 225,
            gen_num_beams: int = 1,
            best_adapter_dir: str = "best_adapter",
            early_stopping_patience: int = 5,
            **kwargs,
        ):
            super().__init__(*args, **kwargs)
            self.processor = processor
            self.wer_metric = wer_metric
            self.gen_max_new_tokens = gen_max_new_tokens
            self.gen_num_beams = gen_num_beams
            self.best_adapter_dir = best_adapter_dir
            self.early_stopping_patience = early_stopping_patience
            self.best_wer = float("inf")
            self.evals_without_improvement = 0
            self.wer_history: List[Dict[str, float]] = []

        @torch.no_grad()
        def compute_wer(self, dataset, description: str = "eval") -> Dict[str, Any]:
            model = self.model
            was_training = model.training
            model.eval()

            dataloader = self.get_eval_dataloader(dataset)
            device = self.args.device
            use_cuda = device.type == "cuda"

            predictions: List[str] = []
            references: List[str] = []
            tokenizer = self.processor.tokenizer

            for batch in dataloader:
                input_features = batch["input_features"].to(device)
                with torch.autocast(device_type=device.type,
                                    dtype=torch.float16,
                                    enabled=use_cuda):
                    generated = model.generate(
                        input_features=input_features,
                        max_new_tokens=self.gen_max_new_tokens,
                        num_beams=self.gen_num_beams,
                        use_cache=True,
                    )
                generated = generated.detach().cpu().numpy()

                labels = batch["labels"].cpu().numpy()
                labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

                predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
                references.extend(tokenizer.batch_decode(labels, skip_special_tokens=True))

                del generated, input_features
                if use_cuda:
                    torch.cuda.empty_cache()

            predictions = [normalize_text(p) for p in predictions]
            references = [normalize_text(r) for r in references]

            # An empty reference makes WER undefined; drop those pairs and say how many.
            kept = [(p, r) for p, r in zip(predictions, references) if r]
            dropped = len(references) - len(kept)
            if dropped:
                logger.warning("%s: dropped %d sample(s) with empty reference", description, dropped)
            if not kept:
                raise ValueError(f"{description}: no scorable samples (all references empty)")

            preds, refs = zip(*kept)
            wer = 100.0 * self.wer_metric.compute(predictions=list(preds), references=list(refs))

            if was_training:
                model.train()

            return {"wer": wer, "n_scored": len(kept), "n_dropped": dropped,
                    "predictions": list(preds), "references": list(refs)}

        def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
            dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
            result = self.compute_wer(dataset, description=metric_key_prefix)
            wer = result["wer"]

            metrics = {
                f"{metric_key_prefix}_wer": wer,
                f"{metric_key_prefix}_n_scored": result["n_scored"],
            }
            self.log(metrics)
            self.wer_history.append({"step": int(self.state.global_step), "wer": wer})

            if wer < self.best_wer:
                self.best_wer = wer
                self.evals_without_improvement = 0
                os.makedirs(self.best_adapter_dir, exist_ok=True)
                # Save the adapter only — the 4-bit backbone is reloaded from the hub.
                unwrapped = self.accelerator.unwrap_model(self.model)
                unwrapped.save_pretrained(self.best_adapter_dir)
                self.processor.save_pretrained(self.best_adapter_dir)
                with open(os.path.join(self.best_adapter_dir, "best_metric.json"), "w") as fh:
                    json.dump({"step": int(self.state.global_step), "val_wer": wer}, fh, indent=2)
                logger.info("New best val WER %.2f%% at step %d -> saved to %s",
                            wer, self.state.global_step, self.best_adapter_dir)
            else:
                self.evals_without_improvement += 1
                logger.info("No improvement (%.2f%% vs best %.2f%%), patience %d/%d",
                            wer, self.best_wer,
                            self.evals_without_improvement, self.early_stopping_patience)
                if self.evals_without_improvement >= self.early_stopping_patience:
                    logger.info("Early stopping triggered.")
                    self.control.should_training_stop = True

            return metrics

    return WhisperWERTrainer


# --------------------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------------------

def load_splits(csv_path: str, audio_dir: str, n_samples: int, split_seed: int):
    """Rebuild the thesis 80:10:10 clip-level split from the same CSV.

    Uses `head(n_samples)` and `random_state=42` exactly as the thesis notebook did, so
    the partition is identical and results are directly comparable. The split is done
    before any augmentation, so no augmented variant of a training clip can leak into
    val or test.
    """
    import pandas as pd
    from datasets import Dataset, DatasetDict
    from sklearn.model_selection import train_test_split

    df = pd.read_csv(csv_path).head(n_samples)
    logger.info("Loaded %d rows from %s", len(df), csv_path)

    df["path"] = df["path"].apply(
        lambda p: os.path.join(audio_dir, os.path.basename(str(p)))
    )
    df["transcript"] = df["transcript"].apply(normalize_text)
    df = df[df["transcript"].notnull() & (df["transcript"] != "")]

    exists = df["path"].apply(os.path.exists)
    if (~exists).any():
        logger.warning("Dropping %d row(s) whose audio file is missing", int((~exists).sum()))
        df = df[exists]

    if len(df) == 0:
        raise ValueError("No usable rows after cleaning — check --csv and --audio-dir.")

    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=split_seed, shuffle=True)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=split_seed, shuffle=True)

    def to_ds(frame):
        # Paths stay plain strings; decoding happens in the transform (see load_audio).
        return Dataset.from_dict({
            "audio_path": frame["path"].tolist(),
            "sentence": frame["transcript"].tolist(),
        })

    splits = DatasetDict({"train": to_ds(train_df), "val": to_ds(val_df), "test": to_ds(test_df)})
    logger.info("Split sizes — train %d, val %d, test %d",
                len(splits["train"]), len(splits["val"]), len(splits["test"]))
    return splits


# --------------------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------------------

def load_quantized_model(base_model: str, use_4bit: bool, gradient_checkpointing: bool):
    import torch
    from transformers import WhisperForConditionalGeneration

    if use_4bit:
        from transformers import BitsAndBytesConfig

        # fp16 compute, not bf16: the Tesla T4 is Turing (sm_75) and has no bf16 support.
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = WhisperForConditionalGeneration.from_pretrained(
            base_model, quantization_config=quant_config, device_map="auto"
        )
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=gradient_checkpointing
        )
    else:
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = WhisperForConditionalGeneration.from_pretrained(base_model, torch_dtype=dtype)
        if torch.cuda.is_available():
            model = model.to("cuda")

    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
        # Required so gradients reach the adapters through a frozen, checkpointed backbone.
        model.enable_input_require_grads()

    return model


def configure_generation(model, processor, language: str = "sw", task: str = "transcribe"):
    """Pin the decoder prompt to Swahili transcription and clear legacy forced ids."""
    model.generation_config.language = language
    model.generation_config.task = task
    model.generation_config.forced_decoder_ids = None
    if hasattr(model.config, "forced_decoder_ids"):
        model.config.forced_decoder_ids = None
    # Deliberately NOT clearing `suppress_tokens`. Older Whisper fine-tuning notebooks
    # set `config.suppress_tokens = []`; that both trips a deprecation path in current
    # transformers and removes Whisper's non-speech token suppression, which costs WER.
    return model


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------

def _resolve_argv(argv: Optional[List[str]]) -> List[str]:
    """Return the argument list to parse, tolerating a Jupyter/Colab kernel.

    A notebook kernel is launched as `... -f /root/.../kernel-<uuid>.json`, so a bare
    `main()` called from a cell would otherwise see that flag in `sys.argv` and argparse
    would exit with "unrecognized arguments". Inside a kernel there are no real
    command-line arguments anyway, so pass them explicitly:

        from swahili_asr_qlora import main
        main(["--base-model", "openai/whisper-large-v3", "--epochs", "5"])
    """
    if argv is not None:
        return list(argv)

    raw = list(sys.argv[1:])
    in_kernel = "ipykernel" in sys.modules or "google.colab" in sys.modules
    if not in_kernel:
        return raw

    cleaned: List[str] = []
    skip_next = False
    for token in raw:
        if skip_next:
            skip_next = False
            continue
        if token == "-f":
            skip_next = True          # drop the kernel connection-file path that follows
            continue
        if token.startswith("--f=") or token.startswith("-f="):
            continue
        if token.endswith(".json") and "kernel" in os.path.basename(token):
            continue
        cleaned.append(token)

    if cleaned:
        logger.warning("Running inside a notebook kernel; parsing %r from sys.argv. "
                       "Prefer passing arguments explicitly: main([...])", cleaned)
    return cleaned


def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="QLoRA fine-tuning of Whisper for Swahili ASR (Gaussian-noise augmentation only).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )

    # Data
    p.add_argument("--csv", default="/content/drive/MyDrive/Colab/New Swahili Dataset/"
                                    "mozilla-swa-dataset/swahili_dataset.csv",
                   help="CSV with 'path' and 'transcript' columns.")
    p.add_argument("--audio-dir", default="/content/drive/MyDrive/Colab/New Swahili Dataset/"
                                          "mozilla-swa-dataset/clips",
                   help="Directory holding the audio clips.")
    p.add_argument("--n-samples", type=int, default=9000,
                   help="Clips to take from the head of the CSV (9000 = thesis subset).")
    p.add_argument("--split-seed", type=int, default=THESIS_SPLIT_SEED,
                   help="Keep at 42 to reproduce the thesis split exactly.")

    # Model / PEFT
    p.add_argument("--base-model", default="openai/whisper-large-v3",
                   help="HF model id. Use openai/whisper-small for the like-for-like arm.")
    p.add_argument("--no-4bit", action="store_true",
                   help="Disable NF4 quantization (only if you have an A100/L4 with headroom).")
    p.add_argument("--lora-r", type=int, default=32)
    p.add_argument("--lora-alpha", type=int, default=64)
    p.add_argument("--lora-dropout", type=float, default=0.05)
    p.add_argument("--layer-adaptive", action="store_true",
                   help="Layer-adaptive ranks (Han et al. 2024): high encoder rank, low decoder rank.")
    p.add_argument("--encoder-r", type=int, default=64)
    p.add_argument("--decoder-r", type=int, default=16)

    # Augmentation
    p.add_argument("--noise-prob", type=float, default=0.3,
                   help="Probability of applying Gaussian noise to a training clip.")
    p.add_argument("--no-augment", action="store_true",
                   help="Train without augmentation (control arm).")

    # Optimisation
    p.add_argument("--epochs", type=float, default=5.0,
                   help="25 matches the thesis for whisper-small; 4-6 is ample for large-v3.")
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--grad-accum", type=int, default=16,
                   help="batch-size * grad-accum = 32 reproduces the thesis effective batch.")
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--warmup-steps", type=int, default=200)
    p.add_argument("--weight-decay", type=float, default=0.01)
    p.add_argument("--lr-scheduler", default="linear",
                   choices=["linear", "cosine", "constant_with_warmup"])
    p.add_argument("--no-gradient-checkpointing", action="store_true")
    p.add_argument("--seed", type=int, default=42,
                   help="Training seed. Vary this across >=3 runs before ranking anything.")

    # Evaluation
    p.add_argument("--eval-steps", type=int, default=250)
    p.add_argument("--eval-subset", type=int, default=300,
                   help="Val clips used for mid-training WER (generation is slow). "
                        "0 = use the full validation split.")
    p.add_argument("--num-beams", type=int, default=1,
                   help="1 = greedy, matching the submitted thesis §3.7 protocol.")
    p.add_argument("--max-new-tokens", type=int, default=225)
    p.add_argument("--early-stopping-patience", type=int, default=5)
    p.add_argument("--fleurs-csv", default=None,
                   help="Optional CSV for out-of-corpus FLEURS evaluation.")
    p.add_argument("--fleurs-audio-dir", default=None)

    # Bookkeeping
    p.add_argument("--output-dir", default="./whisper-qlora-gaussian")
    p.add_argument("--resume", default=None,
                   help="Path to a Trainer checkpoint to resume from (Colab sessions time out).")
    p.add_argument("--skip-test", action="store_true",
                   help="Skip the held-out test evaluation (for quick sanity runs).")

    return p.parse_args(_resolve_argv(argv))


def main(argv: Optional[List[str]] = None) -> int:
    args = parse_args(argv)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)-7s  %(message)s",
        datefmt="%H:%M:%S",
        stream=sys.stdout,
    )

    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import Seq2SeqTrainingArguments, WhisperProcessor, set_seed

    use_4bit = not args.no_4bit
    if use_4bit and not torch.cuda.is_available():
        raise SystemExit(
            "4-bit quantization requires a CUDA GPU. In Colab: Runtime > Change runtime "
            "type > T4 GPU. Pass --no-4bit only if you intend to train in fp16/fp32."
        )

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    logger.info("Base model         : %s", args.base_model)
    logger.info("Quantization       : %s", "4-bit NF4 (double quant, fp16 compute)" if use_4bit else "none")
    logger.info("Augmentation       : %s", "none" if args.no_augment
                else f"additive Gaussian noise, p={args.noise_prob}, sigma~U(0.001,0.010)")
    logger.info("LoRA               : %s", "layer-adaptive" if args.layer_adaptive
                else f"uniform r={args.lora_r}, alpha={args.lora_alpha}")
    logger.info("Seed               : %d", args.seed)

    # ---- data -------------------------------------------------------------------------
    processor = WhisperProcessor.from_pretrained(
        args.base_model, language="Swahili", task="transcribe"
    )
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

    splits = load_splits(args.csv, args.audio_dir, args.n_samples, args.split_seed)

    splits["train"].set_transform(
        make_transform(processor, augment=not args.no_augment,
                       seed=args.seed, noise_prob=args.noise_prob)
    )
    eval_transform = make_transform(processor, augment=False, seed=args.seed, noise_prob=0.0)
    splits["val"].set_transform(eval_transform)
    splits["test"].set_transform(eval_transform)

    val_for_training = splits["val"]
    if args.eval_subset and args.eval_subset < len(splits["val"]):
        val_for_training = splits["val"].select(range(args.eval_subset))
        val_for_training.set_transform(eval_transform)
        logger.info("Mid-training WER measured on a %d-clip validation subset; "
                    "the full %d-clip split is scored once at the end.",
                    args.eval_subset, len(splits["val"]))

    # ---- model ------------------------------------------------------------------------
    gradient_checkpointing = not args.no_gradient_checkpointing
    model = load_quantized_model(args.base_model, use_4bit, gradient_checkpointing)
    model = configure_generation(model, processor)

    lora_kwargs = build_lora_kwargs(
        r=args.lora_r, alpha=args.lora_alpha, dropout=args.lora_dropout,
        layer_adaptive=args.layer_adaptive,
        encoder_r=args.encoder_r, decoder_r=args.decoder_r,
    )
    model = get_peft_model(model, LoraConfig(task_type=None, **lora_kwargs))
    model.print_trainable_parameters()
    model.config.use_cache = False  # incompatible with gradient checkpointing

    collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    # ---- training ---------------------------------------------------------------------
    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_steps=args.warmup_steps,
        num_train_epochs=args.epochs,
        lr_scheduler_type=args.lr_scheduler,
        weight_decay=args.weight_decay,
        fp16=torch.cuda.is_available(),
        gradient_checkpointing=gradient_checkpointing,
        eval_strategy="steps",
        eval_steps=args.eval_steps,
        save_strategy="steps",
        save_steps=args.eval_steps,
        save_total_limit=2,
        logging_steps=25,
        predict_with_generate=False,   # generation is handled by WhisperWERTrainer
        load_best_model_at_end=False,  # best adapter is saved explicitly instead
        remove_unused_columns=False,   # set_transform supplies the columns
        label_names=["labels"],
        dataloader_num_workers=2,
        seed=args.seed,
        report_to=[],
    )

    wer_metric = load_wer_metric()
    best_adapter_dir = os.path.join(args.output_dir, "best_adapter")
    WhisperWERTrainer = _build_trainer_class()

    trainer = WhisperWERTrainer(
        model=model,
        args=training_args,
        train_dataset=splits["train"],
        eval_dataset=val_for_training,
        data_collator=collator,
        processing_class=processor.feature_extractor,
        processor=processor,
        wer_metric=wer_metric,
        gen_max_new_tokens=args.max_new_tokens,
        gen_num_beams=args.num_beams,
        best_adapter_dir=best_adapter_dir,
        early_stopping_patience=args.early_stopping_patience,
    )

    started = time.time()
    trainer.train(resume_from_checkpoint=args.resume)
    train_minutes = (time.time() - started) / 60.0
    logger.info("Training finished in %.1f min. Best validation WER: %.2f%%",
                train_minutes, trainer.best_wer)

    # ---- final evaluation -------------------------------------------------------------
    results: Dict[str, Any] = {
        "base_model": args.base_model,
        "quantization": "nf4-4bit" if use_4bit else "none",
        "augmentation": "none" if args.no_augment else "gaussian_noise",
        "noise_prob": None if args.no_augment else args.noise_prob,
        "lora": lora_kwargs,
        "layer_adaptive": args.layer_adaptive,
        "seed": args.seed,
        "epochs": args.epochs,
        "effective_batch_size": args.batch_size * args.grad_accum,
        "learning_rate": args.lr,
        "num_beams": args.num_beams,
        "decoding": "greedy" if args.num_beams == 1 else f"beam-{args.num_beams}",
        "best_val_wer_subset": trainer.best_wer,
        "val_wer_history": trainer.wer_history,
        "train_minutes": round(train_minutes, 1),
    }

    if os.path.isdir(best_adapter_dir):
        logger.info("Reloading best adapter from %s for final scoring", best_adapter_dir)
        from peft import PeftModel

        # Free the training model and its optimizer state first. Holding both copies
        # plus Adam moments is what pushes a T4 over the edge on large-v3.
        trainer.model = None
        trainer.optimizer = None
        trainer.lr_scheduler = None
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        base = load_quantized_model(args.base_model, use_4bit, gradient_checkpointing=False)
        base = configure_generation(base, processor)
        best_model = PeftModel.from_pretrained(base, best_adapter_dir)
        best_model.eval()
        trainer.model = best_model

    full_val = trainer.compute_wer(splits["val"], description="val_full")
    results["val_wer_full"] = full_val["wer"]
    logger.info("Full validation WER: %.2f%% (n=%d)", full_val["wer"], full_val["n_scored"])

    if not args.skip_test:
        test = trainer.compute_wer(splits["test"], description="test")
        results["test_wer"] = test["wer"]
        results["test_n"] = test["n_scored"]
        logger.info("HELD-OUT TEST WER: %.2f%%  (n=%d)", test["wer"], test["n_scored"])

        preds_path = os.path.join(args.output_dir, f"test_transcriptions_seed{args.seed}.csv")
        import pandas as pd
        pd.DataFrame({"reference": test["references"], "prediction": test["predictions"]}) \
            .to_csv(preds_path, index=False)
        logger.info("Test transcriptions written to %s", preds_path)

    if args.fleurs_csv and args.fleurs_audio_dir:
        logger.info("Running out-of-corpus FLEURS evaluation")
        fleurs = load_splits(args.fleurs_csv, args.fleurs_audio_dir,
                             n_samples=10 ** 9, split_seed=args.split_seed)
        # load_splits partitions; for FLEURS we want every clip, so re-merge.
        from datasets import concatenate_datasets
        fleurs_all = concatenate_datasets([fleurs["train"], fleurs["val"], fleurs["test"]])
        fleurs_all.set_transform(eval_transform)
        fl = trainer.compute_wer(fleurs_all, description="fleurs")
        results["fleurs_wer"] = fl["wer"]
        results["fleurs_n"] = fl["n_scored"]
        logger.info("FLEURS (out-of-corpus) WER: %.2f%%  (n=%d)", fl["wer"], fl["n_scored"])

    results_path = os.path.join(args.output_dir, f"results_seed{args.seed}.json")
    with open(results_path, "w") as fh:
        json.dump(results, fh, indent=2)
    logger.info("Results written to %s", results_path)

    print("\n" + "=" * 68)
    print(f"  base model     : {args.base_model}")
    print(f"  quantization   : {results['quantization']}")
    print(f"  augmentation   : {results['augmentation']}")
    print(f"  LoRA           : {'layer-adaptive' if args.layer_adaptive else f'uniform r={args.lora_r}'}")
    print(f"  seed           : {args.seed}   decoding: {results['decoding']}")
    print(f"  validation WER : {results['val_wer_full']:.2f}%")
    if "test_wer" in results:
        print(f"  TEST WER       : {results['test_wer']:.2f}%   <-- report this one")
    print("=" * 68)
    print("  Repeat with --seed 1 --seed 2 ... and report mean +/- SD. Thesis §4.7")
    print("  showed single-run differences of ~1 point are inside seed variance.")
    print("=" * 68 + "\n")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 Swahili ASR — QLoRA (4-bit NF4) fine-tuning of Whisper with Gaussian-noise augmentation -*-

from __future__ import annotations

import argparse
import dataclasses
import gc
import json
import logging
import math
import os
import re
import sys
import time
from typing import Any, Dict, List, Optional

import numpy as np

logger = logging.getLogger("swahili_qlora")

SAMPLING_RATE = 16_000
THESIS_SPLIT_SEED = 42
NOISE_MIN_AMPLITUDE = 0.001
NOISE_MAX_AMPLITUDE = 0.010
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]



_PUNCT_RE = re.compile(r"[^\w\s']")
_WS_RE = re.compile(r"\s+")


def normalize_text(text: Any) -> str:
    """Lower-case, drop punctuation except the apostrophe, collapse whitespace.

    """
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""
    text = str(text).lower().strip()
    text = _PUNCT_RE.sub("", text)
    text = _WS_RE.sub(" ", text)
    return text.strip()


# --------------------------------------------------------------------------------------
# Augmentation — additive Gaussian noise only (thesis Eq. 3.4)
# --------------------------------------------------------------------------------------

def add_gaussian_noise(
    samples: np.ndarray,
    rng: np.random.Generator,
    p: float = 0.3,
    min_amplitude: float = NOISE_MIN_AMPLITUDE,
    max_amplitude: float = NOISE_MAX_AMPLITUDE,
) -> np.ndarray:
    """x'[n] = x[n] + e[n],  e ~ N(0, sigma^2),  sigma ~ U(min_amplitude, max_amplitude).

    Matches `audiomentations.AddGaussianNoise(min_amplitude, max_amplitude, p)` as used
    in the thesis, but implemented directly so the run has one fewer dependency and the
    noise draw is reproducible from a single seeded Generator.
    """
    if p <= 0.0 or rng.random() >= p:
        return samples
    sigma = rng.uniform(min_amplitude, max_amplitude)
    noise = rng.standard_normal(samples.shape).astype(np.float32)
    return (samples + sigma * noise).astype(np.float32)


# --------------------------------------------------------------------------------------
# WER metric
# --------------------------------------------------------------------------------------

class _JiwerWer:


    def compute(self, predictions: List[str], references: List[str]) -> float:
        import jiwer
        return float(jiwer.wer(list(references), list(predictions)))


def load_wer_metric():
    """Prefer the `evaluate` metric (as the thesis did); fall back to jiwer."""
    try:
        import evaluate as hf_evaluate
        metric = hf_evaluate.load("wer")
        logger.info("WER metric: huggingface `evaluate`")
        return metric
    except Exception as exc:
        logger.warning("Could not load `evaluate` WER (%s); using jiwer directly "
                       "(numerically identical).", type(exc).__name__)
        return _JiwerWer()


# --------------------------------------------------------------------------------------
# Log noise
# --------------------------------------------------------------------------------------

_LENGTH_WARNING_FILTERED = False


def silence_generation_length_warning() -> None:

    global _LENGTH_WARNING_FILTERED
    if _LENGTH_WARNING_FILTERED:
        return

    class _DropLengthWarning(logging.Filter):
        def filter(self, record: logging.LogRecord) -> bool:
            return "seem to have been set" not in record.getMessage()

    flt = _DropLengthWarning()
    # A filter on a logger only sees records made by that logger, not propagated ones,
    # so attach to the emitting module loggers AND to the library handler.
    for name in ("transformers.generation.utils",
                 "transformers.generation.configuration_utils",
                 "transformers"):
        logging.getLogger(name).addFilter(flt)
    for handler in logging.getLogger("transformers").handlers:
        handler.addFilter(flt)

    _LENGTH_WARNING_FILTERED = True


# --------------------------------------------------------------------------------------
# LoRA / QLoRA configuration
# --------------------------------------------------------------------------------------

def build_lora_kwargs(
    r: int = 32,
    alpha: int = 64,
    dropout: float = 0.05,
    layer_adaptive: bool = False,
    encoder_r: int = 64,
    decoder_r: int = 16,
) -> Dict[str, Any]:

    kwargs: Dict[str, Any] = dict(
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        target_modules=list(LORA_TARGET_MODULES),
        bias="none",
    )
    if layer_adaptive:
        scaling = alpha / r  # preserve alpha/r == 2.0
        kwargs["rank_pattern"] = {
            r"encoder\..*": encoder_r,
            r"decoder\..*": decoder_r,
        }
        kwargs["alpha_pattern"] = {
            r"encoder\..*": int(round(encoder_r * scaling)),
            r"decoder\..*": int(round(decoder_r * scaling)),
        }
    return kwargs


# --------------------------------------------------------------------------------------
# Dataset transform + collator
# --------------------------------------------------------------------------------------

def load_audio(path: str, target_sr: int = SAMPLING_RATE) -> np.ndarray:
    arr = None
    sr = None

    # 1. First attempt: soundfile
    try:
        import soundfile as sf
        arr, sr = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        arr = None

    # 2. Second attempt: librosa (Wrapped in try-except to prevent the crash)
    if arr is None:
        try:
            import librosa
            arr, sr = librosa.load(path, sr=None, mono=True)
        except Exception:
            arr = None

    # 3. CRASH PREVENTION: If the file is completely corrupted or unreadable
    if arr is None or len(arr) == 0:
        logger = logging.getLogger("swahili_qlora")
        logger.error(f"CORRUPT AUDIO FILE DETECTED & SKIPPED: {path}")
        # Return 1 second of silent audio so the evaluation batch can continue without crashing
        return np.zeros(target_sr, dtype=np.float32)

    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)

    if sr != target_sr:
        try:
            import soxr
            arr = soxr.resample(arr, sr, target_sr)
        except ImportError:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
        arr = np.asarray(arr, dtype=np.float32)

    return arr


def make_transform(processor, augment: bool, seed: int, noise_prob: float):
    """Build an on-the-fly `set_transform` callable.

    Decoding, noise injection and log-Mel extraction all happen per batch access, so
    every epoch sees a fresh noise draw. `augment` must be False for val/test.
    """
    feature_extractor = processor.feature_extractor
    tokenizer = processor.tokenizer
    rng = np.random.default_rng(seed)

    def _transform(batch: Dict[str, List[Any]]) -> Dict[str, List[Any]]:
        paths = batch["audio_path"]
        sentences = batch["sentence"]

        waveforms = []
        for path in paths:
            arr = load_audio(path)
            if augment:
                arr = add_gaussian_noise(arr, rng, p=noise_prob)
            waveforms.append(arr)

        features = feature_extractor(
            waveforms,
            sampling_rate=SAMPLING_RATE,
            return_tensors="np",
        ).input_features

        labels = tokenizer(
            [normalize_text(s) for s in sentences]
        ).input_ids

        return {
            "input_features": [f.astype(np.float32) for f in features],
            "labels": labels,
        }

    return _transform


@dataclasses.dataclass
class DataCollatorSpeechSeq2SeqWithPadding:


    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        import torch

        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        # Whisper's decoder prepends the start token itself; drop it if the tokenizer
        # already added one, or the model sees it twice.
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


# --------------------------------------------------------------------------------------
# Trainer with generation-based WER evaluation
# --------------------------------------------------------------------------------------

def _build_trainer_class():
    """Defined lazily so the module imports without transformers present (for unit tests)."""
    import torch
    from transformers import Seq2SeqTrainer

    class WhisperWERTrainer(Seq2SeqTrainer):

        def __init__(
            self,
            *args,
            processor=None,
            wer_metric=None,
            gen_max_new_tokens: int = 225,
            gen_num_beams: int = 1,
            best_adapter_dir: str = "best_adapter",
            early_stopping_patience: int = 5,
            **kwargs,
        ):
            super().__init__(*args, **kwargs)
            self.processor = processor
            self.wer_metric = wer_metric
            self.gen_max_new_tokens = gen_max_new_tokens
            self.gen_num_beams = gen_num_beams
            self.best_adapter_dir = best_adapter_dir
            self.early_stopping_patience = early_stopping_patience
            self.best_wer = float("inf")
            self.evals_without_improvement = 0
            self.wer_history: List[Dict[str, float]] = []
            silence_generation_length_warning()

            # Carry the best WER across a resume. Without this, best_wer restarts at
            # infinity and the first evaluation of the resumed run overwrites the saved
            # best adapter even when it is worse than what the earlier run achieved.
            marker = os.path.join(self.best_adapter_dir, "best_metric.json")
            if os.path.exists(marker):
                try:
                    with open(marker) as fh:
                        previous = json.load(fh)
                    self.best_wer = float(previous["val_wer"])
                    logger.info("Carried over best val WER %.2f%% (step %s) from the "
                                "interrupted run", self.best_wer, previous.get("step", "?"))
                except Exception as exc:
                    logger.warning("Could not read %s (%s); best WER starts fresh",
                                   marker, type(exc).__name__)

        @torch.no_grad()
        def compute_wer(self, dataset, description: str = "eval") -> Dict[str, Any]:
            model = self.model
            was_training = model.training
            model.eval()

            dataloader = self.get_eval_dataloader(dataset)
            device = self.args.device
            use_cuda = device.type == "cuda"

            predictions: List[str] = []
            references: List[str] = []
            tokenizer = self.processor.tokenizer

            for batch in dataloader:
                input_features = batch["input_features"].to(device)
                with torch.autocast(device_type=device.type,
                                    dtype=torch.float16,
                                    enabled=use_cuda):
                    generated = model.generate(
                        input_features=input_features,
                        max_new_tokens=self.gen_max_new_tokens,
                        num_beams=self.gen_num_beams,
                        use_cache=True,
                    )
                generated = generated.detach().cpu().numpy()

                labels = batch["labels"].cpu().numpy()
                labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

                predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
                references.extend(tokenizer.batch_decode(labels, skip_special_tokens=True))

                del generated, input_features
                if use_cuda:
                    torch.cuda.empty_cache()

            predictions = [normalize_text(p) for p in predictions]
            references = [normalize_text(r) for r in references]

            # An empty reference makes WER undefined; drop those pairs and say how many.
            kept = [(p, r) for p, r in zip(predictions, references) if r]
            dropped = len(references) - len(kept)
            if dropped:
                logger.warning("%s: dropped %d sample(s) with empty reference", description, dropped)
            if not kept:
                raise ValueError(f"{description}: no scorable samples (all references empty)")

            preds, refs = zip(*kept)
            wer = 100.0 * self.wer_metric.compute(predictions=list(preds), references=list(refs))

            if was_training:
                model.train()

            return {"wer": wer, "n_scored": len(kept), "n_dropped": dropped,
                    "predictions": list(preds), "references": list(refs)}

        def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
            dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
            result = self.compute_wer(dataset, description=metric_key_prefix)
            wer = result["wer"]

            metrics = {
                f"{metric_key_prefix}_wer": wer,
                f"{metric_key_prefix}_n_scored": result["n_scored"],
            }
            self.log(metrics)
            self.wer_history.append({"step": int(self.state.global_step), "wer": wer})

            if wer < self.best_wer:
                self.best_wer = wer
                self.evals_without_improvement = 0
                os.makedirs(self.best_adapter_dir, exist_ok=True)
                # Save the adapter only — the 4-bit backbone is reloaded from the hub.
                unwrapped = self.accelerator.unwrap_model(self.model)
                unwrapped.save_pretrained(self.best_adapter_dir)
                self.processor.save_pretrained(self.best_adapter_dir)
                with open(os.path.join(self.best_adapter_dir, "best_metric.json"), "w") as fh:
                    json.dump({"step": int(self.state.global_step), "val_wer": wer}, fh, indent=2)
                logger.info("New best val WER %.2f%% at step %d -> saved to %s",
                            wer, self.state.global_step, self.best_adapter_dir)
            else:
                self.evals_without_improvement += 1
                logger.info("No improvement (%.2f%% vs best %.2f%%), patience %d/%d",
                            wer, self.best_wer,
                            self.evals_without_improvement, self.early_stopping_patience)
                if self.evals_without_improvement >= self.early_stopping_patience:
                    logger.info("Early stopping triggered.")
                    self.control.should_training_stop = True

            return metrics

    return WhisperWERTrainer


# --------------------------------------------------------------------------------------
# Data loading
# --------------------------------------------------------------------------------------

def load_splits(csv_path: str, audio_dir: str, n_samples: int, split_seed: int):

    import pandas as pd
    from datasets import Dataset, DatasetDict
    from sklearn.model_selection import train_test_split

    df = pd.read_csv(csv_path).head(n_samples)
    logger.info("Loaded %d rows from %s", len(df), csv_path)

    df["path"] = df["path"].apply(
        lambda p: os.path.join(audio_dir, os.path.basename(str(p)))
    )
    df["transcript"] = df["transcript"].apply(normalize_text)
    df = df[df["transcript"].notnull() & (df["transcript"] != "")]

    exists = df["path"].apply(os.path.exists)
    if (~exists).any():
        logger.warning("Dropping %d row(s) whose audio file is missing", int((~exists).sum()))
        df = df[exists]

    if len(df) == 0:
        raise ValueError("No usable rows after cleaning — check --csv and --audio-dir.")

    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=split_seed, shuffle=True)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=split_seed, shuffle=True)

    def to_ds(frame):
        # Paths stay plain strings; decoding happens in the transform (see load_audio).
        return Dataset.from_dict({
            "audio_path": frame["path"].tolist(),
            "sentence": frame["transcript"].tolist(),
        })

    splits = DatasetDict({"train": to_ds(train_df), "val": to_ds(val_df), "test": to_ds(test_df)})
    logger.info("Split sizes — train %d, val %d, test %d",
                len(splits["train"]), len(splits["val"]), len(splits["test"]))
    return splits


# --------------------------------------------------------------------------------------
# Environment preflight
# --------------------------------------------------------------------------------------

def find_latest_checkpoint(output_dir: str) -> Optional[str]:
    """Return the highest-numbered `checkpoint-N` directory in output_dir, or None."""
    if not os.path.isdir(output_dir):
        return None
    found = []
    for name in os.listdir(output_dir):
        match = re.fullmatch(r"checkpoint-(\d+)", name)
        path = os.path.join(output_dir, name)
        if match and os.path.isdir(path):
            found.append((int(match.group(1)), path))
    if not found:
        return None
    return max(found)[1]


def check_output_dir_persistence(output_dir: str) -> None:
    """Warn loudly if checkpoints are going to the ephemeral Colab VM disk.

    A Colab VM that disconnects or is reclaimed takes /content with it. A multi-hour
    run whose --output-dir is not under /content/drive has nothing to resume from.
    """
    abs_dir = os.path.abspath(output_dir)
    on_colab = os.path.isdir("/content")
    persistent = abs_dir.startswith(("/content/drive", "/gdrive", "/content/gdrive"))
    if on_colab and not persistent:
        logger.warning(
            "\n" + "!" * 72 +
            f"\n  --output-dir is {abs_dir}"
            "\n  That is the EPHEMERAL VM disk. If this runtime disconnects or is"
            "\n  reclaimed, every checkpoint is lost and there is nothing to resume."
            "\n  For a multi-hour run, mount Drive and point --output-dir at it:"
            "\n      from google.colab import drive; drive.mount('/content/drive')"
            "\n      --output-dir /content/drive/MyDrive/swahili-asr/qlora-large-v3"
            "\n" + "!" * 72
        )
    elif persistent:
        logger.info("Checkpoints        : %s (persistent)", abs_dir)


def preflight(use_4bit: bool, base_model: str) -> None:

    import torch

    if not torch.cuda.is_available():
        raise SystemExit(
            "No CUDA GPU visible.\n"
            "  Colab: Runtime > Change runtime type > T4 GPU, then re-run.\n"
            "  (Pass --no-4bit only if you genuinely intend to train on CPU/fp32.)"
        )

    name = torch.cuda.get_device_name(0)
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    major, _ = torch.cuda.get_device_capability(0)
    logger.info("GPU                : %s (%.1f GB, sm_%d.x)", name, total_gb, major)
    if major < 8:
        logger.info("                     Turing or older — fp16 compute, no bf16. Handled.")

    if not use_4bit:
        if "large" in base_model and total_gb < 24:
            logger.warning(
                "--no-4bit with %s on a %.0f GB card will almost certainly OOM. "
                "Drop --no-4bit to use NF4.", base_model, total_gb
            )
        return

    try:
        import bitsandbytes  # noqa: F401
    except Exception as exc:
        detail = str(exc)
        if "triton.ops" in detail:
            raise SystemExit(
                "bitsandbytes is too old for the installed Triton.\n"
                "  `triton.ops` was removed in Triton 3.x, but bitsandbytes < 0.44 still\n"
                "  imports it, so the import fails before any of your code runs.\n\n"
                "  Fix (both steps — the second is not optional):\n"
                "      !pip install -q -U bitsandbytes\n"
                "      then Runtime > Restart session\n\n"
                "  The restart matters: the failed import is cached in sys.modules and in\n"
                "  transformers' lazy-module table, so a reinstall alone changes nothing."
            ) from exc
        raise SystemExit(
            f"bitsandbytes could not be imported: {type(exc).__name__}: {detail}\n"
            "  Try:  !pip install -q -U bitsandbytes   then restart the session."
        ) from exc

    logger.info("bitsandbytes       : %s", getattr(bitsandbytes, "__version__", "unknown"))


# --------------------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------------------

def load_quantized_model(base_model: str, use_4bit: bool, gradient_checkpointing: bool):
    import torch
    from transformers import WhisperForConditionalGeneration

    if use_4bit:
        from transformers import BitsAndBytesConfig

        # fp16 compute, not bf16: the Tesla T4 is Turing (sm_75) and has no bf16 support.
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = WhisperForConditionalGeneration.from_pretrained(
            base_model, quantization_config=quant_config, device_map="auto"
        )
        from peft import prepare_model_for_kbit_training
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=gradient_checkpointing
        )
    else:
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = WhisperForConditionalGeneration.from_pretrained(base_model, torch_dtype=dtype)
        if torch.cuda.is_available():
            model = model.to("cuda")

    if gradient_checkpointing:
        model.gradient_checkpointing_enable()
        # Required so gradients reach the adapters through a frozen, checkpointed backbone.
        model.enable_input_require_grads()

    return model


def configure_generation(model, processor, language: str = "sw", task: str = "transcribe"):
    """Pin the decoder prompt to Swahili transcription and clear legacy forced ids."""
    model.generation_config.language = language
    model.generation_config.task = task
    model.generation_config.forced_decoder_ids = None
    if hasattr(model.config, "forced_decoder_ids"):
        model.config.forced_decoder_ids = None
    # Deliberately NOT clearing `suppress_tokens`. Older Whisper fine-tuning notebooks
    # set `config.suppress_tokens = []`; that both trips a deprecation path in current
    # transformers and removes Whisper's non-speech token suppression, which costs WER.
    return model


# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------

def _resolve_argv(argv: Optional[List[str]]) -> List[str]:
    """Return the argument list to parse, tolerating a Jupyter/Colab kernel.

    A notebook kernel is launched as `... -f /root/.../kernel-<uuid>.json`, so a bare
    `main()` called from a cell would otherwise see that flag in `sys.argv` and argparse
    would exit with "unrecognized arguments". Inside a kernel there are no real
    command-line arguments anyway, so pass them explicitly:

        from swahili_asr_qlora import main
        main(["--base-model", "openai/whisper-large-v3", "--epochs", "5"])
    """
    if argv is not None:
        return list(argv)

    raw = list(sys.argv[1:])
    in_kernel = "ipykernel" in sys.modules or "google.colab" in sys.modules
    if not in_kernel:
        return raw

    cleaned: List[str] = []
    skip_next = False
    for token in raw:
        if skip_next:
            skip_next = False
            continue
        if token == "-f":
            skip_next = True          # drop the kernel connection-file path that follows
            continue
        if token.startswith("--f=") or token.startswith("-f="):
            continue
        if token.endswith(".json") and "kernel" in os.path.basename(token):
            continue
        cleaned.append(token)

    if cleaned:
        logger.warning("Running inside a notebook kernel; parsing %r from sys.argv. "
                       "Prefer passing arguments explicitly: main([...])", cleaned)
    return cleaned


def parse_args(argv: Optional[List[str]] = None) -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="QLoRA fine-tuning of Whisper for Swahili ASR (Gaussian-noise augmentation only).",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )

    # Data
    p.add_argument("--csv", default="/content/drive/MyDrive/Colab/New Swahili Dataset/"
                                    "mozilla-swa-dataset/swahili_dataset.csv",
                   help="CSV with 'path' and 'transcript' columns.")
    p.add_argument("--audio-dir", default="/content/drive/MyDrive/Colab/New Swahili Dataset/"
                                          "mozilla-swa-dataset/clips",
                   help="Directory holding the audio clips.")
    p.add_argument("--n-samples", type=int, default=9000,
                   help="Clips to take from the head of the CSV (9000 = thesis subset).")
    p.add_argument("--split-seed", type=int, default=THESIS_SPLIT_SEED,
                   help="Keep at 42 to reproduce the thesis split exactly.")

    # Model / PEFT
    p.add_argument("--base-model", default="openai/whisper-large-v3",
                   help="HF model id. Use openai/whisper-small for the like-for-like arm.")
    p.add_argument("--no-4bit", action="store_true",
                   help="Disable NF4 quantization (only if you have an A100/L4 with headroom).")
    p.add_argument("--lora-r", type=int, default=32)
    p.add_argument("--lora-alpha", type=int, default=64)
    p.add_argument("--lora-dropout", type=float, default=0.05)
    p.add_argument("--layer-adaptive", action="store_true",
                   help="Layer-adaptive ranks (Han et al. 2024): high encoder rank, low decoder rank.")
    p.add_argument("--encoder-r", type=int, default=64)
    p.add_argument("--decoder-r", type=int, default=16)

    # Augmentation
    p.add_argument("--noise-prob", type=float, default=0.3,
                   help="Probability of applying Gaussian noise to a training clip.")
    p.add_argument("--no-augment", action="store_true",
                   help="Train without augmentation (control arm).")

    # Optimisation
    p.add_argument("--epochs", type=float, default=5.0,
                   help="25 matches the thesis for whisper-small; 4-6 is ample for large-v3.")
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--grad-accum", type=int, default=16,
                   help="batch-size * grad-accum = 32 reproduces the thesis effective batch.")
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--warmup-steps", type=int, default=200)
    p.add_argument("--weight-decay", type=float, default=0.01)
    p.add_argument("--lr-scheduler", default="linear",
                   choices=["linear", "cosine", "constant_with_warmup"])
    p.add_argument("--no-gradient-checkpointing", action="store_true")
    p.add_argument("--seed", type=int, default=42,
                   help="Training seed. Vary this across >=3 runs before ranking anything.")

    # Evaluation
    p.add_argument("--eval-steps", type=int, default=250)
    p.add_argument("--save-steps", type=int, default=100,
                   help="Checkpoint interval. Kept well below --eval-steps on purpose: a "
                        "checkpoint is cheap, and it caps how much wall-clock a crash costs. "
                        "Evaluation runs generation and is the expensive one.")
    p.add_argument("--eval-subset", type=int, default=300,
                   help="Val clips used for mid-training WER (generation is slow). "
                        "0 = use the full validation split.")
    p.add_argument("--num-beams", type=int, default=1,
                   help="1 = greedy, matching the submitted thesis §3.7 protocol.")
    p.add_argument("--max-new-tokens", type=int, default=225)
    p.add_argument("--early-stopping-patience", type=int, default=5)
    p.add_argument("--fleurs-csv", default=None,
                   help="Optional CSV for out-of-corpus FLEURS evaluation.")
    p.add_argument("--fleurs-audio-dir", default=None)

    # Bookkeeping
    p.add_argument("--output-dir", default="./whisper-qlora-gaussian",
                   help="Put this on Google Drive for any run over ~1 hour, or a "
                        "disconnect loses every checkpoint.")
    p.add_argument("--resume", default="auto",
                   help="'auto' picks up the newest checkpoint in --output-dir if one "
                        "exists and starts fresh otherwise; 'none' always starts from "
                        "scratch; or pass an explicit checkpoint-N path.")
    p.add_argument("--ignore-data-skip", action="store_true",
                   help="On resume, do not fast-forward the dataloader to the saved step. "
                        "Saves replaying (and re-decoding) every batch already seen, at the "
                        "cost of a different sample order for the remainder of that epoch.")
    p.add_argument("--skip-test", action="store_true",
                   help="Skip the held-out test evaluation (for quick sanity runs).")

    return p.parse_args(_resolve_argv(argv))


def main(argv: Optional[List[str]] = None) -> int:
    args = parse_args(argv)

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s  %(levelname)-7s  %(message)s",
        datefmt="%H:%M:%S",
        stream=sys.stdout,
    )

    import torch
    from peft import LoraConfig, get_peft_model
    from transformers import Seq2SeqTrainingArguments, WhisperProcessor, set_seed

    use_4bit = not args.no_4bit
    preflight(use_4bit, args.base_model)
    check_output_dir_persistence(args.output_dir)

    set_seed(args.seed)
    os.makedirs(args.output_dir, exist_ok=True)

    logger.info("Base model         : %s", args.base_model)
    logger.info("Quantization       : %s", "4-bit NF4 (double quant, fp16 compute)" if use_4bit else "none")
    logger.info("Augmentation       : %s", "none" if args.no_augment
                else f"additive Gaussian noise, p={args.noise_prob}, sigma~U(0.001,0.010)")
    logger.info("LoRA               : %s", "layer-adaptive" if args.layer_adaptive
                else f"uniform r={args.lora_r}, alpha={args.lora_alpha}")
    logger.info("Seed               : %d", args.seed)

    # ---- data -------------------------------------------------------------------------
    processor = WhisperProcessor.from_pretrained(
        args.base_model, language="Swahili", task="transcribe"
    )
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

    splits = load_splits(args.csv, args.audio_dir, args.n_samples, args.split_seed)

    splits["train"].set_transform(
        make_transform(processor, augment=not args.no_augment,
                       seed=args.seed, noise_prob=args.noise_prob)
    )
    eval_transform = make_transform(processor, augment=False, seed=args.seed, noise_prob=0.0)
    splits["val"].set_transform(eval_transform)
    splits["test"].set_transform(eval_transform)

    val_for_training = splits["val"]
    if args.eval_subset and args.eval_subset < len(splits["val"]):
        val_for_training = splits["val"].select(range(args.eval_subset))
        val_for_training.set_transform(eval_transform)
        logger.info("Mid-training WER measured on a %d-clip validation subset; "
                    "the full %d-clip split is scored once at the end.",
                    args.eval_subset, len(splits["val"]))

    # ---- model ------------------------------------------------------------------------
    gradient_checkpointing = not args.no_gradient_checkpointing
    model = load_quantized_model(args.base_model, use_4bit, gradient_checkpointing)
    model = configure_generation(model, processor)

    lora_kwargs = build_lora_kwargs(
        r=args.lora_r, alpha=args.lora_alpha, dropout=args.lora_dropout,
        layer_adaptive=args.layer_adaptive,
        encoder_r=args.encoder_r, decoder_r=args.decoder_r,
    )
    model = get_peft_model(model, LoraConfig(task_type=None, **lora_kwargs))
    model.print_trainable_parameters()
    model.config.use_cache = False  # incompatible with gradient checkpointing

    collator = DataCollatorSpeechSeq2SeqWithPadding(
        processor=processor,
        decoder_start_token_id=model.config.decoder_start_token_id,
    )

    # ---- training ---------------------------------------------------------------------
    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_steps=args.warmup_steps,
        num_train_epochs=args.epochs,
        lr_scheduler_type=args.lr_scheduler,
        weight_decay=args.weight_decay,
        fp16=torch.cuda.is_available(),
        gradient_checkpointing=gradient_checkpointing,
        eval_strategy="steps",
        eval_steps=args.eval_steps,
        save_strategy="steps",
        save_steps=args.save_steps,
        save_total_limit=2,
        ignore_data_skip=args.ignore_data_skip,
        logging_steps=25,
        predict_with_generate=False,   # generation is handled by WhisperWERTrainer
        load_best_model_at_end=False,  # best adapter is saved explicitly instead
        remove_unused_columns=False,   # set_transform supplies the columns
        label_names=["labels"],
        dataloader_num_workers=2,
        seed=args.seed,
        report_to=[],
    )

    wer_metric = load_wer_metric()
    best_adapter_dir = os.path.join(args.output_dir, "best_adapter")
    WhisperWERTrainer = _build_trainer_class()

    trainer = WhisperWERTrainer(
        model=model,
        args=training_args,
        train_dataset=splits["train"],
        eval_dataset=val_for_training,
        data_collator=collator,
        processing_class=processor.feature_extractor,
        processor=processor,
        wer_metric=wer_metric,
        gen_max_new_tokens=args.max_new_tokens,
        gen_num_beams=args.num_beams,
        best_adapter_dir=best_adapter_dir,
        early_stopping_patience=args.early_stopping_patience,
    )

    resume_from: Optional[str] = None
    if args.resume in ("none", "None", ""):
        logger.info("Resume             : disabled (--resume none) — starting from scratch")
    elif args.resume == "auto":
        resume_from = find_latest_checkpoint(args.output_dir)
        if resume_from:
            logger.info("Resume             : %s  (pass --resume none to start over)",
                        resume_from)
        else:
            logger.info("Resume             : no checkpoint found in %s — starting fresh",
                        args.output_dir)
    else:
        resume_from = args.resume
        if not os.path.isdir(resume_from):
            raise SystemExit(
                f"--resume {resume_from} is not a directory.\n"
                f"  Checkpoints found in {args.output_dir}: "
                f"{find_latest_checkpoint(args.output_dir) or 'none'}\n"
                "  Use --resume auto to pick the newest automatically."
            )
        logger.info("Resume             : %s (explicit)", resume_from)

    started = time.time()
    trainer.train(resume_from_checkpoint=resume_from)
    train_minutes = (time.time() - started) / 60.0
    logger.info("Training finished in %.1f min. Best validation WER: %.2f%%",
                train_minutes, trainer.best_wer)

    # ---- final evaluation -------------------------------------------------------------
    results: Dict[str, Any] = {
        "base_model": args.base_model,
        "quantization": "nf4-4bit" if use_4bit else "none",
        "augmentation": "none" if args.no_augment else "gaussian_noise",
        "noise_prob": None if args.no_augment else args.noise_prob,
        "lora": lora_kwargs,
        "layer_adaptive": args.layer_adaptive,
        "seed": args.seed,
        "epochs": args.epochs,
        "effective_batch_size": args.batch_size * args.grad_accum,
        "learning_rate": args.lr,
        "num_beams": args.num_beams,
        "decoding": "greedy" if args.num_beams == 1 else f"beam-{args.num_beams}",
        "best_val_wer_subset": trainer.best_wer,
        "val_wer_history": trainer.wer_history,
        "train_minutes": round(train_minutes, 1),
    }

    if os.path.isdir(best_adapter_dir):
        logger.info("Reloading best adapter from %s for final scoring", best_adapter_dir)
        from peft import PeftModel

        # Free the training model and its optimizer state first. Holding both copies
        # plus Adam moments is what pushes a T4 over the edge on large-v3.
        trainer.model = None
        trainer.optimizer = None
        trainer.lr_scheduler = None
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        base = load_quantized_model(args.base_model, use_4bit, gradient_checkpointing=False)
        base = configure_generation(base, processor)
        best_model = PeftModel.from_pretrained(base, best_adapter_dir)
        best_model.eval()
        trainer.model = best_model

    full_val = trainer.compute_wer(splits["val"], description="val_full")
    results["val_wer_full"] = full_val["wer"]
    logger.info("Full validation WER: %.2f%% (n=%d)", full_val["wer"], full_val["n_scored"])

    if not args.skip_test:
        test = trainer.compute_wer(splits["test"], description="test")
        results["test_wer"] = test["wer"]
        results["test_n"] = test["n_scored"]
        logger.info("HELD-OUT TEST WER: %.2f%%  (n=%d)", test["wer"], test["n_scored"])

        preds_path = os.path.join(args.output_dir, f"test_transcriptions_seed{args.seed}.csv")
        import pandas as pd
        pd.DataFrame({"reference": test["references"], "prediction": test["predictions"]}) \
            .to_csv(preds_path, index=False)
        logger.info("Test transcriptions written to %s", preds_path)

    if args.fleurs_csv and args.fleurs_audio_dir:
        logger.info("Running out-of-corpus FLEURS evaluation")
        fleurs = load_splits(args.fleurs_csv, args.fleurs_audio_dir,
                             n_samples=10 ** 9, split_seed=args.split_seed)
        # load_splits partitions; for FLEURS we want every clip, so re-merge.
        from datasets import concatenate_datasets
        fleurs_all = concatenate_datasets([fleurs["train"], fleurs["val"], fleurs["test"]])
        fleurs_all.set_transform(eval_transform)
        fl = trainer.compute_wer(fleurs_all, description="fleurs")
        results["fleurs_wer"] = fl["wer"]
        results["fleurs_n"] = fl["n_scored"]
        logger.info("FLEURS (out-of-corpus) WER: %.2f%%  (n=%d)", fl["wer"], fl["n_scored"])

    results_path = os.path.join(args.output_dir, f"results_seed{args.seed}.json")
    with open(results_path, "w") as fh:
        json.dump(results, fh, indent=2)
    logger.info("Results written to %s", results_path)

    print("\n" + "=" * 68)
    print(f"  base model     : {args.base_model}")
    print(f"  quantization   : {results['quantization']}")
    print(f"  augmentation   : {results['augmentation']}")
    print(f"  LoRA           : {'layer-adaptive' if args.layer_adaptive else f'uniform r={args.lora_r}'}")
    print(f"  seed           : {args.seed}   decoding: {results['decoding']}")
    print(f"  validation WER : {results['val_wer_full']:.2f}%")
    if "test_wer" in results:
        print(f"  TEST WER       : {results['test_wer']:.2f}%   <-- report this one")
    print("=" * 68)
    print("  Repeat with --seed 1 --seed 2 ... and report mean +/- SD. Thesis §4.7")
    print("  showed single-run differences of ~1 point are inside seed variance.")
    print("=" * 68 + "\n")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
!pip install -q --upgrade "transformers>=4.46" "peft>=0.13" "accelerate>=1.0"
!pip install -q --upgrade "datasets>=3.0" "bitsandbytes>=0.46"
!pip install -q --upgrade "evaluate>=0.4" "jiwer>=3.0"
!pip install -q --upgrade "soundfile>=0.12" "librosa>=0.10" "soxr>=0.3"

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Swahili ASR — QLoRA (4-bit NF4) fine-tuning of whisper-large-v3 with Layer-Adaptive
Rank Allocation and Gaussian-noise augmentation.
"""

from __future__ import annotations

import argparse
import dataclasses
import gc
import json
import logging
import math
import os
import re
import sys
import time
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import wandb

logger = logging.getLogger("swahili_qlora")

# --------------------------------------------------------------------------------------
# Constants
# --------------------------------------------------------------------------------------
SAMPLING_RATE = 16_000
THESIS_SPLIT_SEED = 42
NOISE_MIN_AMPLITUDE = 0.001
NOISE_MAX_AMPLITUDE = 0.010
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"]

_PUNCT_RE = re.compile(r"[^\w\s']")
_WS_RE = re.compile(r"\s+")

def normalize_text(text: Any) -> str:
    if text is None or (isinstance(text, float) and math.isnan(text)):
        return ""
    text = str(text).lower().strip()
    text = _PUNCT_RE.sub("", text)
    text = _WS_RE.sub(" ", text)
    return text.strip()

# --------------------------------------------------------------------------------------
# Augmentation — Additive Gaussian noise only
# --------------------------------------------------------------------------------------
def add_gaussian_noise(
    samples: np.ndarray,
    rng: np.random.Generator,
    p: float = 0.3,
    min_amplitude: float = NOISE_MIN_AMPLITUDE,
    max_amplitude: float = NOISE_MAX_AMPLITUDE,
) -> np.ndarray:
    if p <= 0.0 or rng.random() >= p:
        return samples
    sigma = rng.uniform(min_amplitude, max_amplitude)
    noise = rng.standard_normal(samples.shape).astype(np.float32)
    return (samples + sigma * noise).astype(np.float32)

# --------------------------------------------------------------------------------------
# WER metric
# --------------------------------------------------------------------------------------
class _JiwerWer:
    def compute(self, predictions: List[str], references: List[str]) -> float:
        import jiwer
        return float(jiwer.wer(list(references), list(predictions)))

def load_wer_metric():
    try:
        import evaluate as hf_evaluate
        metric = hf_evaluate.load("wer")
        logger.info("WER metric: huggingface `evaluate`")
        return metric
    except Exception as exc:
        logger.warning("Could not load `evaluate` WER; using jiwer directly.")
        return _JiwerWer()

_LENGTH_WARNING_FILTERED = False

def silence_generation_length_warning() -> None:
    global _LENGTH_WARNING_FILTERED
    if _LENGTH_WARNING_FILTERED:
        return
    class _DropLengthWarning(logging.Filter):
        def filter(self, record: logging.LogRecord) -> bool:
            return "seem to have been set" not in record.getMessage()
    flt = _DropLengthWarning()
    for name in ("transformers.generation.utils", "transformers.generation.configuration_utils", "transformers"):
        logging.getLogger(name).addFilter(flt)
    for handler in logging.getLogger("transformers").handlers:
        handler.addFilter(flt)
    _LENGTH_WARNING_FILTERED = True

# --------------------------------------------------------------------------------------
# Layer-Adaptive LoRA Configuration
# --------------------------------------------------------------------------------------
def build_lora_kwargs(
    alpha: int = 64,
    dropout: float = 0.05,
    encoder_r: int = 64,
    decoder_r: int = 16,
) -> Dict[str, Any]:
    kwargs: Dict[str, Any] = dict(
        lora_dropout=dropout,
        target_modules=list(LORA_TARGET_MODULES),
        bias="none",
    )

    scaling_factor = alpha / max(encoder_r, decoder_r)

    kwargs["rank_pattern"] = {
        r"encoder\..*": encoder_r,
        r"decoder\..*": decoder_r,
    }
    kwargs["alpha_pattern"] = {
        r"encoder\..*": int(round(encoder_r * scaling_factor)),
        r"decoder\..*": int(round(decoder_r * scaling_factor)),
    }
    return kwargs

# --------------------------------------------------------------------------------------
# Dataset transform + collator
# --------------------------------------------------------------------------------------
def load_audio(path: str, target_sr: int = SAMPLING_RATE) -> np.ndarray:
    arr = None
    sr = None

    # 1. First attempt: soundfile
    try:
        import soundfile as sf
        arr, sr = sf.read(path, dtype="float32", always_2d=False)
    except Exception:
        arr = None

    # 2. Second attempt: librosa (Wrapped in try-except to prevent the crash)
    if arr is None:
        try:
            import librosa
            arr, sr = librosa.load(path, sr=None, mono=True)
        except Exception:
            arr = None

    # 3. CRASH PREVENTION: If the file is completely corrupted or unreadable
    if arr is None or len(arr) == 0:
        logger = logging.getLogger("swahili_qlora")
        logger.error(f"CORRUPT AUDIO FILE DETECTED & SKIPPED: {path}")
        # Return 1 second of silent audio so the evaluation batch can continue without crashing
        return np.zeros(target_sr, dtype=np.float32)

    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim > 1:
        arr = arr.mean(axis=1)

    if sr != target_sr:
        try:
            import soxr
            arr = soxr.resample(arr, sr, target_sr)
        except ImportError:
            import librosa
            arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)
        arr = np.asarray(arr, dtype=np.float32)

    return arr

def make_transform(processor, augment: bool, seed: int, noise_prob: float):
    feature_extractor = processor.feature_extractor
    tokenizer = processor.tokenizer
    rng = np.random.default_rng(seed)

    def _transform(batch: Dict[str, List[Any]]) -> Dict[str, List[Any]]:
        paths = batch["audio_path"]
        sentences = batch["sentence"]
        waveforms = []
        for path in paths:
            arr = load_audio(path)
            if augment:
                arr = add_gaussian_noise(arr, rng, p=noise_prob)
            waveforms.append(arr)

        features = feature_extractor(
            waveforms, sampling_rate=SAMPLING_RATE, return_tensors="np",
        ).input_features

        labels = tokenizer([normalize_text(s) for s in sentences]).input_ids
        return {
            "input_features": [f.astype(np.float32) for f in features],
            "labels": labels,
        }
    return _transform

@dataclasses.dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

# --------------------------------------------------------------------------------------
# Trainer
# --------------------------------------------------------------------------------------
def _build_trainer_class():
    from transformers import Seq2SeqTrainer
    class WhisperWERTrainer(Seq2SeqTrainer):
        def __init__(
            self, *args, processor=None, wer_metric=None,
            gen_max_new_tokens: int = 225, gen_num_beams: int = 5,
            best_adapter_dir: str = "best_adapter", early_stopping_patience: int = 5,
            **kwargs,
        ):
            super().__init__(*args, **kwargs)
            self.processor = processor
            self.wer_metric = wer_metric
            self.gen_max_new_tokens = gen_max_new_tokens
            self.gen_num_beams = gen_num_beams
            self.best_adapter_dir = best_adapter_dir
            self.early_stopping_patience = early_stopping_patience
            self.best_wer = float("inf")
            self.evals_without_improvement = 0
            self.wer_history: List[Dict[str, float]] = []
            silence_generation_length_warning()

            marker = os.path.join(self.best_adapter_dir, "best_metric.json")
            if os.path.exists(marker):
                try:
                    with open(marker) as fh:
                        previous = json.load(fh)
                    self.best_wer = float(previous["val_wer"])
                    logger.info("Carried over best val WER %.2f%%", self.best_wer)
                except Exception:
                    pass

        @torch.no_grad()
        def compute_wer(self, dataset, description: str = "eval") -> Dict[str, Any]:
            model = self.model
            was_training = model.training
            model.eval()
            dataloader = self.get_eval_dataloader(dataset)
            device = self.args.device
            use_cuda = device.type == "cuda"
            predictions, references = [], []
            tokenizer = self.processor.tokenizer

            for batch in dataloader:
                input_features = batch["input_features"].to(device)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_cuda):
                    generated = model.generate(
                        input_features=input_features,
                        max_new_tokens=self.gen_max_new_tokens,
                        num_beams=self.gen_num_beams,
                        use_cache=True,
                    )
                generated = generated.detach().cpu().numpy()
                labels = batch["labels"].cpu().numpy()
                labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
                predictions.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
                references.extend(tokenizer.batch_decode(labels, skip_special_tokens=True))

            predictions = [normalize_text(p) for p in predictions]
            references = [normalize_text(r) for r in references]
            kept = [(p, r) for p, r in zip(predictions, references) if r]

            if not kept:
                raise ValueError(f"{description}: no scorable samples")

            preds, refs = zip(*kept)
            wer = 100.0 * self.wer_metric.compute(predictions=list(preds), references=list(refs))

            if was_training:
                model.train()
            return {"wer": wer, "n_scored": len(kept), "predictions": list(preds), "references": list(refs)}

        def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
            dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
            result = self.compute_wer(dataset, description=metric_key_prefix)
            wer = result["wer"]
            metrics = {f"{metric_key_prefix}_wer": wer, f"{metric_key_prefix}_n_scored": result["n_scored"]}
            self.log(metrics)
            self.wer_history.append({"step": int(self.state.global_step), "wer": wer})

            if wer < self.best_wer:
                self.best_wer = wer
                self.evals_without_improvement = 0
                os.makedirs(self.best_adapter_dir, exist_ok=True)
                unwrapped = self.accelerator.unwrap_model(self.model)
                unwrapped.save_pretrained(self.best_adapter_dir)
                self.processor.save_pretrained(self.best_adapter_dir)
                with open(os.path.join(self.best_adapter_dir, "best_metric.json"), "w") as fh:
                    json.dump({"step": int(self.state.global_step), "val_wer": wer}, fh, indent=2)
                logger.info("New best val WER %.2f%% at step %d", wer, self.state.global_step)
            else:
                self.evals_without_improvement += 1
                if self.evals_without_improvement >= self.early_stopping_patience:
                    logger.info("Early stopping triggered.")
                    self.control.should_training_stop = True
            return metrics
    return WhisperWERTrainer

# --------------------------------------------------------------------------------------
# Model Preflight & Loading
# --------------------------------------------------------------------------------------
def load_splits(csv_path: str, audio_dir: str, n_samples: int, split_seed: int):
    import pandas as pd
    from datasets import Dataset, DatasetDict
    from sklearn.model_selection import train_test_split

    df = pd.read_csv(csv_path).head(n_samples)
    df["path"] = df["path"].apply(lambda p: os.path.join(audio_dir, os.path.basename(str(p))))
    df["transcript"] = df["transcript"].apply(normalize_text)
    df = df[df["transcript"].notnull() & (df["transcript"] != "")]

    exists = df["path"].apply(os.path.exists)
    df = df[exists]

    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=split_seed, shuffle=True)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=split_seed, shuffle=True)

    def to_ds(frame):
        return Dataset.from_dict({"audio_path": frame["path"].tolist(), "sentence": frame["transcript"].tolist()})

    return DatasetDict({"train": to_ds(train_df), "val": to_ds(val_df), "test": to_ds(test_df)})

def load_quantized_model(base_model: str):
    from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = WhisperForConditionalGeneration.from_pretrained(
        base_model, quantization_config=quant_config, device_map="auto"
    )
    from peft import prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    return model

def configure_generation(model, processor, language: str = "sw", task: str = "transcribe"):
    model.generation_config.language = language
    model.generation_config.task = task
    forced_ids = processor.get_decoder_prompt_ids(language=language, task=task)
    model.generation_config.forced_decoder_ids = forced_ids
    if hasattr(model.config, "forced_decoder_ids"):
        model.config.forced_decoder_ids = forced_ids
    return model

# --------------------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------------------
def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    p.add_argument("--csv", default="/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv")
    p.add_argument("--audio-dir", default="/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips")
    p.add_argument("--base-model", default="openai/whisper-large-v3")
    p.add_argument("--encoder-r", type=int, default=64)
    p.add_argument("--decoder-r", type=int, default=16)
    p.add_argument("--lora-alpha", type=int, default=64)
    p.add_argument("--noise-prob", type=float, default=0.3)
    p.add_argument("--epochs", type=float, default=5.0)
    p.add_argument("--batch-size", type=int, default=2)
    p.add_argument("--grad-accum", type=int, default=16)
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--num-beams", type=int, default=5)
    p.add_argument("--output-dir", default="/content/drive/MyDrive/whisper-qlora-gaussian")
    p.add_argument("--wandb-project", default="swahili-asr-qlora-large")
    return p.parse_args(sys.argv[1:] if not "ipykernel" in sys.modules else [])

def main() -> int:
    args = parse_args()
    logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-7s  %(message)s")

    from peft import LoraConfig, get_peft_model
    from transformers import Seq2SeqTrainingArguments, WhisperProcessor, set_seed

    set_seed(args.seed)
    wandb.init(project=args.wandb_project, name=f"large-v3-adaptive-gaussian-s{args.seed}")

    processor = WhisperProcessor.from_pretrained(args.base_model, language="Swahili", task="transcribe")
    if processor.tokenizer.pad_token_id is None:
        processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

    splits = load_splits(args.csv, args.audio_dir, 9000, THESIS_SPLIT_SEED)
    splits["train"].set_transform(make_transform(processor, augment=True, seed=args.seed, noise_prob=args.noise_prob))
    eval_transform = make_transform(processor, augment=False, seed=args.seed, noise_prob=0.0)
    splits["val"].set_transform(eval_transform)
    splits["test"].set_transform(eval_transform)

    model = load_quantized_model(args.base_model)

    # CRITICAL FIX 1: Disable cache on the base model before PEFT application to avoid tensor recomputation mismatch
    model.config.use_cache = False

    model = configure_generation(model, processor)

    lora_kwargs = build_lora_kwargs(alpha=args.lora_alpha, encoder_r=args.encoder_r, decoder_r=args.decoder_r)
    model = get_peft_model(model, LoraConfig(task_type=None, **lora_kwargs))

    collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=model.config.decoder_start_token_id)

    training_args = Seq2SeqTrainingArguments(
        output_dir=args.output_dir,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,
        learning_rate=args.lr,
        warmup_steps=200,
        num_train_epochs=args.epochs,
        fp16=True,

        # CRITICAL FIX 2: Explicitly enable gradient checkpointing with reentrant fallback
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": True},

        eval_strategy="steps",
        eval_steps=250,
        save_strategy="steps",
        save_steps=250,
        logging_steps=25,
        predict_with_generate=False,
        report_to=["wandb"],
        remove_unused_columns=False,
        label_names=["labels"],
        dataloader_num_workers=2,
        seed=args.seed,
    )

    WhisperWERTrainer = _build_trainer_class()
    trainer = WhisperWERTrainer(
        model=model, args=training_args,
        train_dataset=splits["train"], eval_dataset=splits["val"],
        data_collator=collator, processing_class=processor.feature_extractor, processor=processor,
        wer_metric=load_wer_metric(), gen_num_beams=args.num_beams, best_adapter_dir=os.path.join(args.output_dir, "best_adapter"),
    )

    trainer.train()

    # Final Test Set Evaluation
    test = trainer.compute_wer(splits["test"], description="test")
    logger.info("FINAL TEST WER: %.2f%%", test["wer"])
    wandb.log({"final_test_wer": test["wer"]})
    wandb.finish()
    return 0

if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
import json, os
D = "/content/drive/MyDrive/whisper-qlora-gaussian"
!ls -la {D}
print("---")
m = f"{D}/best_adapter/best_metric.json"
print(json.load(open(m)) if os.path.exists(m) else "NO best_metric.json")

In [ ]:
!ls -la --time-style=full-iso /content/drive/MyDrive/whisper-qlora-gaussian/best_adapter/

In [ ]:
!python evaluate_best_adapter.py --splits test val --also-beam5 --batch-size 4

In [ ]:
from google.colab import files
files.upload()          # choose evaluate_best_adapter.py

!mkdir -p "/content/drive/MyDrive/Colab/scripts"
!cp evaluate_best_adapter.py "/content/drive/MyDrive/Colab/scripts/"
!ls -la evaluate_best_adapter.py

In [ ]:
!python evaluate_best_adapter.py --splits test val --also-beam5 --batch-size 4

In [ ]:
from google.colab import files
files.upload()          # choose evaluate_best_adapter.py

!mkdir -p "/content/drive/MyDrive/Colab/scripts"
!cp evaluate_best_adapter.py "/content/drive/MyDrive/Colab/scripts/"
!ls -la evaluate_best_adapter.py

In [ ]:
!rm -f evaluate_best_adapter.py "/content/drive/MyDrive/Colab/scripts/evaluate_best_adapter.py"

from google.colab import files
files.upload()          # choose the new evaluate_best_adapter.py

!cp evaluate_best_adapter.py "/content/drive/MyDrive/Colab/scripts/"
!python evaluate_best_adapter.py --help | grep -c fleurs

In [ ]:
!python evaluate_best_adapter.py --splits fleurs --also-beam5 --batch-size 4 \
  --fleurs-csv "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test.tsv" \
  --fleurs-audio-dir "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test/test"

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv")
print(df.columns.tolist())
print(df.shape)

In [ ]:
import glob, os, pandas as pd

CSV = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/swahili_dataset.csv"
df = pd.read_csv(CSV).head(9000)
df["base"] = df["path"].map(lambda p: os.path.basename(str(p)))

cands = sorted(set(glob.glob("/content/drive/MyDrive/**/validated.tsv", recursive=True)))
print("validated.tsv found:", cands or "NONE")

if cands:
    meta = pd.read_csv(cands[0], sep="\t", quoting=3, low_memory=False)
    print("columns:", meta.columns.tolist())
    meta["base"] = meta["path"].map(lambda p: os.path.basename(str(p)))
    m = df.merge(meta[["base", "client_id"]], on="base", how="left")
    n = int(m["client_id"].notna().sum())
    print(f"\nmatched {n}/{len(m)} clips to a speaker")
    if n:
        vc = m["client_id"].value_counts()
        print(f"speakers:            {len(vc)}")
        print(f"top-1 speaker:       {100*vc.iloc[0]/n:.1f}% of clips")
        print(f"top-5 speakers:      {100*vc.head(5).sum()/n:.1f}%")
        print(f"top-10 speakers:     {100*vc.head(10).sum()/n:.1f}%")
        print(f"clips/speaker:       median {vc.median():.0f}, max {vc.max()}")
        out = "/content/drive/MyDrive/Colab/swahili_with_speakers.csv"
        m.to_csv(out, index=False); print(f"\nsaved -> {out}")

In [ ]:
import os, json, math, statistics, subprocess, sys

# ----------------------------- edit these -----------------------------
SEEDS  = [42, 1, 2]
MODELS = [   # cheapest first, so a full row of results lands early
    dict(tag="small",  hf="openai/whisper-small",    epochs="12", bs="8", ga="4"),
    dict(tag="medium", hf="openai/whisper-medium",   epochs="8",  bs="4", ga="8"),
    dict(tag="large",  hf="openai/whisper-large-v3", epochs="5",  bs="2", ga="16"),
]
ROOT    = "/content/drive/MyDrive/Colab/SwahiliASR_study"
SPEAKER = "/content/drive/MyDrive/Colab/swahili_with_speakers.csv"
AUDIO   = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips"
FL_CSV  = "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test.tsv"
FL_DIR  = "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test/test"
SPLIT, TRAIN, EVAL = ("/content/make_speaker_split.py",
                      "/content/swahili_asr_qlora.py",
                      "/content/evaluate_best_adapter.py")
CONDITIONS = ["seen_test", "unseen_test", "fleurs"]
# ----------------------------------------------------------------------

T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571}
def sh(c): print("$", " ".join(c), flush=True); return subprocess.call(c)
os.makedirs(f"{ROOT}/splits", exist_ok=True)

# 1. splits (cheap, idempotent)
for s in SEEDS:
    f = f"{ROOT}/splits/splits_seed{s}.csv"
    if not os.path.exists(f):
        if sh([sys.executable, SPLIT, "--csv", SPEAKER, "--seed", str(s), "--out", f]):
            raise SystemExit(f"split for seed {s} failed")

# 2. the grid
stop = False
for m in MODELS:
    if stop: break
    for s in SEEDS:
        tag  = f"{m['tag']}_seed{s}"
        out  = f"{ROOT}/{tag}"
        done = f"{out}/done.json"
        split = f"{ROOT}/splits/splits_seed{s}.csv"
        os.makedirs(out, exist_ok=True)
        if os.path.exists(done):
            print(f"{tag}: done, skipping"); continue
        print(f"\n===== {tag} =====", flush=True)

        if not os.path.isdir(f"{out}/best_adapter"):
            rc = sh([sys.executable, TRAIN, "--base-model", m["hf"],
                     "--split-csv", split, "--audio-dir", AUDIO, "--output-dir", out,
                     "--seed", str(s), "--epochs", m["epochs"],
                     "--batch-size", m["bs"], "--grad-accum", m["ga"],
                     "--layer-adaptive", "--encoder-r", "64", "--decoder-r", "16",
                     "--eval-steps", "250", "--save-steps", "100",
                     "--eval-subset", "300", "--num-beams", "1",
                     "--resume", "auto", "--skip-test"])
            if rc:
                print(f"\n{tag} stopped. RE-RUN THIS CELL — it resumes."); stop = True; break

        rc = sh([sys.executable, EVAL, "--adapter-dir", f"{out}/best_adapter",
                 "--base-model", m["hf"], "--split-csv", split, "--audio-dir", AUDIO,
                 "--splits", *CONDITIONS, "--fleurs-csv", FL_CSV,
                 "--fleurs-audio-dir", FL_DIR, "--batch-size", "4",
                 "--out-dir", f"{out}/eval_results"])
        if rc:
            print(f"\n{tag}: eval failed. RE-RUN THIS CELL."); stop = True; break
        json.dump({"model": m["hf"], "seed": s}, open(done, "w"))
        print(f"{tag}: complete")

# 3. aggregate
def stat(vals):
    n = len(vals); mu = statistics.mean(vals)
    if n < 2: return mu, 0.0, 0.0
    sd = statistics.stdev(vals)
    return mu, sd, T95.get(n - 1, 2.0) * sd / math.sqrt(n)

print("\n" + "=" * 78)
print("  WER by base model and speaker familiarity   (greedy, mean ± SD over seeds)")
print("=" * 78)
print(f"  {'model':<9}{'n':>3}{'seen_test':>16}{'unseen_test':>16}{'fleurs':>16}"
      f"{'unseen/seen':>13}")
print("-" * 78)
table = {}
for m in MODELS:
    got, cells = {}, []
    for cond in CONDITIONS:
        vals = []
        for s in SEEDS:
            f = f"{ROOT}/{m['tag']}_seed{s}/eval_results/eval_results.json"
            if os.path.exists(f):
                d = json.load(open(f))
                if f"{cond}_wer_greedy" in d: vals.append(d[f"{cond}_wer_greedy"])
        got[cond] = vals
        if not vals:                      cells.append("—")
        elif len(vals) < 2:               cells.append(f"{vals[0]:.2f} (n=1)")
        else: mu, sd, _ = stat(vals);     cells.append(f"{mu:.2f}±{sd:.2f}")
    n = len(got["seen_test"])
    ratio = (statistics.mean(got["unseen_test"]) / statistics.mean(got["seen_test"])
             if got["seen_test"] and got["unseen_test"] else None)
    print(f"  {m['tag']:<9}{n:>3}" + "".join(f"{c:>16}" for c in cells) +
          (f"{ratio:>12.2f}x" if ratio else f"{'—':>13}"))
    table[m["tag"]] = {c: got[c] for c in CONDITIONS}
print("=" * 78)
for m in MODELS:
    v = table[m["tag"]]["seen_test"]
    if len(v) >= 2:
        mu, sd, ci = stat(v)
        print(f"  {m['tag']:<9} seen_test  mean {mu:.2f}%  SD {sd:.2f}  "
              f"95% CI [{mu-ci:.2f}, {mu+ci:.2f}]")
json.dump(table, open(f"{ROOT}/study_summary.json", "w"), indent=2)
missing = [f"{m['tag']}_seed{s}" for m in MODELS for s in SEEDS
           if not os.path.exists(f"{ROOT}/{m['tag']}_seed{s}/done.json")]
if missing: print(f"\n  outstanding ({len(missing)}): {', '.join(missing[:6])}"
                  f"{' …' if len(missing) > 6 else ''}\n  re-run this cell")
print(f"  summary -> {ROOT}/study_summary.json")

In [ ]:
!ls -la /content/drive/MyDrive/Colab/*.csv

In [ ]:
from google.colab import files
files.upload()          # choose make_speaker_split.py
!mkdir -p "/content/drive/MyDrive/Colab/scripts"
!cp make_speaker_split.py "/content/drive/MyDrive/Colab/scripts/"
!ls -la /content/*.py

In [ ]:
import os, json, math, statistics, subprocess, sys

# ----------------------------- edit these -----------------------------
SEEDS  = [42, 1, 2]
MODELS = [   # cheapest first, so a full row of results lands early
    dict(tag="small",  hf="openai/whisper-small",    epochs="12", bs="8", ga="4"),
    dict(tag="medium", hf="openai/whisper-medium",   epochs="8",  bs="4", ga="8"),
    dict(tag="large",  hf="openai/whisper-large-v3", epochs="5",  bs="2", ga="16"),
]
ROOT    = "/content/drive/MyDrive/Colab/SwahiliASR_study"
SPEAKER = "/content/drive/MyDrive/Colab/swahili_with_speakers.csv"
AUDIO   = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips"
FL_CSV  = "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test.tsv"
FL_DIR  = "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test/test"
SPLIT, TRAIN, EVAL = ("/content/make_speaker_split.py",
                      "/content/swahili_asr_qlora.py",
                      "/content/evaluate_best_adapter.py")
CONDITIONS = ["seen_test", "unseen_test", "fleurs"]
# ----------------------------------------------------------------------

T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571}
def sh(c):
    # subprocess.call writes to the OS file descriptors, which Jupyter does NOT capture
    # into the cell — a child traceback would vanish and you would be debugging blind.
    # Inside IPython, route through its own shell so output streams live; otherwise
    # stream it line by line ourselves.
    import shlex
    cmd = " ".join(shlex.quote(x) for x in c)
    print("$", cmd, flush=True)
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    if ip is not None:
        ip.system(cmd)
        return int(ip.user_ns.get("_exit_code", 0) or 0)
    p_ = subprocess.Popen(c, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
    for line in p_.stdout:
        print(line.rstrip(), flush=True)
    return p_.wait()
os.makedirs(f"{ROOT}/splits", exist_ok=True)

# 1. splits (cheap, idempotent)
for s in SEEDS:
    f = f"{ROOT}/splits/splits_seed{s}.csv"
    if not os.path.exists(f):
        if sh([sys.executable, SPLIT, "--csv", SPEAKER, "--seed", str(s), "--out", f]):
            raise SystemExit(f"split for seed {s} failed — see the traceback above")

# 2. the grid
stop = False
for m in MODELS:
    if stop: break
    for s in SEEDS:
        tag  = f"{m['tag']}_seed{s}"
        out  = f"{ROOT}/{tag}"
        done = f"{out}/done.json"
        split = f"{ROOT}/splits/splits_seed{s}.csv"
        os.makedirs(out, exist_ok=True)
        if os.path.exists(done):
            print(f"{tag}: done, skipping"); continue
        print(f"\n===== {tag} =====", flush=True)

        if not os.path.isdir(f"{out}/best_adapter"):
            rc = sh([sys.executable, TRAIN, "--base-model", m["hf"],
                     "--split-csv", split, "--audio-dir", AUDIO, "--output-dir", out,
                     "--seed", str(s), "--epochs", m["epochs"],
                     "--batch-size", m["bs"], "--grad-accum", m["ga"],
                     "--layer-adaptive", "--encoder-r", "64", "--decoder-r", "16",
                     "--eval-steps", "250", "--save-steps", "100",
                     "--eval-subset", "300", "--num-beams", "1",
                     "--resume", "auto", "--skip-test"])
            if rc:
                print(f"\n{tag} stopped. RE-RUN THIS CELL — it resumes."); stop = True; break

        rc = sh([sys.executable, EVAL, "--adapter-dir", f"{out}/best_adapter",
                 "--base-model", m["hf"], "--split-csv", split, "--audio-dir", AUDIO,
                 "--splits", *CONDITIONS, "--fleurs-csv", FL_CSV,
                 "--fleurs-audio-dir", FL_DIR, "--batch-size", "4",
                 "--out-dir", f"{out}/eval_results"])
        if rc:
            print(f"\n{tag}: eval failed. RE-RUN THIS CELL."); stop = True; break
        json.dump({"model": m["hf"], "seed": s}, open(done, "w"))
        print(f"{tag}: complete")

# 3. aggregate
def stat(vals):
    n = len(vals); mu = statistics.mean(vals)
    if n < 2: return mu, 0.0, 0.0
    sd = statistics.stdev(vals)
    return mu, sd, T95.get(n - 1, 2.0) * sd / math.sqrt(n)

print("\n" + "=" * 78)
print("  WER by base model and speaker familiarity   (greedy, mean ± SD over seeds)")
print("=" * 78)
print(f"  {'model':<9}{'n':>3}{'seen_test':>16}{'unseen_test':>16}{'fleurs':>16}"
      f"{'unseen/seen':>13}")
print("-" * 78)
table = {}
for m in MODELS:
    got, cells = {}, []
    for cond in CONDITIONS:
        vals = []
        for s in SEEDS:
            f = f"{ROOT}/{m['tag']}_seed{s}/eval_results/eval_results.json"
            if os.path.exists(f):
                d = json.load(open(f))
                if f"{cond}_wer_greedy" in d: vals.append(d[f"{cond}_wer_greedy"])
        got[cond] = vals
        if not vals:                      cells.append("—")
        elif len(vals) < 2:               cells.append(f"{vals[0]:.2f} (n=1)")
        else: mu, sd, _ = stat(vals);     cells.append(f"{mu:.2f}±{sd:.2f}")
    n = len(got["seen_test"])
    ratio = (statistics.mean(got["unseen_test"]) / statistics.mean(got["seen_test"])
             if got["seen_test"] and got["unseen_test"] else None)
    print(f"  {m['tag']:<9}{n:>3}" + "".join(f"{c:>16}" for c in cells) +
          (f"{ratio:>12.2f}x" if ratio else f"{'—':>13}"))
    table[m["tag"]] = {c: got[c] for c in CONDITIONS}
print("=" * 78)
for m in MODELS:
    v = table[m["tag"]]["seen_test"]
    if len(v) >= 2:
        mu, sd, ci = stat(v)
        print(f"  {m['tag']:<9} seen_test  mean {mu:.2f}%  SD {sd:.2f}  "
              f"95% CI [{mu-ci:.2f}, {mu+ci:.2f}]")
json.dump(table, open(f"{ROOT}/study_summary.json", "w"), indent=2)
missing = [f"{m['tag']}_seed{s}" for m in MODELS for s in SEEDS
           if not os.path.exists(f"{ROOT}/{m['tag']}_seed{s}/done.json")]
if missing: print(f"\n  outstanding ({len(missing)}): {', '.join(missing[:6])}"
                  f"{' …' if len(missing) > 6 else ''}\n  re-run this cell")
print(f"  summary -> {ROOT}/study_summary.json")

In [ ]:
!cp "/content/drive/MyDrive/Colab/scripts"/*.py /content/
!ls /content/*.py

In [ ]:
import json, os
R = "/content/drive/MyDrive/Colab/SwahiliASR_study"
for tag in ["small_seed42", "medium_seed42"]:
    print(tag, json.load(open(f"{R}/{tag}/best_adapter/best_metric.json")))
    f = f"{R}/{tag}/results_seed42.json"
    if os.path.exists(f):
        print("   val history:", json.load(open(f)).get("val_wer_history"))

In [ ]:
!pip install -q --upgrade "transformers>=4.46" "peft>=0.13" "accelerate>=1.0"
!pip install -q --upgrade "datasets>=3.0" "bitsandbytes>=0.46"
!pip install -q --upgrade "evaluate>=0.4" "jiwer>=3.0"
!pip install -q --upgrade "soundfile>=0.12" "librosa>=0.10" "soxr>=0.3"

In [ ]:
import shutil, os
R = "/content/drive/MyDrive/Colab/SwahiliASR_study"
shutil.rmtree(f"{R}/medium_seed42")

In [ ]:
import os, json, math, statistics, subprocess, sys

# ----------------------------- edit these -----------------------------
SEEDS  = [42, 1, 2]
MODELS = [   # cheapest first, so a full row of results lands early
    dict(tag="small",  hf="openai/whisper-small",    epochs="12", bs="8", ga="4"),
    dict(tag="medium", hf="openai/whisper-medium",   epochs="12", bs="4", ga="8"),
    dict(tag="large",  hf="openai/whisper-large-v3", epochs="6",  bs="2", ga="16"),
]
ROOT    = "/content/drive/MyDrive/Colab/SwahiliASR_study"
SPEAKER = "/content/drive/MyDrive/Colab/swahili_with_speakers.csv"
AUDIO   = "/content/drive/MyDrive/Colab/New Swahili Dataset/mozilla-swa-dataset/clips"
FL_CSV  = "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test.tsv"
FL_DIR  = "/content/drive/MyDrive/Colab/Fleurs Swahili Dataset/test/test"
SPLIT, TRAIN, EVAL = ("/content/make_speaker_split.py",
                      "/content/swahili_asr_qlora.py",
                      "/content/evaluate_best_adapter.py")
CONDITIONS = ["seen_test", "unseen_test", "fleurs"]
# ----------------------------------------------------------------------

for _p in (SPLIT, TRAIN, EVAL):
    if not os.path.exists(_p):
        raise SystemExit(f"{_p} is missing. The Colab VM is wiped between sessions:\n"
                         f'  !cp "/content/drive/MyDrive/Colab/scripts"/*.py /content/')

T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571}
def sh(c):
    # subprocess.call writes to the OS file descriptors, which Jupyter does NOT capture
    # into the cell — a child traceback would vanish and you would be debugging blind.
    import shlex
    cmd = " ".join(shlex.quote(x) for x in c)
    print("$", cmd, flush=True)
    try:
        ip = get_ipython()
    except NameError:
        ip = None
    if ip is not None:
        ip.system(cmd)
        return int(ip.user_ns.get("_exit_code", 0) or 0)
    p_ = subprocess.Popen(c, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1)
    for line in p_.stdout:
        print(line.rstrip(), flush=True)
    return p_.wait()
os.makedirs(f"{ROOT}/splits", exist_ok=True)

# 1. splits (cheap, idempotent)
for s in SEEDS:
    f = f"{ROOT}/splits/splits_seed{s}.csv"
    if not os.path.exists(f):
        if sh([sys.executable, SPLIT, "--csv", SPEAKER, "--seed", str(s), "--out", f]):
            raise SystemExit(f"split for seed {s} failed — see the traceback above")

# 2. the grid
stop = False
for m in MODELS:
    if stop: break
    for s in SEEDS:
        tag  = f"{m['tag']}_seed{s}"
        out  = f"{ROOT}/{tag}"
        done = f"{out}/done.json"
        split = f"{ROOT}/splits/splits_seed{s}.csv"
        os.makedirs(out, exist_ok=True)
        if os.path.exists(done):
            print(f"{tag}: done, skipping"); continue
        print(f"\n===== {tag} =====", flush=True)

        # Training is complete only when the script has written its results file at the
        # very end of main(). An existing best_adapter/ proves nothing: a run killed
        # after its first checkpoint leaves one behind, and treating that as "trained"
        # silently evaluates a half-trained model. Always re-enter training; --resume
        # auto continues from the newest checkpoint, and returns almost immediately if
        # the run had in fact finished.
        if not os.path.exists(f"{out}/results_seed{s}.json"):
            rc = sh([sys.executable, TRAIN, "--base-model", m["hf"],
                     "--split-csv", split, "--audio-dir", AUDIO, "--output-dir", out,
                     "--seed", str(s), "--epochs", m["epochs"],
                     "--batch-size", m["bs"], "--grad-accum", m["ga"],
                     "--layer-adaptive", "--encoder-r", "64", "--decoder-r", "16",
                     "--eval-steps", "250", "--save-steps", "100",
                     "--eval-subset", "300", "--num-beams", "1",
                     "--resume", "auto", "--skip-test"])
            if rc:
                print(f"\n{tag} stopped. RE-RUN THIS CELL — it resumes."); stop = True; break
        if not os.path.exists(f"{out}/results_seed{s}.json"):
            print(f"\n{tag}: training exited 0 but wrote no results file; stopping "
                  f"rather than scoring a possibly half-trained adapter."); stop = True; break

        rc = sh([sys.executable, EVAL, "--adapter-dir", f"{out}/best_adapter",
                 "--base-model", m["hf"], "--split-csv", split, "--audio-dir", AUDIO,
                 "--splits", *CONDITIONS, "--fleurs-csv", FL_CSV,
                 "--fleurs-audio-dir", FL_DIR, "--batch-size", "4",
                 "--out-dir", f"{out}/eval_results"])
        if rc:
            print(f"\n{tag}: eval failed. RE-RUN THIS CELL."); stop = True; break
        json.dump({"model": m["hf"], "seed": s}, open(done, "w"))
        print(f"{tag}: complete")

# 3. aggregate
def stat(vals):
    n = len(vals); mu = statistics.mean(vals)
    if n < 2: return mu, 0.0, 0.0
    sd = statistics.stdev(vals)
    return mu, sd, T95.get(n - 1, 2.0) * sd / math.sqrt(n)

print("\n" + "=" * 78)
print("  WER by base model and speaker familiarity   (greedy, mean ± SD over seeds)")
print("=" * 78)
print(f"  {'model':<9}{'n':>3}{'seen_test':>16}{'unseen_test':>16}{'fleurs':>16}"
      f"{'unseen/seen':>13}")
print("-" * 78)
table = {}
for m in MODELS:
    got, cells = {}, []
    for cond in CONDITIONS:
        vals = []
        for s in SEEDS:
            f = f"{ROOT}/{m['tag']}_seed{s}/eval_results/eval_results.json"
            if os.path.exists(f):
                d = json.load(open(f))
                if f"{cond}_wer_greedy" in d: vals.append(d[f"{cond}_wer_greedy"])
        got[cond] = vals
        if not vals:                      cells.append("—")
        elif len(vals) < 2:               cells.append(f"{vals[0]:.2f} (n=1)")
        else: mu, sd, _ = stat(vals);     cells.append(f"{mu:.2f}±{sd:.2f}")
    n = len(got["seen_test"])
    ratio = (statistics.mean(got["unseen_test"]) / statistics.mean(got["seen_test"])
             if got["seen_test"] and got["unseen_test"] else None)
    print(f"  {m['tag']:<9}{n:>3}" + "".join(f"{c:>16}" for c in cells) +
          (f"{ratio:>12.2f}x" if ratio else f"{'—':>13}"))
    table[m["tag"]] = {c: got[c] for c in CONDITIONS}
print("=" * 78)
for m in MODELS:
    v = table[m["tag"]]["seen_test"]
    if len(v) >= 2:
        mu, sd, ci = stat(v)
        print(f"  {m['tag']:<9} seen_test  mean {mu:.2f}%  SD {sd:.2f}  "
              f"95% CI [{mu-ci:.2f}, {mu+ci:.2f}]")
json.dump(table, open(f"{ROOT}/study_summary.json", "w"), indent=2)
missing = [f"{m['tag']}_seed{s}" for m in MODELS for s in SEEDS
           if not os.path.exists(f"{ROOT}/{m['tag']}_seed{s}/done.json")]
if missing: print(f"\n  outstanding ({len(missing)}): {', '.join(missing[:6])}"
                  f"{' …' if len(missing) > 6 else ''}\n  re-run this cell")
print(f"  summary -> {ROOT}/study_summary.json")

In [ ]:
from huggingface_hub import login, HfApi

login()   # paste a token from https://huggingface.co/settings/tokens

REPO = "SWataki/whisper-large-v3-swahili-qlora"
api = HfApi()
api.create_repo(REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="/content/merged-swahili", repo_id=REPO, repo_type="model")